[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/alg/u5_ortogonalidad.ipynb)

# Álg. U5 · Ortogonalidad y mínimos cuadrados
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 5 del bloque de álgebra lineal. Tiene una sección por subtema, en el mismo orden que el libro:

1. Producto interior, longitud y ortogonalidad
2. Proyección ortogonal sobre una recta y sobre un subespacio
3. Proceso de Gram-Schmidt
4. Mínimos cuadrados y ajuste de modelos lineales

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección tiene una celda de teoría, código que la recorre paso a paso, una calculadora, sus casos de prueba y una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `ALG-U5` (bloque, unidad). Las celdas de código de cada sección vienen de `alg/u5_ortogonalidad/NN_*.py`; los fragmentos que el libro imprime, de `alg/u5_ortogonalidad/fragmentos/`.

**Convenciones.** En esta unidad hay dos maneras de escribir datos, y cada calculadora dice cuál usa.

- **Una lista de vectores** (secciones 1, 2 y 3) se escribe **un vector por renglón**, o con los vectores separados por `;`; las componentes se separan con comas o espacios. **Cada renglón escrito es un vector**, y la calculadora lo guarda como **una columna** de la matriz $A=[\,\mathbf v_1\ \mathbf v_2\ \cdots\ \mathbf v_k\,]$, de $n\times k$ ($n$ componentes, $k$ vectores). Por ejemplo, `1, 0, 1; 1, 1, 0` son los vectores $\mathbf v_1=(1,0,1)$ y $\mathbf v_2=(1,1,0)$, y

  $$A=\begin{bmatrix}1&1\\0&1\\1&0\end{bmatrix}.$$

  Se aceptan hasta 6 vectores de hasta 6 componentes, todos con el mismo número de componentes. También se acepta `(1, 0, 1), (1, 1, 0)` o `{(1, 0, 1), (1, 1, 0)}`.
- Un vector suelto ($\mathbf x$, la dirección $\mathbf u$ de una recta) se escribe como lista de números: `1, 2, 3`.
- **Los datos de la sección 4** se escriben como listas de números separados por espacios o comas: `0 1 2 3 4` o `0, 1, 2, 3, 4`. La lista de $x$ y la de $y$ deben tener el mismo número de datos (hasta 500). Para ajustar un plano, cada renglón es un dato escrito como `x, y, z`.
- Punto decimal. Puedes escribir fracciones como `2/3` (también `2 / 3`); los decimales se convierten a fracciones exactas (`0.1` se lee como $\tfrac{1}{10}$), así que los productos interiores, las proyecciones y las ecuaciones normales se calculan sin error de redondeo (en la sección 4, con hasta 12 datos y grado hasta 3; con más datos, más grado o logaritmos se calcula en decimales y la calculadora lo dice). Se aceptan los signos menos tipográficos (− y –) y un espacio después del signo (`- 1`).
- Los vectores que escribes se nombran $\mathbf v_1, \mathbf v_2,\dots$ en el orden en que los escribes (en lo que imprimen las calculadoras, `v1`, `v2`, ...); los que da Gram-Schmidt, $\mathbf u_1,\mathbf u_2,\dots$ (ortogonales) y $\mathbf q_1,\mathbf q_2,\dots$ (de longitud 1). Vectores, renglones, columnas y datos se cuentan desde 1.
- Un ángulo pide **siempre** su unidad, `"grados"` o `"radianes"`: no hay unidad por omisión.
- En lo que muestran las calculadoras, el signo menos es siempre `-`, las raíces se escriben con `√`, la longitud con `‖ ‖` y el seno con `sen`. Los decimales que se muestran se *redondean* (no se truncan) a las cifras significativas que elijas; si una fracción exacta resulta demasiado larga, la calculadora muestra solo sus decimales y lo avisa.
- Si usas las funciones desde código: a `lista_vectores` (y a las funciones que reciben vectores) le puedes pasar texto, una lista de vectores como `[(1, 0, 1), (1, 1, 0)]` o una `Matrix` de sympy; una `Matrix` se lee **por columnas** (cada columna es un vector), igual que la matriz $A$ que arma la calculadora. A las funciones de la sección 4 les puedes pasar texto o listas de números.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re, io, textwrap, contextlib, time
from decimal import Decimal, ROUND_HALF_UP, localcontext
from fractions import Fraction
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registra la proyección "3d")
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import sympy as sp
import ipywidgets as widgets
from IPython.display import display, Math

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Utilidades compartidas por todo el notebook (las mismas de la unidad 4): leer números y matrices exactos,
# redondear y mostrar
matplotlib.rcParams["axes.unicode_minus"] = False     # signo menos ASCII también en los ejes de las gráficas
_NUMERO = re.compile(r"[+-]?(\d+\.?\d*|\.\d+)([eE][+-]?\d+)?")
_AYUDA_NUMERO = "usa enteros, fracciones como 2/3 o decimales con punto"


_LIMITE = Fraction(10) ** 15                  # entradas entre 1e-15 y 1e15 en valor absoluto (o 0)


def _en_rango(q, texto):
    if q != 0 and not Fraction(1) / _LIMITE <= abs(q) <= _LIMITE:
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    return sp.Rational(q.numerator, q.denominator)


def numero(valor, nombre="el número"):
    """Un número como racional exacto de sympy: '0.1' -> 1/10, '2/3' -> 2/3, 5 -> 5, 1.2 -> 6/5."""
    if valor is None:
        raise ValueError(f"{nombre} está vacío: escribe un número.")
    if isinstance(valor, bool):
        raise ValueError(f"«{valor}» no es un número: {_AYUDA_NUMERO}.")
    if isinstance(valor, (int, np.integer)):
        return _en_rango(Fraction(int(valor)), valor)
    if isinstance(valor, (float, np.floating)) or (isinstance(valor, sp.Basic) and valor.is_Float):
        v = float(valor)
        if not math.isfinite(v):
            raise ValueError(f"{nombre} no es finito (escribiste {v}).")
        return _en_rango(Fraction(repr(v)), repr(v))      # repr da el decimal más corto: 0.1 -> '0.1' -> 1/10 exacto
    if isinstance(valor, sp.Basic):
        if valor.is_Rational:
            return valor                             # ya es exacto (por ejemplo, un resultado intermedio): no se acota
        raise ValueError(f"«{valor}» no es un número racional: {_AYUDA_NUMERO}.")
    texto = re.sub(r"\s+", "", str(valor)).replace("−", "-").replace("–", "-")      # acepta los signos menos tipográficos
    partes = texto.split("/")
    formas = [_NUMERO.fullmatch(p) for p in partes]
    if not texto or len(partes) > 2 or not all(formas):
        raise ValueError(f"«{str(valor).strip()}» no es un número: {_AYUDA_NUMERO}." if texto else
                         f"{nombre} está vacío: escribe un número.")
    if any(f.group(2) and abs(int(f.group(2)[1:])) > 30 for f in formas):   # 1e999999 no se calcula: se rechaza antes
        raise ValueError(f"«{texto}» está fuera del intervalo que maneja esta calculadora: "
                         "usa valores entre 1e-15 y 1e15 en valor absoluto (o 0).")
    num, *den = (Fraction(p) for p in partes)
    if den and den[0] == 0:
        raise ValueError(f"«{texto}» divide entre cero.")
    return _en_rango(num / den[0] if den else num, texto)


def _entradas(k):
    return f"{k} entrada" if k == 1 else f"{k} entradas"


def _componentes(k):
    return f"{k} componente" if k == 1 else f"{k} componentes"


def _desde_renglones(renglones, nombre):
    """Lista de renglones (cada uno, lista de entradas) -> Matrix de racionales, con mensajes claros."""
    if not renglones or not renglones[0]:
        raise ValueError(f"escribe {nombre} por renglones, por ejemplo: 1, 2; 3, 4")
    n = len(renglones[0])
    salida = []
    for i, renglon in enumerate(renglones, 1):
        if len(renglon) != n:
            raise ValueError(f"en {nombre}, el renglón {i} tiene {_entradas(len(renglon))} y el 1 tiene {n}: "
                             "todos los renglones deben tener el mismo número de entradas")
        fila = []
        for j, v in enumerate(renglon, 1):
            try:
                fila.append(numero(v))
            except ValueError as err:
                raise ValueError(f"en {nombre}, renglón {i}, columna {j}: {err}") from None
        salida.append(fila)
    return sp.Matrix(salida)


def _normaliza_entrada(texto):
    """Signos menos tipográficos (− y –) como '-', llaves como paréntesis y sin espacios alrededor de '/' ni entre un
    signo y su número: '2 / 3' -> '2/3', '- 1' -> '-1', '{(1, 1), (0, 1)}' -> '((1, 1), (0, 1))'."""
    t = texto.replace("−", "-").replace("–", "-").replace("{", "(").replace("}", ")")
    t = re.sub(r"\s*/\s*", "/", t)
    return re.sub(r"(^|[\s,;(\[])([+-])\s+(?=[\d.])", r"\1\2", t)


def _trozos(texto):
    """Texto -> lista de renglones, cada uno lista de entradas (texto). Acepta ';' o saltos de línea entre renglones,
    comas o espacios entre entradas, y la forma de Python [[1, 2], [3, 4]], (1, 2), (3, 4) o {(1, 2), (3, 4)}.
    Devuelve también el número del primer renglón con una entrada vacía (o None)."""
    t = _normaliza_entrada(texto)
    t = re.sub(r"[\]\)]\s*,?\s*[\[\(]", ";", t)            # [[1, 2], [3, 4]] o (1, 2), (3, 4): cada grupo es un renglón
    t = re.sub(r"[\[\]\(\)]", " ", t)
    renglones = [r.strip() for r in re.split(r"[;\n]", t)]
    renglones = [r for r in renglones if r]
    entradas, vacia = [], None
    for i, r in enumerate(renglones, 1):
        partes = re.split(r"\s*,\s*|\s+", r)
        if "" in partes and vacia is None:
            vacia = i
        entradas.append(partes)
    return entradas, vacia


def matriz(texto, nombre="la matriz"):
    """Lee una matriz escrita por renglones ('1, 2; 3, 4' o un renglón por línea) como Matrix de racionales exactos."""
    if not isinstance(texto, str):
        return como_matriz(texto, nombre)
    entradas, vacia = _trozos(texto)
    if vacia is not None:
        raise ValueError(f"en {nombre}, el renglón {vacia} tiene una entrada vacía: "
                         "revisa que no haya dos comas seguidas ni una coma al principio o al final")
    try:
        return _desde_renglones(entradas, nombre)
    except ValueError as err:
        if "todos los renglones deben tener" in str(err):
            raise ValueError(f"{err}{pista_coma(texto)}") from None
        raise


def pista_coma(texto):
    """La pista de usar punto decimal, solo si la lectura es de veras ambigua: hay comas entre dígitos sin espacio
    (1,5) y además números separados solo por espacios (1,5 2,5), que es como se escriben decimales con coma."""
    if isinstance(texto, str) and re.search(r"\d,\d", texto) and re.search(r"\d[ \t]+[-+.]?\d", texto):
        return "; si escribiste decimales, usa punto decimal: 1.5, no 1,5"
    return ""


def como_matriz(A, nombre="la matriz"):
    """Acepta texto, Matrix de sympy, lista de listas o arreglo de numpy y devuelve una Matrix de racionales exactos."""
    if isinstance(A, str):
        return matriz(A, nombre)
    if isinstance(A, sp.MatrixBase):
        renglones = A.tolist()
    elif isinstance(A, np.ndarray):
        if A.ndim > 2:
            raise ValueError(f"{nombre} debe ser una tabla de números (renglones y columnas)")
        renglones = np.atleast_2d(A).tolist()
    elif isinstance(A, (list, tuple)):
        anidada = [isinstance(f, (list, tuple, np.ndarray, sp.MatrixBase)) for f in A]
        if A and all(anidada):
            renglones = [list(f) for f in A]
        elif not any(anidada):
            renglones = [list(A)]                  # una lista sola es un renglón
        else:
            raise ValueError(f"en {nombre}, mezclaste números sueltos con renglones")
    else:
        renglones = [[A]]
    return _desde_renglones(renglones, nombre)


def vector(texto, nombre="el vector"):
    """Lee un vector ('1, 2, 3', '1; 2; 3', (1, 2, 3) o [1, 2, 3]) como columna de racionales exactos.
    Los mensajes de error dicen qué componente está mal."""
    ejemplo = f"escribe {nombre} como lista de números, por ejemplo: 5, 4, 1"
    if texto is None:
        raise ValueError(ejemplo)
    if isinstance(texto, str):
        filas, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre} hay una componente vacía: revisa que no haya dos comas seguidas "
                             "ni una coma al principio o al final")
        if not filas:
            raise ValueError(ejemplo)
        if len(filas) == 1:
            entradas = filas[0]
        elif all(len(f) == 1 for f in filas):
            entradas = [f[0] for f in filas]
        else:
            raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 5, 4, 1 (escribiste varios renglones)")
    elif isinstance(texto, (list, tuple, np.ndarray, sp.MatrixBase)):
        plano = ((isinstance(texto, np.ndarray) and texto.ndim == 1) or
                 (isinstance(texto, (list, tuple)) and
                  not any(isinstance(e, (list, tuple, np.ndarray, sp.MatrixBase)) for e in texto)))
        if not plano:                                   # Matrix, arreglo 2D o lista de listas
            return _vector_de_matriz(texto, nombre, ejemplo)
        entradas = list(texto)
        if not entradas:
            raise ValueError(ejemplo)
    else:
        entradas = [texto]                              # un número solo
    valores = []
    for j, e in enumerate(entradas, 1):
        try:
            valores.append(numero(e))
        except ValueError as err:
            raise ValueError(f"en {nombre}, componente {j}: {err}") from None
    return sp.Matrix(valores)


def _vector_de_matriz(texto, nombre, ejemplo):
    """Un vector dado como Matrix, arreglo o lista anidada: debe tener un solo renglón o una sola columna."""
    try:
        v = como_matriz(texto, nombre)
    except ValueError as err:
        if str(err).startswith("escribe "):
            raise ValueError(ejemplo) from None
        raise
    if v.rows == 1 or v.cols == 1:
        return v.reshape(v.rows * v.cols, 1)
    raise ValueError(f"{nombre} debe ser una lista de números, por ejemplo: 5, 4, 1 (escribiste una matriz {dims(v)})")


def dims(M):
    """Tamaño 'm×n' de una matriz."""
    return f"{M.rows}×{M.cols}"


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Un racional exacto se redondea desde su valor exacto (p/q con 50 cifras), no desde su aproximación en punto
    flotante: 1249999999999999999/10^19 con 2 cifras da 0.12, no 0.13. Una expresión exacta con raíces, senos o
    cosenos se evalúa con 50 cifras antes de redondear."""
    with localcontext() as ctx:
        ctx.prec = 50
        if isinstance(valor, sp.Basic) and not valor.is_Rational and valor.is_number:
            valor = sp.N(valor, 50)
            if valor.is_real is False:
                return "no definido"
            d = Decimal(str(valor)) if valor.is_Float else Decimal(repr(float(valor)))
        elif isinstance(valor, (sp.Rational, Fraction, int, np.integer)) and not isinstance(valor, bool):
            p, q = (int(valor.p), int(valor.q)) if isinstance(valor, sp.Rational) else (
                (valor.numerator, valor.denominator) if isinstance(valor, Fraction) else (int(valor), 1))
            d = Decimal(p) / Decimal(q)
        else:
            v = float(valor)
            if math.isnan(v):
                return "no definido"
            if math.isinf(v):
                return "+∞" if v > 0 else "-∞"
            d = Decimal(repr(v))
        if d == 0:
            return "0"
        q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        if q.adjusted() != d.adjusted():      # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
            q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
        e = q.adjusted()
        if e > 15 or e < -4:                  # muy grandes o muy chicos: notación científica, 1.234×10^-7
            return f"{format(q.scaleb(-e), 'f')}×10^{e}"
        if q.adjusted() >= n:                 # enteros grandes: exactos tal cual; redondeados, con ×10^
            return f"{int(q):d}" if q == d else f"{format(q.scaleb(-e), 'f')}×10^{e}"
        return format(q, "f")


def decimales_fijos(valor, d=2):
    """Redondea (5 sube) a d decimales fijos, desde el valor exacto: sqrt(3) con 2 -> '1.73'; 1 -> '1.00'."""
    with localcontext() as ctx:
        ctx.prec = 50
        x = Decimal(str(sp.N(sp.sympify(valor), 40)))
        return format(x.quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP), "f")


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una función rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


def _mensaje(fn):
    """Texto del ValueError que lanza fn() ('' si no lanza): para probar que el mensaje explica el error."""
    try:
        fn()
    except ValueError as err:
        return str(err)
    return ""


# --- formato de números y matrices ---------------------------------------------------------------
_SUB = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
_SUP = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")


def _sub(*indices):
    """Subíndices: _sub(1, 2) -> '₁₂'; con índices de dos cifras los separa con coma."""
    if all(k < 10 for k in indices):
        return "".join(str(k) for k in indices).translate(_SUB)
    return "₍" + ",".join(str(k) for k in indices).translate(_SUB) + "₎"


def _R(n):
    """Nombre del espacio en texto plano: _R(3) -> 'R³'."""
    return "R" + str(n).translate(_SUP)


def _txt(e):
    """Una cantidad exacta como texto legible: sqrt(3)/2 -> √3/2, 2*sin(30) -> 2·sen(30), pi/6 -> π/6."""
    t = str(e)
    t = re.sub(r"sqrt\((\d+)\)", r"√\1", t).replace("sqrt(", "√(")
    return t.replace("**", "^").replace("*", "·").replace("sin(", "sen(").replace("pi", "π")


def _latex(x):
    """LaTeX de sympy con la convención del libro: sen en lugar de sin."""
    return sp.latex(x).replace(r"\sin", r"\operatorname{sen}")


def _exacta_legible(M, largo=28, total=400):
    """True si la forma exacta de M se lee bien: cada entrada con pocos caracteres y no demasiadas en total.
    Si no, las calculadoras muestran solo los decimales y lo avisan."""
    textos = [_txt(e) for e in M]
    return all(len(t) <= largo for t in textos) and sum(len(t) for t in textos) <= total


_LARGA = "la forma exacta es muy larga: se muestra redondeada"


def _igual(nombre, x, n=4):
    """'nombre = x' con x exacto (y sus decimales si no es entero); si la forma exacta es muy larga,
    'nombre ≈ decimales (aviso)'."""
    x = sp.sympify(x)
    if len(_txt(x)) > 28:
        return f"{nombre} ≈ {cifras(x, n)} ({_LARGA})"
    return f"{nombre} = {_txt(x)}" + ("" if x.is_integer else f" ≈ {cifras(x, n)}")


def _en_linea(M):
    """Matriz en una línea con la misma notación de entrada: [2, 3; 5, -2]."""
    return "[" + "; ".join(", ".join(_txt(e) for e in M.row(i)) for i in range(M.rows)) + "]"


def _tupla(v, n=None):
    """Vector como (a, b, c); con n, además sus decimales redondeados a n cifras. Si la forma exacta es muy larga,
    solo los decimales (a 4 cifras si no se da n), con el aviso."""
    if not _exacta_legible(v):
        return "≈ (" + ", ".join(cifras(e, n or 4) for e in v) + f") ({_LARGA})"
    exacto = "(" + ", ".join(_txt(e) for e in v) + ")"
    if n is None or all(e.is_integer for e in v):
        return exacto
    return exacto + " ≈ (" + ", ".join(cifras(e, n) for e in v) + ")"


def _aprox(v, n):
    """Decimales redondeados de un vector: '(0.5000, 1.333)'; con una sola entrada, sin paréntesis."""
    textos = [cifras(e, n) for e in v]
    return textos[0] if len(textos) == 1 else "(" + ", ".join(textos) + ")"


def _plano(M, fmt=None, corte=None):
    """Texto plano de una matriz con columnas alineadas; corte = columna tras la que se dibuja una barra."""
    fmt = fmt or _txt
    celdas = [[fmt(M[i, j]) for j in range(M.cols)] for i in range(M.rows)]
    anchos = [max(len(celdas[i][j]) for i in range(M.rows)) for j in range(M.cols)]
    lineas = []
    for fila in celdas:
        partes = [c.rjust(a) for c, a in zip(fila, anchos)]
        if corte:
            partes = partes[:corte] + ["|"] + partes[corte:]
        lineas.append("[ " + "  ".join(partes) + " ]")
    return "\n".join(lineas)


_TEX = [("⁻¹", "^{-1}"), ("ᵀ", "^{T}"), ("²", "^{2}"), ("·", r"\cdot "), ("×", r"\times "), ("≈", r"\approx "),
        ("←", r"\leftarrow "), ("−", "-"), ("|", r"\,|\,")]
_SUB_TEX = re.compile("[₀₁₂₃₄₅₆₇₈₉]+")
_PROSA = re.compile(r"[A-Za-zÁÉÍÓÚáéíóúñü]{3,}")


def _a_tex(nombre):
    """Nombre de una cantidad en LaTeX. Si lleva palabras ('forma escalonada reducida'), va completo en \\text{...},
    con sus espacios. Si no, se traducen los símbolos (P_C⁻¹·P_B -> P_C^{-1}\\cdot P_B) y los subíndices (A2 -> A_{2},
    v1 -> \\mathbf v_{1}). Las calculadoras pasan el LaTeX a mano cuando el nombre mezcla palabras y símbolos."""
    if _PROSA.search(re.sub(r"\bdet\b", "", nombre)):
        return r"\text{" + nombre.replace("{", r"\{").replace("}", r"\}") + "}"
    t = nombre
    for a, b in _TEX:
        t = t.replace(a, b)
    t = re.sub(r"\bdet\s+", r"\\det ", t)
    t = _SUB_TEX.sub(lambda m: "_{" + m.group(0).translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")) + "}", t)
    return re.sub(r"(?<![A-Za-z0-9\\])([A-Za-z])(\d+)(?![\d/.])",
                  lambda m: (r"\mathbf " if m.group(1) in "vbeu" else "") + m.group(1) + "_{" + m.group(2) + "}", t)


def _tex_cifra(t):
    """Un número ya redondeado, en LaTeX: '1.234×10^-7' -> '1.234\\times 10^{-7}'."""
    return re.sub(r"×10\^(-?\d+)", r"\\times 10^{\1}", t)


def _tex_decimales(M, n):
    filas = r" \\ ".join(" & ".join(_tex_cifra(cifras(M[i, j], n)) for j in range(M.cols)) for i in range(M.rows))
    return r"\left[\begin{matrix}" + filas + r"\end{matrix}\right]"


class _Formula(Math):
    """Math de IPython con una versión en texto plano (la que se ve donde no se dibuja LaTeX)."""
    def __init__(self, tex, texto):
        super().__init__(tex)
        self._texto = texto

    def __repr__(self):
        return self._texto


class _Bloque:
    """Salida en Markdown (texto en recuadro y fórmulas en LaTeX) con su versión en texto plano."""
    def __init__(self, partes):
        self._partes = partes

    @staticmethod
    def _envuelve(texto, ancho=100):
        """Corta los renglones largos (en un recuadro de código no se ajustan solos); conserva la sangría."""
        lineas = []
        for linea in texto.rstrip("\n").split("\n"):
            sangria = " " * (len(linea) - len(linea.lstrip(" ")))
            lineas += textwrap.wrap(linea, ancho, subsequent_indent=sangria + "  ") if len(linea) > ancho else [linea]
        return "\n".join(lineas)

    def _repr_markdown_(self):
        return "\n\n".join(f"$${tex}$$" if tipo == "tex" else "```text\n" + self._envuelve(texto) + "\n```"
                            for tipo, tex, texto in self._partes)

    def __repr__(self):
        return "\n".join(texto.rstrip("\n") for _, _, texto in self._partes)


_BLOQUES = []      # bloques abiertos: lo que se muestra dentro de uno se junta y se entrega de una vez


@contextlib.contextmanager
def bloque():
    """Junta lo que se imprime y las fórmulas que se muestran dentro del bloque y lo entrega como UNA sola salida.
    Las calculadoras lo usan para que su resultado aparezca completo de una vez, en el orden en que se produjo."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue().replace("= ≈", "≈")))     # un valor que salió redondeado
            buf.seek(0); buf.truncate()

    _BLOQUES.append((partes, vacia))
    try:
        with contextlib.redirect_stdout(buf):
            yield
    finally:
        vacia()
        _BLOQUES.pop()
        if partes:
            _emite(_Bloque(partes)) if _BLOQUES else display(_Bloque(partes))


def _emite(formula):
    """Muestra una fórmula, o la agrega al bloque abierto si lo hay."""
    if not _BLOQUES:
        display(formula)
        return
    partes, vacia = _BLOQUES[-1]
    vacia()
    if isinstance(formula, _Bloque):
        partes.extend(formula._partes)
    else:
        partes.append(("tex", formula.data, formula._texto))


def muestra(nombre, M, n=None, tex=None):
    """Muestra 'nombre = M' con LaTeX (con su tamaño si es matriz). Con n agrega los decimales redondeados a n cifras.
    tex es el nombre en LaTeX (si no se da, se arma con _a_tex). Si la forma exacta es muy larga, solo los decimales."""
    tex_nombre = tex if tex is not None else _a_tex(nombre)
    if isinstance(M, sp.MatrixBase):
        if _exacta_legible(M):
            tex = f"{tex_nombre} = {_latex(M)}"
            texto = f"{nombre}  ({dims(M)}) =\n{_plano(M)}"
            if n is not None and not all(e.is_integer for e in M):
                tex += r" \approx " + _tex_decimales(M, n)
                texto += f"\n≈\n{_plano(M, lambda e: cifras(e, n))}"
        else:
            tex = f"{tex_nombre} \\approx {_tex_decimales(M, n or 4)}" + r"\quad\text{(" + _LARGA + ")}"
            texto = f"{nombre}  ({dims(M)}) ≈\n{_plano(M, lambda e: cifras(e, n or 4))}\n({_LARGA})"
        tex += rf"\qquad ({M.rows}\times {M.cols})"
    else:
        M = sp.sympify(M)
        tex, texto = f"{tex_nombre} = {_latex(M)}", f"{nombre} = {_txt(M)}"
        if n is not None and M.is_number and not M.is_integer:
            tex += rf" \approx {_tex_cifra(cifras(M, n))}"
            texto += f" ≈ {cifras(M, n)}"
    _emite(_Formula(tex, texto))

In [ ]:
# Utilidades de esta unidad: listas de vectores (cada vector es una COLUMNA), eliminación exacta, núcleos,
# textos de combinaciones, gráficas, pruebas y lectura de datos
_MAX_VECTORES, _MAX_COMPONENTES = 6, 6        # hasta 6 vectores de hasta 6 componentes


def lista_vectores(texto, nombre="la lista de vectores"):
    """Lee una lista de vectores escrita UN VECTOR POR RENGLÓN o con los vectores separados por ';'
    ('1, 0, 1; 2, 1, 1') y devuelve la matriz A de n×k que tiene esos vectores como COLUMNAS.
    También acepta una lista de vectores ([(1, 0, 1), (2, 1, 1)] o ['1, 0, 1', '2, 1, 1']) y una Matrix de sympy,
    que se toma tal cual: sus columnas ya son los vectores."""
    if isinstance(texto, sp.MatrixBase):
        A = como_matriz(texto, nombre)
        lista = [list(A[:, j]) for j in range(A.cols)]
    elif isinstance(texto, str):
        lista, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre}, el vector {vacia} tiene una componente vacía: "
                             "revisa que no haya dos comas seguidas ni una coma al principio o al final")
    elif isinstance(texto, (list, tuple)):
        lista = []
        for i, v in enumerate(texto, 1):
            if isinstance(v, (str, list, tuple, np.ndarray, sp.MatrixBase)):
                lista.append(list(vector(v, f"el vector {i} de {nombre}")))
            else:
                raise ValueError(f"en {nombre}, escribe cada vector como una lista de números, por ejemplo: "
                                 "[(1, 0, 1), (2, 1, 1)]")
    else:
        raise ValueError(f"escribe {nombre} con un vector por renglón o separados por «;», por ejemplo: 1, 0, 1; 2, 1, 1")
    if not lista or not lista[0]:
        raise ValueError(f"escribe {nombre} con un vector por renglón o separados por «;», por ejemplo: 1, 0, 1; 2, 1, 1")
    k, n = len(lista), len(lista[0])
    for i, v in enumerate(lista, 1):
        if len(v) != n:
            raise ValueError(f"en {nombre}, el vector {i} tiene {_componentes(len(v))} y el vector 1 tiene {n}: "
                             f"todos los vectores deben tener el mismo número de componentes{pista_coma(texto)}")
    if k > _MAX_VECTORES:
        raise ValueError(f"{nombre} tiene {k} vectores; esta calculadora acepta hasta {_MAX_VECTORES} "
                         "(recuerda: cada renglón escrito es un vector)")
    if n > _MAX_COMPONENTES:
        raise ValueError(f"en {nombre}, los vectores tienen {n} componentes; esta calculadora acepta hasta "
                         f"{_MAX_COMPONENTES}{pista_coma(texto)}")
    columnas = []
    for i, v in enumerate(lista, 1):
        col = []
        for j, e in enumerate(v, 1):
            try:
                col.append(numero(e))
            except ValueError as err:
                raise ValueError(f"en {nombre}, vector {i}, componente {j}: {err}") from None
        columnas.append(col)
    return sp.Matrix(columnas).T


def _es_cero(e):
    """True si e vale 0. Un racional se compara directo; una expresión exacta (con raíces, senos o cosenos) se
    simplifica y, si sympy no la reduce, se evalúa con 50 cifras."""
    e = sp.sympify(e)
    if e.is_Rational:
        return e == 0
    v = sp.simplify(e)
    if v == 0:
        return True
    if v.is_number:
        return bool(abs(sp.N(v, 50)) < sp.Float("1e-40", 50))
    return False


def _simplifica(M):
    """Simplifica las entradas que no son racionales (las racionales se quedan como están)."""
    return M.applyfunc(lambda e: e if e.is_Rational else sp.simplify(e))


def _factor(a):
    """Coeficiente delante de un renglón: 1 -> '', 2 -> '2·', 1/2 -> '(1/2)·', -1 -> '(-1)·'."""
    if a == 1:
        return ""
    if a.is_integer and a > 0:
        return f"{a}·"
    return f"({a})·"


def _op_suma(i, m, k):
    """Texto de R_i ← R_i - m·R_k con el signo que toca: m = 2 -> 'R2 ← R2 - 2·R1'; m = -3 -> 'R2 ← R2 + 3·R1'."""
    if m.is_number and m.is_real and m < 0:
        return f"R{i} ← R{i} + {_factor(-m)}R{k}"
    return f"R{i} ← R{i} - {_factor(m)}R{k}"


def _elimina(M, n_a, con_b=True):
    """Eliminación de Gauss y después de Jordan, exacta, sobre una copia de M (adaptada de la unidad 3).

    Los pivotes se buscan en las primeras n_a columnas; se intercambian renglones solo si el pivote es 0 y abajo hay
    una entrada distinta de cero (como a mano). Con con_b, al final se busca un pivote en la columna n_a (la de b):
    si aparece, hay un renglón 0 = c. Funciona también con entradas exactas no racionales (senos y cosenos de una
    rotación): para decidir si una entrada es cero, primero se simplifica.
    Devuelve un diccionario: E (escalonada), R (reducida), pivotes (columnas con pivote, desde 0), libres, r (rango),
    r_aum (rango con la columna b), imposible ((renglón, c) o None) y pasos, una lista de (tipo, texto, matriz)."""
    E = M.copy()
    racional = all(e.is_Rational for e in E)
    limpia = (lambda X: X) if racional else _simplifica
    m = E.rows
    pasos = [("fase", "Gauss: ceros debajo de cada pivote (forma escalonada)", None)]
    pivotes, libres, f, imposible = [], [], 0, None
    for c in range(n_a + (1 if con_b and M.cols > n_a else 0)):
        es_b = c == n_a
        k = next((i for i in range(f, m) if not _es_cero(E[i, c])), None)
        if k is None:
            if not es_b:
                libres.append(c)
                motivo = "ya no quedan renglones abajo" if f >= m else f"de R{f + 1} hacia abajo todo es cero"
                pasos.append(("nota", f"la columna {c + 1} no tiene pivote: {motivo}", None))
            continue
        if k != f:
            pasos.append(("nota", f"el pivote de la columna {c + 1} es cero; se intercambian R{f + 1} y R{k + 1}", None))
            E.row_swap(f, k)
            pasos.append(("op", f"R{f + 1} ↔ R{k + 1}", E.copy()))
        for i in range(f + 1, m):
            if not _es_cero(E[i, c]):
                mult = limpia(sp.Matrix([E[i, c] / E[f, c]]))[0]
                E[i, :] = limpia(E[i, :] - mult * E[f, :])
                E[i, c] = 0
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), E.copy()))
        if es_b:
            imposible = (f + 1, E[f, c])
            pasos.append(("nota", f"R{f + 1} quedó como 0 = {E[f, c]}: ninguna combinación lo cumple", None))
        else:
            pivotes.append(c)
        f += 1
    if len(pasos) == 1:
        pasos.append(("nota", "la matriz ya estaba escalonada: no hace falta ninguna operación", None))
    R = E.copy()
    pasos.append(("fase", "Jordan: cada pivote en 1 y ceros también arriba (forma escalonada reducida)", None))
    hechas = len(pasos)
    todos = list(enumerate(pivotes)) + ([(len(pivotes), n_a)] if imposible is not None else [])
    for f, c in reversed(todos):
        if not _es_cero(R[f, c] - 1):
            k = limpia(sp.Matrix([1 / R[f, c]]))[0]
            R[f, :] = limpia(k * R[f, :])
            R[f, c] = 1
            pasos.append(("op", f"R{f + 1} ← {_factor(k)}R{f + 1}", R.copy()))
        for i in range(f):
            if not _es_cero(R[i, c]):
                mult = R[i, c]
                R[i, :] = limpia(R[i, :] - mult * R[f, :])
                R[i, c] = 0
                pasos.append(("op", _op_suma(i + 1, mult, f + 1), R.copy()))
    if len(pasos) == hechas:
        pasos.append(("nota", "la forma escalonada ya era reducida: no hace falta ninguna operación", None))
    return {"E": E, "R": R, "pivotes": pivotes, "libres": libres, "r": len(pivotes),
            "r_aum": len(pivotes) + (imposible is not None), "imposible": imposible, "pasos": pasos}


def _direcciones(R, pivotes, n):
    """Base de Nul(A) leída de la forma reducida R de A (sus primeras n columnas): un vector por variable libre,
    con esa libre en 1 y las demás libres en 0."""
    direcciones = []
    for c_libre in [c for c in range(n) if c not in pivotes]:
        d = sp.zeros(n, 1)
        d[c_libre] = 1
        for f, c in enumerate(pivotes):
            d[c] = -R[f, c_libre]
        direcciones.append(d)
    return direcciones


def base_nulo(M):
    """Base exacta de Nul(M) = {x : M x = 0} (lista vacía si solo contiene al 0)."""
    el = _elimina(M, M.cols, con_b=False)
    return _direcciones(el["R"], el["pivotes"], M.cols)


def _primitivo(v):
    """El vector racional v (no nulo) escalado a enteros sin factor común y con su primera entrada distinta de cero
    positiva: (1/2, -1, 0) -> (1, -2, 0); (-2, -1, 1) -> (2, 1, -1)."""
    v = sp.Matrix(v)
    m = math.lcm(*[int(sp.Rational(e).q) for e in v])
    w = [int(e * m) for e in v]
    g = math.gcd(*w)
    signo = 1 if next(e for e in w if e != 0) > 0 else -1
    return sp.Matrix([signo * e // g for e in w])


def _termino(c, nombre, primero):
    """'2·v1', '- 1·v3', '+ (1/2)·v2': el coeficiente siempre se escribe, también el 1 y el 0. Si la fracción exacta
    es muy larga, se escribe redondeada: '+ (≈1.234×10^5)·v2'."""
    c = sp.sympify(c)
    mag = abs(c)
    coef = _txt(mag) if mag.is_integer else f"({_txt(mag)})"
    if len(coef) > 30:
        coef = f"(≈{cifras(mag, 4)})"
    negativo = bool(c.is_real and c < 0)
    if primero:
        return f"{'-' if negativo else ''}{coef}·{nombre}"
    return f"{'-' if negativo else '+'} {coef}·{nombre}"


def _subespacio(dim, n):
    """Nombre del subespacio según su dimensión: 'una recta por el origen', 'un plano por el origen', 'todo R³'..."""
    if dim == 0:
        return "{0} (solo el origen)"
    if dim == n:
        return f"todo {_R(n)}"
    if dim == 1:
        return "una recta por el origen"
    if dim == 2 and n == 3:
        return "un plano por el origen"
    return f"un subespacio de dimensión {dim} de {_R(n)} (pasa por el origen)"


def _lista(cosas):
    """Números en una frase: [1, 2, 3] -> '1, 2 y 3'."""
    cosas = [str(c) for c in cosas]
    return cosas[0] if len(cosas) == 1 else ", ".join(cosas[:-1]) + " y " + cosas[-1]


def _coma(cosas):
    """Nombres separados con comas: ['v1', 'v2'] -> 'v1, v2'."""
    return ", ".join(str(c) for c in cosas)


def _flotante(M):
    """Matrix de sympy -> arreglo de numpy en punto flotante."""
    return np.array(sp.Matrix(M).evalf().tolist(), dtype=float)


def normaliza_texto(texto):
    """Minúsculas, sin acentos ni signos de puntuación y con un solo espacio entre palabras: '¿Dependientes?' -> 'dependientes'."""
    import unicodedata
    t = unicodedata.normalize("NFD", str(texto).lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    return " ".join(re.sub(r"[^a-z0-9ñ ]+", " ", t).split())


# Gráficas: un color, un estilo de línea y un marcador por vector, en orden fijo, para que se lean también en blanco y negro
_COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
_ESTILOS = ["-", "--", ":", "-.", (0, (5, 1, 1, 1.5, 1, 1.5)), (0, (1, 4))]          # continuo, discontinuo, punteado...
_MARCAS = ["o", "s", "^", "D", "v", "P"]


def _flecha(ax, origen, punta, color, estilo, ancho, texto=None, desplazamiento=(6, 6)):
    """Flecha en el plano de origen a punta; con texto, su rótulo junto a la punta (sobre un fondo blanco)."""
    if np.hypot(*(np.asarray(punta, float) - np.asarray(origen, float))) > 1e-12:
        ax.annotate("", xy=punta, xytext=origen,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=ancho, linestyle=estilo, shrinkA=0, shrinkB=0))
    if texto:
        ax.annotate(texto, xy=punta, xytext=desplazamiento, textcoords="offset points", fontsize=11,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))


def _captura(fn, *args, **kwargs):
    """Ejecuta fn sin mostrar nada (ni texto ni gráficas) y devuelve (texto, error): lo que imprimió o mostró y la
    excepción que lanzó (None si no lanzó). Sirve para las pruebas de humo de las calculadoras."""
    partes, buf = [], io.StringIO()

    def vacia():
        if buf.getvalue():
            partes.append(("texto", "", buf.getvalue()))
            buf.seek(0); buf.truncate()

    mostrar, error = plt.show, None
    _BLOQUES.append((partes, vacia))
    plt.show = lambda *a, **k: plt.close("all")
    try:
        with contextlib.redirect_stdout(buf):
            fn(*args, **kwargs)
    except Exception as err:
        error = err
    finally:
        vacia()
        _BLOQUES.pop()
        plt.show = mostrar
        plt.close("all")
    return "\n".join(texto for _, _, texto in partes), error


def _humo(calculadora, controles, **cambios):
    """Prueba de humo: la calculadora corre con los valores de sus controles (y los cambios dados), no lanza ninguna
    excepción, escribe algo y no dice «Revisa la entrada»."""
    valores = {k: w.value for k, w in controles.items()}
    valores.update(cambios)
    texto, error = _captura(calculadora, **valores)
    return error is None and texto.strip() != "" and "Revisa la entrada" not in texto


def evalua(pruebas):
    """Ejecuta una vez cada caso (nombre, función que devuelve True o False). Devuelve [(nombre, ok)]; un caso
    que lanza una excepción cuenta como falla y su nombre dice por qué, sin interrumpir a los demás."""
    resultados = []
    for nombre, prueba in pruebas:
        try:
            ok = bool(prueba())
        except Exception as err:
            ok, nombre = False, f"{nombre}  (error: {type(err).__name__}: {err})"
        resultados.append((nombre, ok))
    return resultados


def corre_pruebas(pruebas, seccion):
    """Escribe OK o FALLA en cada caso y el resumen de la sección."""
    resultados = evalua(pruebas)
    for nombre, ok in resultados:
        print("OK  " if ok else "FALLA", nombre)
    print(f"Sección {seccion}: {sum(ok for _, ok in resultados)}/{len(pruebas)} casos pasan")


# --- utilidades propias de esta unidad ------------------------------------------------------------
def _punto(u, v):
    """Producto interior exacto de dos vectores columna: u·v = uᵀv."""
    return (u.T * v)[0]


def _es_nulo(v):
    """True si v es el vector cero."""
    return all(e == 0 for e in v)


def _valor(x, n=4):
    """Un valor exacto como texto, con sus decimales si no es entero: '2', '1/3 ≈ 0.3333', '√6 ≈ 2.449'. Si la forma
    exacta es muy larga, solo los decimales: '≈ 1.234 (aviso)'."""
    x = sp.sympify(x)
    if len(_txt(x)) > 28:
        return f"≈ {cifras(x, n)} ({_LARGA})"
    return _txt(x) + ("" if x.is_integer else f" ≈ {cifras(x, n)}")


def _eq(nombre, texto):
    """'nombre = texto', o 'nombre ≈ ...' si el texto ya es aproximado (empieza con '≈')."""
    return f"{nombre} {texto}" if texto.startswith("≈") else f"{nombre} = {texto}"


def _cuenta(k, singular, plural):
    """'1 dato', '5 datos': el número con el sustantivo en singular o en plural."""
    return f"{k} {singular if k == 1 else plural}"


def _cociente(num, den, n=4):
    """'(3/2)/(1/2) = 3': el cociente con paréntesis si el numerador o el denominador no es entero, y su valor.
    Si el cociente ya es el valor reducido ('3/2'), o si es muy largo, solo el valor."""
    num, den = sp.sympify(num), sp.sympify(den)
    parte = lambda e: _txt(e) if e.is_Integer else f"({_txt(e)})"
    frac = f"{parte(num)}/{parte(den)}"
    if frac == _txt(num / den) or len(frac) > 40:
        return _valor(num / den, n)
    return f"{frac} = {_valor(num / den, n)}"


def _productos_nulos(nombres):
    """'v1·v2 = 0', 'v1·v2 = v1·v3 = v2·v3 = 0'; con cuatro o más vectores, 'el producto de cada par de vectores
    distintos es 0'."""
    if len(nombres) > 3:
        return "el producto de cada par de vectores distintos es 0"
    return " = ".join(f"{a}·{b}" for i, a in enumerate(nombres) for b in nombres[i + 1:]) + " = 0"


def _aviso_coma(*textos):
    """Aviso (aunque no haya error) si la entrada tiene comas entre dígitos y números separados solo por espacios,
    que es como se escriben los decimales con coma: la calculadora leyó cada coma como separador."""
    if any(pista_coma(t) for t in textos):
        print("Aviso: hay comas entre dígitos (1,5) y números separados solo por espacios. Si escribiste decimales "
              "con coma, usa punto decimal (1.5, no 1,5): la calculadora leyó cada coma como separador.")


def _nombra_vectores(A, letra="v"):
    """'Vectores: v1 = (...), v2 = (...)  (2 vectores de R⁴)'."""
    print(("Vector: " if A.cols == 1 else "Vectores: ") +
          ", ".join(f"{letra}{j + 1} = {_tupla(A[:, j])}" for j in range(A.cols)) +
          f"  ({A.cols} {'vector' if A.cols == 1 else 'vectores'} de {_R(A.rows)})")


def _tex_A(k):
    """LaTeX de A = [v1 v2 ... vk] con los vectores en negritas y separados."""
    return r"A=[\," + r"\ ".join(rf"\mathbf v_{{{j}}}" for j in range(1, k + 1)) + r"\,]"


def _nombre_A(k):
    return f"A = [{' '.join(f'v{j}' for j in range(1, k + 1))}]"


_MAX_DATOS = 500                 # datos de la sección 4: hasta 500 por lista


def _texto_dato(e):
    """Un dato como se escribió: el texto tal cual; un número de Python o de sympy, en su forma corta."""
    if isinstance(e, str):
        return e
    if isinstance(e, (float, np.floating)):
        return repr(float(e))
    if isinstance(e, sp.Basic):
        return _txt(e)
    return str(e)


def datos(texto, nombre="los datos", con_texto=False):
    """Lee una lista de números (separados por espacios, comas, ';' o saltos de línea) como lista de racionales
    exactos: '0 1 2 3' -> [0, 1, 2, 3]; '10.0, 12.1' -> [10, 121/10]. Acepta también una lista o un arreglo de
    números. Hasta 500 datos; los mensajes de error dicen qué dato está mal. Con con_texto=True devuelve también
    los datos como se escribieron ('10.0', '12.1'), para mostrarlos igual."""
    ejemplo = f"escribe {nombre} como lista de números separados por espacios o comas, por ejemplo: 0 1 2 3"
    if texto is None:
        raise ValueError(ejemplo)
    if isinstance(texto, str):
        filas, vacia = _trozos(texto)
        if vacia is not None:
            raise ValueError(f"en {nombre} hay un dato vacío: revisa que no haya dos comas seguidas ni una coma al "
                             "principio o al final")
        entradas = [e for fila in filas for e in fila]
    elif isinstance(texto, np.ndarray):
        entradas = list(texto.ravel())
    elif isinstance(texto, sp.MatrixBase):
        entradas = list(texto)
    elif isinstance(texto, (list, tuple)):
        if any(isinstance(e, (list, tuple, np.ndarray, sp.MatrixBase)) for e in texto):
            raise ValueError(f"{nombre} debe ser una lista de números, sin listas dentro")
        entradas = list(texto)
    else:
        entradas = [texto]                              # un número solo
    if not entradas:
        raise ValueError(ejemplo)
    if len(entradas) > _MAX_DATOS:
        raise ValueError(f"{nombre} son {len(entradas)} números; esta calculadora acepta hasta {_MAX_DATOS}")
    valores = []
    for j, e in enumerate(entradas, 1):
        try:
            valores.append(numero(e))
        except ValueError as err:
            raise ValueError(f"en {nombre}, dato {j}: {err}") from None
    return (valores, [_texto_dato(e) for e in entradas]) if con_texto else valores


def _num(c, n=4):
    """Un número para tablas y ecuaciones: si es un racional exacto con decimal finito de a lo más n cifras
    significativas, ese decimal tal cual (10.02, 1.5, -0.2, 3); si no, redondeado a n cifras con cifras()."""
    if isinstance(c, (int, np.integer)) and not isinstance(c, bool):
        c = sp.Integer(int(c))
    if isinstance(c, sp.Rational):
        q = int(c.q)
        for f in (2, 5):
            while q % f == 0:
                q //= f
        if q == 1:
            with localcontext() as ctx:
                ctx.prec = 60
                d = (Decimal(int(c.p)) / Decimal(int(c.q))).normalize()
            if d == 0:
                return "0"
            if len(d.as_tuple().digits) <= n and -4 <= d.adjusted() <= 15:
                return format(d, "f")
    return cifras(c, n)


def muestra_decimales(nombre, M, n=4, tex=None):
    """Muestra una matriz o un vector de punto flotante (de numpy), redondeado a n cifras significativas.
    Un vector (arreglo de una dimensión) se muestra como columna."""
    M = np.asarray(M, dtype=float)
    M = M.reshape(-1, 1) if M.ndim == 1 else M
    S = sp.Matrix(M.tolist())
    tex_nombre = tex if tex is not None else _a_tex(nombre)
    _emite(_Formula(f"{tex_nombre} \\approx {_tex_decimales(S, n)}\\qquad ({S.rows}\\times {S.cols})",
                    f"{nombre}  ({dims(S)}) ≈\n{_plano(S, lambda e: cifras(e, n))}"))

## 1. Producto interior, longitud y ortogonalidad

**Producto interior.** Para $\mathbf u=(u_1,\dots,u_n)$ y $\mathbf v=(v_1,\dots,v_n)$ de $\mathbb R^n$,

$$\mathbf u\cdot\mathbf v=u_1v_1+u_2v_2+\dots+u_nv_n=\mathbf u^T\mathbf v .$$

El resultado es un número, no un vector. Cumple $\mathbf u\cdot\mathbf v=\mathbf v\cdot\mathbf u$, $(c\,\mathbf u)\cdot\mathbf v=c\,(\mathbf u\cdot\mathbf v)$, $\mathbf u\cdot(\mathbf v+\mathbf w)=\mathbf u\cdot\mathbf v+\mathbf u\cdot\mathbf w$ y $\mathbf v\cdot\mathbf v\ge 0$, con $\mathbf v\cdot\mathbf v=0$ solo si $\mathbf v=\mathbf 0$.

**Longitud y distancia.** La *norma* (longitud) es $\|\mathbf v\|=\sqrt{\mathbf v\cdot\mathbf v}$ y la distancia entre $\mathbf u$ y $\mathbf v$ es $\|\mathbf u-\mathbf v\|$. Si $\mathbf v\neq\mathbf 0$, $\mathbf v/\|\mathbf v\|$ es *unitario* (de longitud 1).

**Ángulo.** La desigualdad de Cauchy-Schwarz, $|\mathbf u\cdot\mathbf v|\le\|\mathbf u\|\,\|\mathbf v\|$, garantiza que

$$\cos\theta=\frac{\mathbf u\cdot\mathbf v}{\|\mathbf u\|\,\|\mathbf v\|}$$

está entre $-1$ y $1$ en cualquier $\mathbb R^n$. Por eso define un ángulo $\theta$ entre $0^\circ$ y $180^\circ$ (entre $0$ y $\pi$ radianes) aunque los vectores no se puedan dibujar. Con el vector cero el ángulo no está definido. Ese cociente es la **similitud coseno**: vale 1 si los vectores apuntan igual, 0 si son perpendiculares y $-1$ si apuntan en sentidos opuestos, sin importar su tamaño (por eso se usa para comparar documentos o señales).

**Ortogonalidad.** $\mathbf u$ y $\mathbf v$ son *ortogonales* ($\mathbf u\perp\mathbf v$) si $\mathbf u\cdot\mathbf v=0$. Como $\|\mathbf u+\mathbf v\|^2=\|\mathbf u\|^2+2\,\mathbf u\cdot\mathbf v+\|\mathbf v\|^2$, vale el **teorema de Pitágoras**:

$$\|\mathbf u+\mathbf v\|^2=\|\mathbf u\|^2+\|\mathbf v\|^2\quad\text{si y solo si}\quad\mathbf u\perp\mathbf v .$$

**Conjuntos ortogonales.** $\{\mathbf u_1,\dots,\mathbf u_k\}$ es *ortogonal* si $\mathbf u_i\cdot\mathbf u_j=0$ para cada par $i\neq j$, y *ortonormal* si además cada $\|\mathbf u_i\|=1$. Un conjunto ortogonal **de vectores no nulos** es linealmente independiente: si $c_1\mathbf u_1+\dots+c_k\mathbf u_k=\mathbf 0$, el producto con $\mathbf u_i$ deja $c_i\,(\mathbf u_i\cdot\mathbf u_i)=0$, y como $\mathbf u_i\cdot\mathbf u_i\neq 0$, $c_i=0$. Con los vectores como columnas de $A$, la entrada $(i,j)$ de la *matriz de Gram* $G=A^TA$ es $\mathbf u_i\cdot\mathbf u_j$: el conjunto es ortogonal si $G$ es diagonal y ortonormal si $G=I$.

**Coordenadas en una base ortogonal.** Si $\{\mathbf u_1,\dots,\mathbf u_k\}$ es una base ortogonal de $W$ y $\mathbf x$ está en $W$, no hace falta resolver un sistema: el producto de $\mathbf x=c_1\mathbf u_1+\dots+c_k\mathbf u_k$ con $\mathbf u_i$ deja un solo término, así que

$$c_i=\frac{\mathbf x\cdot\mathbf u_i}{\mathbf u_i\cdot\mathbf u_i}.$$

**Complemento ortogonal.** $W^\perp$ es el conjunto de los vectores perpendiculares a todos los de $W$. Si los vectores que generan $W$ son las columnas de $A$, un $\mathbf y$ es perpendicular a todos ellos si y solo si $A^T\mathbf y=\mathbf 0$, así que

$$W^\perp=\operatorname{Nul}(A^T),\qquad \dim W+\dim W^\perp=n .$$

En $\mathbb R^3$, el complemento de una recta por el origen es el plano perpendicular a ella, y el de un plano por el origen es su recta normal. Los vectores de la base de $W^\perp$ que da la calculadora salen escalados a enteros sin factor común y con la primera entrada distinta de cero positiva; cualquier múltiplo no nulo de cada uno sirve igual (otra herramienta puede darte, por ejemplo, los mismos vectores con el signo cambiado).

In [ ]:
_UNIDADES = {"grados": "grados", "grado": "grados", "radianes": "radianes", "radian": "radianes", "rad": "radianes"}


def _unidad(unidad):
    """'grados' o 'radianes' (acepta mayúsculas, acentos, el singular y 'rad'). No hay unidad por omisión: sin unidad,
    o con otra cosa, ValueError en español."""
    if unidad is None or (isinstance(unidad, str) and not unidad.strip()):
        raise ValueError("indica la unidad: 'grados' o 'radianes'")
    t = normaliza_texto(unidad) if isinstance(unidad, str) else ""
    if t not in _UNIDADES:
        raise ValueError(f"indica la unidad: 'grados' o 'radianes' (escribiste «{unidad}»)")
    return _UNIDADES[t]


def _par(u, v, nu="u", nv="v"):
    """Lee dos vectores y revisa que tengan el mismo número de componentes."""
    u, v = vector(u, nu), vector(v, nv)
    if u.rows != v.rows:
        raise ValueError(f"{nu} tiene {_componentes(u.rows)} y {nv} tiene {v.rows}: deben tener el mismo número "
                         "de componentes")
    return u, v


def producto_interior(u, v):
    """u·v = u1v1 + ... + unvn, exacto."""
    u, v = _par(u, v)
    return _punto(u, v)


def norma(v):
    """‖v‖ = √(v·v), exacta (con raíz si hace falta)."""
    v = vector(v, "el vector")
    return sp.sqrt(_punto(v, v))


def distancia(u, v):
    """Distancia ‖u - v‖, exacta."""
    u, v = _par(u, v)
    return sp.sqrt(_punto(u - v, u - v))


def _coseno(u, v):
    """cos θ = u·v / (‖u‖ ‖v‖), exacto. ValueError si alguno es el vector cero."""
    if _es_nulo(u) or _es_nulo(v):
        raise ValueError("el ángulo no está definido para el vector cero")
    return _punto(u, v) / sp.sqrt(_punto(u, u) * _punto(v, v))


def similitud_coseno(u, v):
    """Similitud coseno u·v / (‖u‖ ‖v‖), exacta: 1 si apuntan igual, 0 si son perpendiculares, -1 si son opuestos.
    Con el vector cero, ValueError «la similitud no está definida para el vector cero»."""
    u, v = _par(u, v)
    if _es_nulo(u) or _es_nulo(v):
        raise ValueError("la similitud no está definida para el vector cero")
    return _coseno(u, v)


def angulo(u, v, unidad=None, n_cifras=4):
    """Ángulo entre u y v. La unidad es obligatoria: angulo(u, v, "grados") o angulo(u, v, "radianes"); sin unidad,
    ValueError «indica la unidad». Devuelve un diccionario con cos (exacto), angulo (exacto si sympy lo simplifica,
    por ejemplo 60 o pi/3; si no, el valor decimal), exacto (True o False), aprox (decimales redondeados a n_cifras),
    unidad y texto ('60 grados', 'π/3 radianes ≈ 1.047', '≈ 73.22 grados')."""
    un = _unidad(unidad)
    u, v = _par(u, v)
    c = _coseno(u, v)
    t = sp.acos(c)
    valor = t if un == "radianes" else t * 180 / sp.pi
    exacto = not valor.has(sp.acos)
    if exacto and not valor.is_Rational:
        valor = sp.simplify(valor)
    aprox = cifras(valor, n_cifras)
    if exacto:
        texto = f"{_txt(valor)} {un}" + ("" if valor.is_integer else f" ≈ {aprox}")
    else:
        texto = f"≈ {aprox} {un}"
    return {"cos": c, "angulo": valor if exacto else float(sp.N(valor, 30)), "exacto": exacto, "aprox": aprox,
            "unidad": un, "texto": texto}


def informe_ortogonalidad(vectores):
    """¿El conjunto es ortogonal? ¿ortonormal? Diccionario con normas (exactas), gram (G = AᵀA exacta), no_ortogonales
    (lista de (i, j, vi·vj) con producto distinto de 0, índices desde 1), ortogonal, ortonormal, hay_cero y A."""
    A = lista_vectores(vectores)
    k = A.cols
    G = A.T * A
    no_ortogonales = [(i + 1, j + 1, G[i, j]) for i in range(k) for j in range(i + 1, k) if G[i, j] != 0]
    hay_cero = any(G[j, j] == 0 for j in range(k))
    ortogonal = not no_ortogonales
    return {"normas": [sp.sqrt(G[j, j]) for j in range(k)], "gram": G, "no_ortogonales": no_ortogonales,
            "ortogonal": ortogonal, "ortonormal": ortogonal and all(G[j, j] == 1 for j in range(k)),
            "hay_cero": hay_cero, "A": A}


def complemento_ortogonal(vectores):
    """W⊥ = Nul(Aᵀ) para W = gen{v1, ..., vk}: diccionario con base (vectores exactos escalados a enteros sin factor
    común, con la primera entrada distinta de cero positiva), dim_W (= rango de A), dim_complemento, n y A."""
    A = lista_vectores(vectores)
    base = [_primitivo(w) for w in base_nulo(A.T)]
    r = _elimina(A, A.cols, con_b=False)["r"]
    return {"base": base, "dim_W": r, "dim_complemento": len(base), "n": A.rows, "A": A}


def coordenadas_ortogonales(base, x):
    """Coordenadas de x en una base ORTOGONAL u1, ..., uk: c_i = x·u_i / (u_i·u_i), exactas, sin resolver ningún
    sistema. Diccionario con coeficientes (lista exacta), en_W (True si x = c1·u1 + ... + ck·uk), suma (ese vector),
    productos (las parejas (x·u_i, u_i·u_i)) y la base como matriz. Si un vector es el cero, o si la base no es
    ortogonal, ValueError."""
    P = lista_vectores(base, "la base")
    x = vector(x, "x")
    n, k = P.shape
    if x.rows != n:
        raise ValueError(f"x tiene {_componentes(x.rows)} y los vectores de la base tienen {n}: deben coincidir")
    ceros = [j + 1 for j in range(k) if _es_nulo(P[:, j])]
    if ceros:
        raise ValueError(f"el vector {ceros[0]} de la base es el cero: una base no puede contener al vector cero")
    G = P.T * P
    malos = [(i + 1, j + 1, G[i, j]) for i in range(k) for j in range(i + 1, k) if G[i, j] != 0]
    if malos:
        i, j, g = malos[0]
        raise ValueError(f"la base no es ortogonal: usa las ecuaciones normales de la sección 2 "
                         f"(v{i}·v{j} = {_valor(g)}, no 0)")
    productos = [(_punto(x, P[:, j]), G[j, j]) for j in range(k)]
    c = [a / b for a, b in productos]
    suma = sum((cj * P[:, j] for j, cj in enumerate(c)), sp.zeros(n, 1))
    return {"coeficientes": c, "en_W": suma == x, "suma": suma, "productos": productos, "base": P, "x": x}


def _informe_ortogonalidad(inf, comp, unidad, n_cifras):
    A, G, k, n = inf["A"], inf["gram"], inf["A"].cols, inf["A"].rows
    _nombra_vectores(A)
    for j, nv in enumerate(inf["normas"], 1):
        print(_igual(f"‖v{j}‖", nv, n_cifras) + ("   (es el vector cero)" if nv == 0 else ""))
    muestra("G = AᵀA", G, n_cifras, tex=r"G=A^TA")
    nombres = [f"v{j}" for j in range(1, k + 1)]
    if k == 1:
        print("G tiene una sola entrada: ‖v1‖². Con un solo vector no hay pares: no hay ángulos que calcular.")
    else:
        print("En la diagonal de G están " + _lista([f"‖{nm}‖²" for nm in nombres]) + "; fuera de ella, los "
              "productos entre vectores distintos (la entrada 1, 2 es v1·v2).")
    for i in range(k):
        for j in range(i + 1, k):
            nombre = f"Ángulo entre v{i + 1} y v{j + 1}"
            if G[i, i] == 0 or G[j, j] == 0:
                print(f"{nombre}: no está definido (hay un vector cero).")
                continue
            a = angulo(A[:, i], A[:, j], unidad, n_cifras)
            c = a["cos"]
            nota = "  (perpendiculares)" if c == 0 else ("  (paralelos, mismo sentido)" if c == 1 else
                                                        ("  (paralelos, sentidos opuestos)" if c == -1 else ""))
            print(f"{nombre}: cos θ = v{i + 1}·v{j + 1}/(‖v{i + 1}‖·‖v{j + 1}‖) = {_valor(c, n_cifras)} → "
                  f"θ = {a['texto']}{nota}")
    if k == 2 and G[0, 0] != 0 and G[1, 1] != 0:
        s = A[:, 0] + A[:, 1]
        izq, der = _punto(s, s), G[0, 0] + G[1, 1]
        if G[0, 1] == 0:
            print(f"Pitágoras: ‖v1 + v2‖² = {_valor(izq, n_cifras)} y ‖v1‖² + ‖v2‖² = "
                  f"{_valor(G[0, 0], n_cifras)} + {_valor(G[1, 1], n_cifras)} = {_valor(der, n_cifras)}: "
                  "se cumple, porque v1·v2 = 0.")
        else:
            print(f"Pitágoras no se cumple: ‖v1 + v2‖² = {_valor(izq, n_cifras)} y ‖v1‖² + ‖v2‖² = "
                  f"{_valor(der, n_cifras)}; la diferencia es 2·v1·v2 = {_valor(2 * G[0, 1], n_cifras)}.")
    if inf["no_ortogonales"]:
        pares = ", ".join(f"v{i}·v{j} = {_valor(g, n_cifras)}" for i, j, g in inf["no_ortogonales"])
        print(f"El conjunto NO es ortogonal: {pares}.")
    elif k == 1:
        print("Un conjunto de un solo vector es ortogonal (no hay pares que revisar).")
    else:
        print(f"El conjunto ES ORTOGONAL: {_productos_nulos(nombres)} (G es diagonal).")
    if inf["ortonormal"]:
        print("Además es ORTONORMAL: " + ("v1 mide 1 (G = [1])." if k == 1 else "cada vector mide 1 (G = I)."))
    elif inf["ortogonal"] and not inf["hay_cero"]:
        print("No es ortonormal: " + ("v1 no mide 1 (v1/‖v1‖ sí)." if k == 1 else
                                     "no todos miden 1. Divide cada vector entre su norma para normalizarlo."))
    if inf["hay_cero"]:
        print("v1 es el vector cero: {0} es dependiente y no sirve de base." if k == 1 else
              "Contiene al vector cero: un conjunto con el cero es dependiente y no sirve de base.")
    elif inf["ortogonal"]:
        print("v1 no es el cero, así que {v1} es linealmente independiente." if k == 1 else
              "Como no contiene al cero, es linealmente independiente.")
    print(f"Complemento ortogonal: W = gen{{{_coma(nombres)}}} tiene dim W = {comp['dim_W']} "
          f"y W⊥ = Nul(Aᵀ) tiene dim W⊥ = {n} - {comp['dim_W']} = {comp['dim_complemento']}.")
    if comp["base"]:
        a_quien = "v1" if k == 1 else _lista(nombres)
        cada = "es perpendicular a" if len(comp["base"]) == 1 else "cada uno es perpendicular a"
        print("Base de W⊥: " + ", ".join(_tupla(w) for w in comp["base"]) +
              (f"   ({cada} {a_quien})" if all(A.T * w == sp.zeros(k, 1) for w in comp["base"])
               else "   (¡alguno NO es perpendicular!)"))
    else:
        print(f"W⊥ = {{0}}: W es todo {_R(n)}.")


def _informe_coordenadas_ortogonales(res, n_cifras):
    P, x = res["base"], res["x"]
    k = P.cols
    print("Base: " + ", ".join(f"u{j} = {_tupla(P[:, j - 1])}" for j in range(1, k + 1)) +
          (f"   (ortogonal: {_productos_nulos([f'u{j}' for j in range(1, k + 1)])})" if k > 1 else ""))
    for j, (a, b) in enumerate(res["productos"], 1):
        print(f"  c{j} = (x·u{j})/(u{j}·u{j}) = {_cociente(a, b, n_cifras)}")
    coef = sp.Matrix(res["coeficientes"])
    if res["en_W"]:
        print(f"Coordenadas: {_tupla(coef, n_cifras)}.  Comprobación: " +
              " ".join(_termino(c, f"u{j}", j == 1) for j, c in enumerate(res["coeficientes"], 1)) +
              f" = {_tupla(res['suma'])} = x")
    else:
        print(f"x NO está en W: la combinación con esos coeficientes da {_tupla(res['suma'], n_cifras)}, que no es x; "
              "es la proyección de x sobre W (sección 2).")

**Recorrido paso a paso.** Primero, $\mathbf u=(1,-1,2,0)$ y $\mathbf v=(3,1,-1,2)$ de $\mathbb R^4$: $\mathbf u\cdot\mathbf v=3-1-2+0=0$, así que son ortogonales y se cumple Pitágoras: $\|\mathbf u\|^2=6$, $\|\mathbf v\|^2=15$ y $\|\mathbf u+\mathbf v\|^2=21$. Después, el ángulo entre $(1,1,1,1,1,1,1,1,1)$ y $(1,0,0,0,0,0,0,0,0)$, de $\mathbb R^9$, que no se puede dibujar pero sí calcular: $\cos\theta=\dfrac{1}{3\cdot 1}=\tfrac13$ y $\theta\approx 70.53^\circ$ (aquí el ángulo no es uno de los «notables», y la calculadora lo da en decimales). Al final, las coordenadas de $\mathbf x=(3,1,2)$ en la base ortogonal $\{(1,1,1),(1,-1,0),(1,1,-2)\}$, sin resolver ningún sistema: $(2,1,0)$.

In [ ]:
_u, _v = vector("1, -1, 2, 0"), vector("3, 1, -1, 2")
print(f"u = {_tupla(_u)},  v = {_tupla(_v)}")
print(f"u·v = 1·3 + (-1)·1 + 2·(-1) + 0·2 = {producto_interior(_u, _v)}  → u y v son ortogonales")
print(f"‖u‖² = {producto_interior(_u, _u)},  ‖v‖² = {producto_interior(_v, _v)},  u + v = {_tupla(_u + _v)},  "
      f"‖u + v‖² = {producto_interior(_u + _v, _u + _v)}")
print(f"Pitágoras: {producto_interior(_u, _u)} + {producto_interior(_v, _v)} = "
      f"{producto_interior(_u, _u) + producto_interior(_v, _v)} = ‖u + v‖²")
muestra("‖u‖", norma(_u), 4, tex=r"\|\mathbf u\|")
muestra("‖u - v‖", distancia(_u, _v), 4, tex=r"\|\mathbf u-\mathbf v\|")
print()
_unos, _e1 = "1, 1, 1, 1, 1, 1, 1, 1, 1", "1, 0, 0, 0, 0, 0, 0, 0, 0"
_a = angulo(_unos, _e1, "grados")
print(f"Ángulo entre {_tupla(vector(_unos))} y {_tupla(vector(_e1))} (de R⁹): u·v = 1, ‖u‖ = 3, ‖v‖ = 1, "
      f"cos θ = 1/(3·1) = {_txt(_a['cos'])} → θ {_a['texto']} ({angulo(_unos, _e1, 'radianes')['texto']})")
print()
_informe_coordenadas_ortogonales(coordenadas_ortogonales("1, 1, 1; 1, -1, 0; 1, 1, -2", "3, 1, 2"), 4)

**Calculadora.** Escribe los vectores **uno por renglón** y elige la unidad de los ángulos. La calculadora da las normas, la matriz de Gram $G=A^TA$, el ángulo de cada par, si el conjunto es ortogonal u ortonormal y una base del complemento ortogonal $W^\perp$. Prueba `5/13, 12/13` y `-12/13, 5/13` (ortonormal), un solo vector de $\mathbb R^3$ (su complemento es un plano) y un conjunto con el vector cero.

In [ ]:
def calculadora_ortogonalidad(vectores_txt, unidad, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            inf = informe_ortogonalidad(vectores_txt)
            comp = complemento_ortogonal(inf["A"])
            _unidad(unidad)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _aviso_coma(vectores_txt)
        _informe_ortogonalidad(inf, comp, unidad, n_cifras)


_CONTROLES_1 = dict(
    vectores_txt=widgets.Textarea(value="1, -1, 2, 0\n3, 1, -1, 2", description="vectores:", continuous_update=False,
                                  layout=widgets.Layout(width="340px", height="120px")),
    unidad=widgets.ToggleButtons(options=["grados", "radianes"], value="grados", description="unidad:"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_ortogonalidad, **_CONTROLES_1);

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("producto_interior((1, -1, 2, 0), (3, 1, -1, 2)) = 0 y producto_interior((2, -1, 3), (1, 4, 2)) = 4",
     lambda: producto_interior("1,-1,2,0", "3,1,-1,2") == 0 and producto_interior("2,-1,3", "1,4,2") == 4),
    ("norma((1, -1, 2, 0)) = √6 y distancia((1, -1, 2, 0), (3, 1, -1, 2)) = √21",
     lambda: norma("1,-1,2,0") == sp.sqrt(6) and distancia("1,-1,2,0", "3,1,-1,2") == sp.sqrt(21)),
    ("ángulo entre (1, 1, 0) y (-1, 0, 1): 120 grados exacto y 2π/3 en radianes; con (1, ..., 1) y (1, 0, ..., 0) "
     "de R⁹, cos θ = 1/3 y ≈ 70.53 grados (no exacto)",
     lambda: (lambda g, r, d: g["angulo"] == 120 and g["exacto"] and g["cos"] == sp.Rational(-1, 2)
                              and r["angulo"] == 2 * sp.pi / 3 and not d["exacto"] and d["cos"] == sp.Rational(1, 3)
                              and d["aprox"] == "70.53")(
         angulo("1,1,0", "-1,0,1", "grados"), angulo("1,1,0", "-1,0,1", "radianes"),
         angulo("1,1,1,1,1,1,1,1,1", "1,0,0,0,0,0,0,0,0", "grados"))),
    ("ángulo con el vector cero, (0, 0) y (1, 2) → ValueError «no está definido para el vector cero»",
     lambda: "vector cero" in _mensaje(lambda: angulo("0,0", "1,2", "grados"))),
    ("ángulo sin unidad o con otra unidad → ValueError «indica la unidad: 'grados' o 'radianes'»",
     lambda: _mensaje(lambda: angulo("1,0", "0,1")).startswith("indica la unidad: 'grados' o 'radianes'")
             and _mensaje(lambda: angulo("1,0", "0,1", "sexagesimales")).startswith("indica la unidad")),
    ("(1, 1, 1), (1, -1, 0), (1, 1, -2) → ortogonal, no ortonormal, sin cero; G diagonal (3, 2, 6)",
     lambda: (lambda r: r["ortogonal"] and not r["ortonormal"] and not r["hay_cero"]
                        and r["gram"] == sp.diag(3, 2, 6))(informe_ortogonalidad("1,1,1; 1,-1,0; 1,1,-2"))),
    ("(5/13, 12/13), (-12/13, 5/13) → ortonormal (G = I)",
     lambda: (lambda r: r["ortonormal"] and r["gram"] == sp.eye(2))(informe_ortogonalidad("5/13, 12/13; -12/13, 5/13"))),
    ("(1, 2), (2, 1) → no ortogonal, no_ortogonales = [(1, 2, 4)]",
     lambda: (lambda r: not r["ortogonal"] and not r["ortonormal"] and r["no_ortogonales"] == [(1, 2, 4)])(
         informe_ortogonalidad("1,2; 2,1"))),
    ("coordenadas de (3, 1, 2) en la base ortogonal (1, 1, 1), (1, -1, 0), (1, 1, -2) → (2, 1, 0), x en W",
     lambda: (lambda r: r["coeficientes"] == [2, 1, 0] and r["en_W"])(
         coordenadas_ortogonales("1,1,1; 1,-1,0; 1,1,-2", "3,1,2"))),
    ("coordenadas_ortogonales con la base (1, 0), (1, 1), que no es ortogonal → ValueError «la base no es ortogonal»",
     lambda: _mensaje(lambda: coordenadas_ortogonales("1,0; 1,1", "1,1")).startswith("la base no es ortogonal")),
    ("complemento de gen{(1, 1, 1)} → dim 2, cada vector de la base ⊥ (1, 1, 1) y los dos independientes",
     lambda: (lambda r: r["dim_complemento"] == 2 and r["dim_W"] == 1
                        and all(_punto(w, vector("1,1,1")) == 0 for w in r["base"])
                        and sp.Matrix.hstack(*r["base"]).rank() == 2)(complemento_ortogonal("1,1,1"))),
    ("complemento de gen{(1, 2, 0, -1), (0, 1, 1, 2)} → dim 2, perpendicular a los dos, 2 + 2 = 4",
     lambda: (lambda r: r["dim_complemento"] == 2 and r["dim_W"] + r["dim_complemento"] == r["n"] == 4
                        and all(_punto(w, vector(a)) == 0 for w in r["base"] for a in ("1,2,0,-1", "0,1,1,2"))
                        and sp.Matrix.hstack(*r["base"]).rank() == 2)(complemento_ortogonal("1,2,0,-1; 0,1,1,2"))),
    ("similitud_coseno((2, 0), (4, 0)) = 1, ((1, 0), (0, 3)) = 0, ((1, 0), (-2, 0)) = -1; con el cero → ValueError "
     "«la similitud no está definida para el vector cero»",
     lambda: similitud_coseno("2,0", "4,0") == 1 and similitud_coseno("1,0", "0,3") == 0
             and similitud_coseno("1,0", "-2,0") == -1
             and _mensaje(lambda: similitud_coseno("0,0", "1,2")) == "la similitud no está definida para el vector cero"),
    ("prueba de humo: la calculadora corre con sus valores iniciales (en grados y en radianes), sin error y sin "
     "«Revisa la entrada»",
     lambda: _humo(calculadora_ortogonalidad, _CONTROLES_1)
             and _humo(calculadora_ortogonalidad, _CONTROLES_1, unidad="radianes")),
    ("base de W⊥ escalada a enteros y con la primera entrada positiva: gen{(1, 1, 1)} → (1, -1, 0), (1, 0, -1); "
     "gen{(2, 3, 0), (0, 0, 1)} → (3, -2, 0)",
     lambda: complemento_ortogonal("1,1,1")["base"] == [vector("1,-1,0"), vector("1,0,-1")]
             and complemento_ortogonal("2,3,0; 0,0,1")["base"] == [vector("3,-2,0")]),
    ("vectores dependientes (1, 2, 3), (2, 4, 6): dim W = rango = 1 (no 2) y dim W⊥ = 2",
     lambda: (lambda r: r["dim_W"] == 1 and r["dim_complemento"] == 2)(complemento_ortogonal("1,2,3; 2,4,6"))),
    ("x = (1, 2, 3) fuera de W = gen{(1, 1, 0), (1, -1, 0)}: coeficientes (3/2, -1/2), en_W False, suma (1, 2, 0)",
     lambda: (lambda r: r["coeficientes"] == [sp.Rational(3, 2), sp.Rational(-1, 2)] and r["en_W"] is False
                        and r["suma"] == vector("1,2,0"))(coordenadas_ortogonales("1,1,0; 1,-1,0", "1,2,3"))),
    ("coordenadas_ortogonales con un vector cero en la base → ValueError «el vector 2 de la base es el cero»",
     lambda: _mensaje(lambda: coordenadas_ortogonales("1,1,0; 0,0,0", "1,1,0")).startswith(
         "el vector 2 de la base es el cero")),
    ("normas exactas con raíz y vector cero: (1, 2), (2, -1) → normas (√5, √5); (1, 2, 0), (0, 0, 0) → hay_cero",
     lambda: informe_ortogonalidad("1,2; 2,-1")["normas"] == [sp.sqrt(5), sp.sqrt(5)]
             and (lambda r: r["hay_cero"] and r["normas"] == [sp.sqrt(5), 0] and not r["ortonormal"])(
                 informe_ortogonalidad("1,2,0; 0,0,0"))
             and not informe_ortogonalidad("1,1,1; 1,-1,0; 1,1,-2")["hay_cero"]),
]
corre_pruebas(PRUEBAS_1, 1)

**Contrasta (sección 1).** Calcula a mano el ángulo entre $(1,1,0,0)$ y $(0,1,1,0)$: primero el producto interior y las dos normas, después $\cos\theta$. Escribe `mi_angulo = ` en **grados**.

In [ ]:
mi_angulo = None     # un número en grados, por ejemplo 45 o 37.5

if mi_angulo is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mio = numero(mi_angulo, "tu ángulo")
    except ValueError as err:
        print(f"Revisa tu ángulo: {str(err).rstrip('.')}.")
    else:
        if not 0 <= mio <= 180:          # respuesta imposible: la referencia todavía no se muestra
            print(f"El ángulo entre dos vectores está entre 0 y 180 grados (escribiste {_num(mio, 6)}): revisa tu "
                  "cálculo y vuelve a ejecutar la celda.")
        else:
            ref = angulo("1, 1, 0, 0", "0, 1, 1, 0", "grados")
            ref_rad = angulo("1, 1, 0, 0", "0, 1, 1, 0", "radianes")
            print(f"La calculadora da: cos θ = {_txt(ref['cos'])}, θ = {ref['texto']} ({ref_rad['texto']}).")
            if abs(float(mio) - float(ref["angulo"])) <= 0.5:
                print("Coinciden (con tolerancia de 0.5 grados).")
            elif abs(float(mio) - float(ref_rad["angulo"])) <= 0.01:
                print(f"NO coinciden: {_num(mio, 6)} es el ángulo en RADIANES. La celda pide grados: multiplica por "
                      "180/π, o pon tu calculadora en grados al sacar el arco coseno.")
            elif abs(float(mio) - float(ref["cos"])) <= 0.005:
                print(f"NO coinciden: {_num(mio, 6)} es cos θ, no θ. Falta sacar el arco coseno (en grados).")
            else:
                print("NO coinciden: u·v cuenta solo las componentes que son distintas de cero en los dos vectores; "
                      "cada norma es la raíz de la suma de los cuadrados. Después, θ = arccos(u·v/(‖u‖‖v‖)).")
            print("\nEl procedimiento completo:")
            _informe_ortogonalidad(informe_ortogonalidad("1, 1, 0, 0; 0, 1, 1, 0"),
                                   complemento_ortogonal("1, 1, 0, 0; 0, 1, 1, 0"), "grados", 4)

## 2. Proyección ortogonal sobre una recta y sobre un subespacio

**Sobre una recta.** La proyección de $\mathbf x$ sobre la recta por el origen de dirección $\mathbf u\neq\mathbf 0$ es el múltiplo $c\,\mathbf u$ que deja el resto $\mathbf x-c\,\mathbf u$ perpendicular a $\mathbf u$. De $(\mathbf x-c\,\mathbf u)\cdot\mathbf u=0$ sale $c=\dfrac{\mathbf x\cdot\mathbf u}{\mathbf u\cdot\mathbf u}$, es decir,

$$\operatorname{proy}_{\mathbf u}\mathbf x=\frac{\mathbf x\cdot\mathbf u}{\mathbf u\cdot\mathbf u}\,\mathbf u,\qquad\text{con matriz}\qquad P=\frac{\mathbf u\,\mathbf u^T}{\mathbf u^T\mathbf u}.$$

**Descomposición ortogonal.** Si $W$ es un subespacio de $\mathbb R^n$, cada $\mathbf x$ se escribe **de una sola manera** como

$$\mathbf x=\mathbf p+\mathbf r,\qquad \mathbf p\ \text{en}\ W,\qquad \mathbf r\ \text{en}\ W^\perp .$$

$\mathbf p=\operatorname{proy}_W\mathbf x$ es la *proyección ortogonal* de $\mathbf x$ sobre $W$ y $\mathbf r=\mathbf x-\mathbf p$ es el *residuo*.

- **Con una base ortogonal** $\{\mathbf u_1,\dots,\mathbf u_k\}$ de $W$, $\mathbf p$ es la suma de las proyecciones sobre cada vector:

  $$\mathbf p=\frac{\mathbf x\cdot\mathbf u_1}{\mathbf u_1\cdot\mathbf u_1}\,\mathbf u_1+\dots+\frac{\mathbf x\cdot\mathbf u_k}{\mathbf u_k\cdot\mathbf u_k}\,\mathbf u_k .$$
- **Con una base cualquiera** (las columnas independientes de $A$), $\mathbf p=A\mathbf c$, y pedir que el residuo sea perpendicular a cada columna, $A^T(\mathbf x-A\mathbf c)=\mathbf 0$, da las **ecuaciones normales**

  $$A^TA\,\mathbf c=A^T\mathbf x,\qquad \mathbf p=A\mathbf c,\qquad P=A\,(A^TA)^{-1}A^T .$$

  $A^TA$ es invertible porque las columnas de $A$ son independientes. Si la base trae un vector que es combinación de los anteriores, se quita: no cambia $W$.

**Mejor aproximación.** $\mathbf p$ es el punto de $W$ más cercano a $\mathbf x$. Para cualquier otro $\mathbf w$ de $W$, $\mathbf x-\mathbf w=\mathbf r+(\mathbf p-\mathbf w)$ con $\mathbf p-\mathbf w$ en $W$, y por Pitágoras

$$\|\mathbf x-\mathbf w\|^2=\|\mathbf r\|^2+\|\mathbf p-\mathbf w\|^2\ \ge\ \|\mathbf r\|^2 .$$

Por eso la **distancia de $\mathbf x$ a $W$** es $\|\mathbf r\|$.

**La matriz de proyección** cumple $P^2=P$ (proyectar dos veces es proyectar una) y $P^T=P$ (es simétrica). Si $\mathbf x$ ya está en $W$, $P\mathbf x=\mathbf x$ y $\mathbf r=\mathbf 0$.

In [ ]:
def _x_y_base(x, base):
    """Lee x y la base de W (lista de vectores: cada uno se vuelve una columna de A) y revisa los tamaños."""
    A = lista_vectores(base, "la base de W")
    x = vector(x, "x")
    if x.rows != A.rows:
        raise ValueError(f"x tiene {_componentes(x.rows)} y los vectores de la base tienen {A.rows}: deben coincidir")
    return x, A


def proyeccion_recta(x, u):
    """Proyección de x sobre la recta por el origen de dirección u: diccionario con p = (x·u/u·u)·u, r = x - p,
    coef = x·u/u·u, distancia = ‖r‖ (exacta), P = u uᵀ/(uᵀu), x y u. Si u es el vector cero, ValueError."""
    u = vector(u, "u")
    x = vector(x, "x")
    if x.rows != u.rows:
        raise ValueError(f"x tiene {_componentes(x.rows)} y u tiene {u.rows}: deben tener el mismo número de componentes")
    if _es_nulo(u):
        raise ValueError("u no puede ser el vector cero: el 0 no define ninguna recta")
    coef = _punto(x, u) / _punto(u, u)
    p = coef * u
    r = x - p
    return {"p": p, "r": r, "coef": coef, "distancia": sp.sqrt(_punto(r, r)), "P": u * u.T / _punto(u, u),
            "x": x, "u": u}


def proyeccion_subespacio(x, base):
    """Proyección de x sobre W = gen{v1, ..., vk}, exacta. Diccionario con p, r, coeficientes (los de p en los
    vectores usados), usados (sus números, desde 1), distancia (exacta) y distancia_decimal, metodo ('base ortogonal:
    suma de proyecciones' o 'ecuaciones normales'), ATA, ATx, r_ortogonal (Aᵀr = 0), P = A(AᵀA)⁻¹Aᵀ, avisos, A (los
    vectores usados como columnas) y x. Si un vector es combinación de los anteriores, no se usa y se avisa."""
    x, A0 = _x_y_base(x, base)
    n, k = A0.shape
    piv = _elimina(A0, k, con_b=False)["pivotes"]
    if not piv:
        raise ValueError("todos los vectores de la base son el cero: generan {0} y no hay sobre qué proyectar")
    avisos = []
    for j in range(k):
        if j not in piv:
            avisos.append(f"el vector {j + 1} es el cero y no se usa" if _es_nulo(A0[:, j]) else
                          f"el vector {j + 1} es combinación de los anteriores y no se usa")
    A = A0.extract(list(range(n)), piv)
    ATA, ATx = A.T * A, A.T * x
    ortogonal = all(ATA[i, j] == 0 for i in range(ATA.rows) for j in range(ATA.cols) if i != j)
    if ortogonal:
        c = sp.Matrix([ATx[i] / ATA[i, i] for i in range(A.cols)])
        metodo = "base ortogonal: suma de proyecciones"
    else:
        c = ATA.LUsolve(ATx)
        metodo = "ecuaciones normales"
    p = A * c
    r = x - p
    if _es_nulo(r):
        avisos.append("x ya está en W: su proyección es x mismo y r = 0")
    dist = sp.sqrt(_punto(r, r))
    return {"p": p, "r": r, "coeficientes": list(c), "usados": [j + 1 for j in piv], "distancia": dist,
            "distancia_decimal": float(sp.N(dist, 30)), "metodo": metodo, "ATA": ATA, "ATx": ATx,
            "r_ortogonal": es_perpendicular(r, A0), "P": A * ATA.inv() * A.T, "avisos": avisos, "A": A, "x": x,
            "ortogonal": ortogonal}


def es_perpendicular(r, vectores):
    """True si r es perpendicular a cada uno de los vectores (r·v = 0 para todos), en forma exacta."""
    A = lista_vectores(vectores)
    r = vector(r, "r")
    if r.rows != A.rows:
        raise ValueError(f"r tiene {_componentes(r.rows)} y los vectores tienen {A.rows}: deben coincidir")
    return all(_punto(r, A[:, j]) == 0 for j in range(A.cols))


def suma_de_proyecciones(x, base):
    """La suma de las proyecciones de x sobre cada vector de la base, proy_v1 x + ... + proy_vk x. Diccionario con
    suma, terminos (cada proyección), residuo (x - suma), p (la proyección de verdad) y es_la_proyeccion (True solo
    si la suma coincide con p, lo que se garantiza cuando la base es ortogonal)."""
    x, A = _x_y_base(x, base)
    for j in range(A.cols):
        if _es_nulo(A[:, j]):
            raise ValueError(f"el vector {j + 1} es el cero: no se puede proyectar sobre él")
    terminos = [_punto(x, A[:, j]) / _punto(A[:, j], A[:, j]) * A[:, j] for j in range(A.cols)]
    suma = sum(terminos, sp.zeros(A.rows, 1))
    p = proyeccion_subespacio(x, A)["p"]
    return {"suma": suma, "terminos": terminos, "residuo": x - suma, "p": p, "es_la_proyeccion": suma == p}


def _factor_comun(v):
    """(c, w) con v = c·w, c > 0 y w de enteros sin factor común: (-7/6, 7/6, 7/3) -> (7/6, (-1, 1, 2))."""
    w = _primitivo(v)
    i = next(i for i, e in enumerate(w) if e != 0)
    c = v[i] / w[i]
    return (c, w) if c > 0 else (-c, -w)


def _informe_recta(res, n_cifras):
    x, u = res["x"], res["u"]
    print(f"x = {_tupla(x)},  u = {_tupla(u)}")
    print(f"x·u = {_valor(_punto(x, u), n_cifras)},  u·u = {_valor(_punto(u, u), n_cifras)}  →  "
          f"c = (x·u)/(u·u) = {_valor(res['coef'], n_cifras)}")
    print(f"p = c·u = {_tupla(res['p'], n_cifras)}")
    print(f"r = x - p = {_tupla(res['r'], n_cifras)};  r·u = {_valor(_punto(res['r'], u), n_cifras)}"
          + ("  (perpendicular a u)" if _punto(res["r"], u) == 0 else "  (¡no es perpendicular!)"))
    print(f"distancia de x a la recta = ‖r‖ = {_valor(res['distancia'], n_cifras)}")
    muestra("P = u uᵀ/(uᵀu)", res["P"], n_cifras, tex=r"P=\dfrac{\mathbf u\,\mathbf u^T}{\mathbf u^T\mathbf u}")


def _informe_proyeccion(res, n_cifras, comparar_suma=True):
    A, x, p, r = res["A"], res["x"], res["p"], res["r"]
    k = A.cols
    nombres = [f"v{j}" for j in res["usados"]]
    print(f"x = {_tupla(x)}")
    print("Base de W que se usa: " + ", ".join(f"{nm} = {_tupla(A[:, i])}" for i, nm in enumerate(nombres)) +
          f"  (dim W = {k}: {_subespacio(k, A.rows)})")
    for aviso in res["avisos"]:
        if "no se usa" in aviso:
            print("Aviso:", aviso + ".")
    if res["ortogonal"]:
        if k > 1:
            print(f"La base es ortogonal ({_productos_nulos(nombres)}): AᵀA es diagonal y p es la suma de las "
                  "proyecciones sobre cada vector.")
        muestra("AᵀA", res["ATA"], n_cifras, tex=r"A^TA")
        muestra("Aᵀx", res["ATx"], n_cifras, tex=r"A^T\mathbf x")
        for i, nm in enumerate(nombres):
            print(f"  c{i + 1} = (x·{nm})/({nm}·{nm}) = {_cociente(res['ATx'][i], res['ATA'][i, i], n_cifras)}")
    else:
        malos = [f"{nombres[i]}·{nombres[j]} = {_valor(res['ATA'][i, j], n_cifras)}" for i in range(k)
                 for j in range(i + 1, k) if res["ATA"][i, j] != 0]
        print(f"La base NO es ortogonal ({', '.join(malos)}): se usan las ecuaciones normales AᵀA c = Aᵀx.")
        muestra("AᵀA", res["ATA"], n_cifras, tex=r"A^TA")
        muestra("Aᵀx", res["ATx"], n_cifras, tex=r"A^T\mathbf x")
        print(f"Solución: c = {_tupla(sp.Matrix(res['coeficientes']), n_cifras)}")
    print(f"p = " + " ".join(_termino(c, nm, i == 0) for i, (c, nm) in enumerate(zip(res["coeficientes"], nombres))) +
          f" = {_tupla(p, n_cifras)}")
    print(f"r = x - p = {_tupla(r, n_cifras)}")
    print("Comprobación: Aᵀr = 0, así que r es perpendicular a cada vector de W." if res["r_ortogonal"] else
          "Comprobación FALLÓ: Aᵀr ≠ 0.")
    print(f"Distancia de x a W = ‖r‖ = {_valor(res['distancia'], n_cifras)}")
    if "x ya está en W: su proyección es x mismo y r = 0" in res["avisos"]:
        print("x ya está en W: su proyección es x mismo y r = 0.")
    P = res["P"]
    muestra("P = A(AᵀA)⁻¹Aᵀ", P, n_cifras, tex=r"P=A\,(A^TA)^{-1}A^T")
    print(f"P² = P: {'sí' if P * P == P else 'no'};  Pᵀ = P: {'sí' if P.T == P else 'no'};  "
          f"P·x = p: {'sí' if P * x == p else 'no'}")
    if comparar_suma and not res["ortogonal"]:
        s = suma_de_proyecciones(x, A)
        print(f"Ojo: sumar las proyecciones sobre {_coma(nombres)} daría {_tupla(s['suma'], n_cifras)}, que NO es p: "
              "con una base que no es ortogonal, la suma no es la proyección.")


# --- dibujo ---------------------------------------------------------------------------------------
def _base_ortonormal(V):
    """Base ortonormal (en punto flotante, solo para dibujar) de las columnas de V que en decimales se distinguen
    como independientes: la parte perpendicular de cada columna debe medir más de 1e-9 veces la columna (tolerancia
    RELATIVA, que no depende de la escala de los datos)."""
    Q = []
    for v in V.T:
        nv = np.linalg.norm(v)
        if nv == 0 or not np.isfinite(nv):
            continue
        w = v - sum((q @ v) * q for q in Q)
        w = w - sum((q @ w) * q for q in Q)                # segunda pasada: menos error de redondeo
        if np.linalg.norm(w) > 1e-9 * nv:
            Q.append(w / np.linalg.norm(w))
    return Q


def _corte_con_caja(normal, L):
    """Vértices, en orden, del polígono en que el plano normal·x = 0 corta a la caja [-L, L]³ (para dibujarlo)."""
    esquinas = np.array([(a, b, c) for a in (-L, L) for b in (-L, L) for c in (-L, L)], dtype=float)
    puntos = []
    for i in range(8):
        for j in range(i + 1, 8):
            p, q = esquinas[i], esquinas[j]
            if np.count_nonzero(p != q) != 1:          # solo las 12 aristas de la caja
                continue
            fp, fq = normal @ p, normal @ q
            if fp == fq or fp * fq > 0:
                continue
            puntos.append(p + fp / (fp - fq) * (q - p))
    P = np.unique(np.round(np.array(puntos) / L, 12), axis=0) * L
    centro = P.mean(axis=0)
    u = P[0] - centro
    u /= np.linalg.norm(u)
    w = np.cross(normal / np.linalg.norm(normal), u)
    angulos = np.arctan2((P - centro) @ w, (P - centro) @ u)
    return P[np.argsort(angulos)]


def _marca_recta(p, d1, d2, s):
    """Los tres puntos de la marca de ángulo recto en p, entre las direcciones unitarias d1 y d2, de lado s."""
    return np.array([p + s * d1, p + s * d1 + s * d2, p + s * d2])


def dibuja_proyeccion(x, base, n_cifras=4):
    """Dibuja x, su proyección p sobre W y el residuo r = x - p, con la marca de ángulo recto en p. En R² W es una
    recta; en R³, una recta o un pedazo de plano (sombreado). Para n > 3 no dibuja y lo dice. n_cifras: las cifras
    de la distancia en el título."""
    res = proyeccion_subespacio(x, base)
    A = res["A"]
    n, dim = A.rows, A.cols
    if n > 3 or n == 1:
        print(f"No se dibuja: los vectores son de {_R(n)}" + (" y solo se pueden dibujar los de R² y R³." if n > 3 else
              "; en R¹ no hay nada que proyectar."))
        return res
    if dim == n:
        print(f"No se dibuja: W es todo {_R(n)}, así que p = x y r = 0.")
        return res
    xf, pf, rf = (_flotante(v).ravel() for v in (res["x"], res["p"], res["r"]))
    V = _flotante(A)
    Q = _base_ortonormal(V)
    if len(Q) < dim:
        print("No se dibuja: los vectores de la base son casi dependientes (en decimales no se distinguen de "
              "vectores paralelos), así que el dibujo no sería confiable. Los resultados exactos de arriba sí valen.")
        return res
    L = 1.25 * max(1.0, float(np.abs(np.vstack([xf, pf] + [V[:, j] for j in range(dim)])).max()))
    rn, pn = np.linalg.norm(rf), np.linalg.norm(pf)
    d1 = -pf / pn if pn > 1e-9 * L else Q[0]                  # de p hacia el origen (dentro del triángulo)
    s = 0.06 * L
    if rn > 1e-9 * L:
        s = min(s, 0.4 * rn) if pn <= 1e-9 * L else min(s, 0.4 * rn, 0.4 * pn)
    marca = _marca_recta(pf, d1, rf / rn, s) if rn > 1e-9 * L else None
    corto = lambda v: _tupla(v) if _exacta_legible(v) else "≈ " + _aprox(v, n_cifras)
    d = res["distancia"]
    distancia = f"‖r‖ = {_valor(d, n_cifras)}" if len(_txt(d)) <= 28 else f"‖r‖ ≈ {cifras(d, n_cifras)}"
    titulo = f"Proyección de x = {corto(res['x'])} sobre W: p = {corto(res['p'])}; distancia {distancia}"
    if n == 2:
        fig, ax = plt.subplots(figsize=(6.4, 6.4))
        t = np.array([-2.0, 2.0]) * L
        ax.plot(t * Q[0][0], t * Q[0][1], "-.", color="0.7", lw=1.2, label="W: recta por el origen (gris, delgada)")
        _flecha(ax, (0, 0), xf, _COLORES[0], "-", 2.4)
        _flecha(ax, (0, 0), pf, _COLORES[1], "--", 2.4)
        _flecha(ax, pf, xf, _COLORES[2], ":", 2.4)
        ax.plot(*xf, "o", color="black", ms=7, zorder=5)
        ax.plot(*pf, "s", color="black", ms=6, zorder=5)
        for punta, texto, d in ((xf, "x", (6, 6)), (pf, "p", (8, -14)), ((xf + pf) / 2, "r = x - p", (8, 0))):
            ax.annotate(texto, xy=punta, xytext=d, textcoords="offset points", fontsize=11,
                        bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.85))
        if marca is not None:
            ax.plot(*marca.T, "-", color="black", lw=1.2)
        ax.plot([], [], "-", color=_COLORES[0], lw=2.4, label="x (flecha continua, punta redonda)")
        ax.plot([], [], "--", color=_COLORES[1], lw=2.4, label="p = proyección (discontinua, punta cuadrada)")
        ax.plot([], [], ":", color=_COLORES[2], lw=2.4, label="r = x - p (punteada)")
        ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_aspect("equal")
        ax.axhline(0, color="black", lw=0.6); ax.axvline(0, color="black", lw=0.6)
        ax.grid(alpha=0.4); ax.set_xlabel("componente 1"); ax.set_ylabel("componente 2")
        lugar = "lower right"
    else:
        fig = plt.figure(figsize=(7, 6.6))
        ax = fig.add_subplot(projection="3d")
        if dim == 2:
            ax.add_collection3d(Poly3DCollection([_corte_con_caja(np.cross(Q[0], Q[1]), L)], facecolor="0.6",
                                                 alpha=0.22, edgecolor="0.4", linewidth=1.2))
            ax.plot([], [], "s", color="0.6", alpha=0.5, ms=10, label="W: plano por el origen (sombreado)")
        else:
            tope = L / float(np.max(np.abs(Q[0])))
            P = np.outer([-tope, tope], Q[0])
            ax.plot(*P.T, "-.", color="0.7", lw=1.2, label="W: recta por el origen (gris, delgada)")
        ax.plot(*np.array([np.zeros(3), xf]).T, "-", color=_COLORES[0], lw=2.4, label="x (continua, punta redonda)")
        ax.plot(*np.array([np.zeros(3), pf]).T, "--", color=_COLORES[1], lw=2.4,
                label="p = proyección (discontinua, punta cuadrada)")
        ax.plot(*np.array([pf, xf]).T, ":", color=_COLORES[2], lw=2.5, label="r = x - p (punteada)")
        ax.scatter(*xf, marker="o", s=45, color="black", depthshade=False)
        ax.scatter(*pf, marker="s", s=40, color="black", depthshade=False)
        if marca is not None:
            ax.plot(*marca.T, "-", color="black", lw=1.2)
        ax.text(*xf, "  x", fontsize=11)
        ax.text(*pf, "  p", fontsize=11)
        ax.text(*((xf + pf) / 2), "  r", fontsize=11)
        ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_zlim(-L, L)
        ax.set_xlabel("componente 1"); ax.set_ylabel("componente 2"); ax.set_zlabel("componente 3")
        ax.set_box_aspect((1, 1, 1))                         # misma escala en los tres ejes
        ax.view_init(elev=22, azim=-55)
        lugar = "upper left"
    ax.set_title(textwrap.fill(titulo, 60), fontsize=10)
    ax.legend(loc=lugar, fontsize=8)
    plt.show()
    return res

**Recorrido paso a paso.** (1) $\mathbf x=(2,3,4)$ sobre la recta de $\mathbf u=(1,1,1)$: $\mathbf x\cdot\mathbf u=9$ y $\mathbf u\cdot\mathbf u=3$, así que $\mathbf p=3\,(1,1,1)=(3,3,3)$, $\mathbf r=(-1,0,1)$ y la distancia es $\sqrt2$. (2) $\mathbf x=(1,2,3)$ sobre el plano con base **ortogonal** $\{(1,1,0),(1,-1,1)\}$: basta sumar dos proyecciones, $\mathbf p=(13/6,\,5/6,\,2/3)$, $\mathbf r=\tfrac76(-1,1,2)$ y la distancia es $7/\sqrt6$. (3) El mismo $\mathbf x$ sobre el plano con base $\{(1,0,1),(1,1,0)\}$, que **no** es ortogonal: las ecuaciones normales tienen $A^TA=\begin{bmatrix}2&1\\1&2\end{bmatrix}$ y $A^T\mathbf x=(4,3)$, de donde $\mathbf c=(5/3,\,2/3)$, $\mathbf p=(7/3,\,2/3,\,5/3)$ y $\mathbf r=\tfrac43(-1,1,1)$. Al final, el dibujo de este último caso.

In [ ]:
print("(1) Sobre una recta")
_informe_recta(proyeccion_recta("2, 3, 4", "1, 1, 1"), 4)
print("\n" + "=" * 90 + "\n\n(2) Sobre un plano con base ortogonal")
_p2 = proyeccion_subespacio("1, 2, 3", "1, 1, 0; 1, -1, 1")
_informe_proyeccion(_p2, 4)
print("r = ({})·{}".format(*(lambda c, w: (c, _tupla(w)))(*_factor_comun(_p2["r"]))))
print("\n" + "=" * 90 + "\n\n(3) Sobre un plano con base no ortogonal")
_p3 = proyeccion_subespacio("1, 2, 3", "1, 0, 1; 1, 1, 0")
_informe_proyeccion(_p3, 4, comparar_suma=False)
print("r = ({})·{}".format(*(lambda c, w: (c, _tupla(w)))(*_factor_comun(_p3["r"]))))
dibuja_proyeccion("1, 2, 3", "1, 0, 1; 1, 1, 0");

**Advertencia: con una base que no es ortogonal, sumar las proyecciones no da la proyección.** Con la base $\{(1,0,1),(1,1,0)\}$ del caso (3) y $\mathbf x=(1,2,3)$, la suma de las proyecciones sobre cada vector es $(7/2,\,3/2,\,2)$: está en el plano (es combinación de los dos vectores), pero **no** es $\mathbf p$. Su residuo, $(-5/2,\,1/2,\,1)$, no es perpendicular a $(1,0,1)$ y es más largo que $\mathbf r$. La suma solo funciona con una base ortogonal; para obtenerla está el proceso de Gram-Schmidt (sección 3).

In [ ]:
_s = suma_de_proyecciones("1, 2, 3", "1, 0, 1; 1, 1, 0")
print(f"proy sobre v1 = {_tupla(_s['terminos'][0])},  proy sobre v2 = {_tupla(_s['terminos'][1])}")
print(f"suma = {_tupla(_s['suma'])}   ¿es la proyección p = {_tupla(_s['p'])}? {'sí' if _s['es_la_proyeccion'] else 'NO'}")
print(f"residuo de la suma = x - suma = {_tupla(_s['residuo'])};  su producto con v1 = (1, 0, 1) es "
      f"{_txt(_punto(_s['residuo'], vector('1, 0, 1')))}, no 0")
print(f"‖x - suma‖² = {_valor(_punto(_s['residuo'], _s['residuo']))}  >  ‖r‖² = {_valor(_punto(_p3['r'], _p3['r']))}: "
      "la suma queda más lejos de x que la proyección.")

**Calculadora.** Escribe la base de $W$ **un vector por renglón** (cada renglón se vuelve una columna de $A$) y el vector $\mathbf x$. Con una base ortogonal suma las proyecciones; con una base cualquiera resuelve las ecuaciones normales. En $\mathbb R^2$ y $\mathbb R^3$ dibuja $\mathbf x$, $\mathbf p$ y $\mathbf r$. Prueba agregar un vector dependiente (por ejemplo `2, 0, 2`), un $\mathbf x$ que ya esté en el plano, una recta de $\mathbb R^2$ y un subespacio de $\mathbb R^4$.

In [ ]:
def calculadora_proyeccion(base_txt, x_txt, dibujar, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; el dibujo va después
        try:
            res = proyeccion_subespacio(x_txt, base_txt)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _aviso_coma(base_txt, x_txt)
        _informe_proyeccion(res, n_cifras)
    if dibujar:
        dibuja_proyeccion(res["x"], res["A"], n_cifras)


_CONTROLES_2 = dict(
    base_txt=widgets.Textarea(value="1, 0, 1\n1, 1, 0", description="base de W:", continuous_update=False,
                              layout=widgets.Layout(width="340px", height="120px"), style={"description_width": "initial"}),
    x_txt=widgets.Text(value="1, 2, 3", description="x =", continuous_update=False),
    dibujar=widgets.Checkbox(value=True, description="dibujar (R² y R³)"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_proyeccion, **_CONTROLES_2);

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
_W3 = "1, 0, 1; 1, 1, 0"                       # la base no ortogonal del recorrido
_x3 = vector("1, 2, 3")
PRUEBAS_2 = [
    ("proyeccion_recta((2, 3, 4), (1, 1, 1)) → p = (3, 3, 3), r = (-1, 0, 1), distancia √2",
     lambda: (lambda r: r["p"] == vector("3,3,3") and r["r"] == vector("-1,0,1") and r["distancia"] == sp.sqrt(2)
                        and r["coef"] == 3)(proyeccion_recta("2,3,4", "1,1,1"))),
    ("(1, 2, 3) sobre gen{(1, 1, 0), (1, -1, 1)} (base ortogonal) → p = (13/6, 5/6, 2/3), distancia 7/√6",
     lambda: (lambda r: r["p"] == vector("13/6, 5/6, 2/3") and r["metodo"] == "base ortogonal: suma de proyecciones"
                        and r["r"] == sp.Rational(7, 6) * vector("-1,1,2") and r["distancia"] == 7 / sp.sqrt(6))(
         proyeccion_subespacio("1,2,3", "1,1,0; 1,-1,1"))),
    ("(1, 2, 3) sobre gen{(1, 0, 1), (1, 1, 0)} → c = (5/3, 2/3), p = (7/3, 2/3, 5/3), ecuaciones normales, Aᵀr = 0",
     lambda: (lambda r: r["coeficientes"] == [sp.Rational(5, 3), sp.Rational(2, 3)] and r["p"] == vector("7/3, 2/3, 5/3")
                        and r["metodo"] == "ecuaciones normales" and r["r_ortogonal"]
                        and r["ATA"] == sp.Matrix([[2, 1], [1, 2]]) and r["ATx"] == vector("4, 3"))(
         proyeccion_subespacio("1,2,3", _W3))),
    ("misma base: P² = P, Pᵀ = P y P·x = p",
     lambda: (lambda r: r["P"] * r["P"] == r["P"] and r["P"].T == r["P"] and r["P"] * _x3 == r["p"])(
         proyeccion_subespacio(_x3, _W3))),
    ("suma de proyecciones con la base no ortogonal → (7/2, 3/2, 2), no es la proyección; con la base ortogonal, sí",
     lambda: (lambda s, t: s["suma"] == vector("7/2, 3/2, 2") and not s["es_la_proyeccion"] and t["es_la_proyeccion"])(
         suma_de_proyecciones("1,2,3", _W3), suma_de_proyecciones("1,2,3", "1,1,0; 1,-1,1"))),
    ("(3, 1, 2) ya está en gen{(1, 0, 1), (1, 1, 0)} → p = x, r = 0 y el aviso",
     lambda: (lambda r: _es_nulo(r["r"]) and r["p"] == vector("3,1,2") and r["distancia"] == 0
                        and any("ya está en W" in a for a in r["avisos"]))(proyeccion_subespacio("3,1,2", _W3))),
    ("base con un vector dependiente, (1, 0, 1), (2, 0, 2), (1, 1, 0) → aviso del vector 2 y la misma p",
     lambda: (lambda r: r["p"] == vector("7/3, 2/3, 5/3") and r["usados"] == [1, 3]
                        and "el vector 2 es combinación de los anteriores y no se usa" in r["avisos"])(
         proyeccion_subespacio("1,2,3", "1,0,1; 2,0,2; 1,1,0"))),
    ("en R⁴: (1, 3, 4, 4) sobre gen{(1, 1, 1, 1), (0, 1, 2, 3)} → p = (3/2, 5/2, 7/2, 9/2)",
     lambda: proyeccion_subespacio("1,3,4,4", "1,1,1,1; 0,1,2,3")["p"] == vector("3/2, 5/2, 7/2, 9/2")),
    ("mejor aproximación: tres vectores w de W quedan a distancia ≥ ‖x - p‖ (y mayor si w ≠ p)",
     lambda: (lambda r: all(_punto(_x3 - w, _x3 - w) > _punto(r["r"], r["r"])
                            for w in (2 * vector("1,0,1") + vector("1,1,0"), vector("1,0,1") + vector("1,1,0"),
                                      sp.Rational(3, 2) * vector("1,0,1") + sp.Rational(1, 2) * vector("1,1,0")))
              and _punto(_x3 - r["p"], _x3 - r["p"]) == _punto(r["r"], r["r"]))(proyeccion_subespacio(_x3, _W3))),
    ("proyeccion_recta((1, 2), (0, 0)) → ValueError (u = 0 no define una recta)",
     lambda: "vector cero" in _mensaje(lambda: proyeccion_recta("1,2", "0,0"))),
    ("prueba de humo: la calculadora corre con sus valores iniciales (con y sin dibujo, y con una recta de R²)",
     lambda: _humo(calculadora_proyeccion, _CONTROLES_2)
             and _humo(calculadora_proyeccion, _CONTROLES_2, dibujar=False)
             and _humo(calculadora_proyeccion, _CONTROLES_2, base_txt="2, 1", x_txt="1, 3")),
    ("base casi dependiente (1, 0, 0), (1, 1e-12, 0) con x = (1, 1, 1): no truena, da el resultado exacto "
     "(p = (1, 1, 0), r = (0, 0, 1)) y dice que no se dibuja",
     lambda: (lambda t, e: e is None and "= (1, 1, 0)" in t and "r = x - p = (0, 0, 1)" in t and "No se dibuja" in t
                           and "casi dependientes" in t)(
         *_captura(calculadora_proyeccion, "1, 0, 0\n1, 1e-12, 0", "1, 1, 1", True, 4))),
    ("base con números muy chicos, (1e-15, 0) con x = (1, 1): dibuja la recta sin error y da p = (1, 0)",
     lambda: (lambda t, e: e is None and "= (1, 0)" in t and "No se dibuja" not in t)(
         *_captura(calculadora_proyeccion, "1e-15, 0", "1, 1", True, 4))),
    ("la P de proyeccion_recta es u uᵀ/(uᵀu): con u = (3, 1), P = (1/10)[[9, 3], [3, 1]]",
     lambda: proyeccion_recta("1,2", "3,1")["P"] == sp.Matrix([[9, 3], [3, 1]]) / 10),
    ("es_perpendicular: el residuo del recorrido sí lo es; (1, 0, 0) no es perpendicular a (1, 1, 0)",
     lambda: es_perpendicular(proyeccion_subespacio(_x3, _W3)["r"], _W3)
             and not es_perpendicular("1,0,0", "1,1,0") and es_perpendicular("1,-1,0", "1,1,0")),
    ("un vector cero en la base se avisa como cero, no como combinación: (1, 0, 1), (0, 0, 0), (1, 1, 0)",
     lambda: (lambda r: "el vector 2 es el cero y no se usa" in r["avisos"]
                        and not any("combinación" in a for a in r["avisos"]) and r["usados"] == [1, 3])(
         proyeccion_subespacio("1,2,3", "1,0,1; 0,0,0; 1,1,0"))),
    ("residuo de suma_de_proyecciones con la base no ortogonal: x - suma = (-5/2, 1/2, 1)",
     lambda: suma_de_proyecciones("1,2,3", _W3)["residuo"] == vector("-5/2, 1/2, 1")),
]
corre_pruebas(PRUEBAS_2, 2)

**Contrasta (sección 2).** Calcula a mano la proyección de $\mathbf x=(0,0,6)$ sobre la recta de $\mathbf u=(1,2,2)$: primero $\mathbf x\cdot\mathbf u$ y $\mathbf u\cdot\mathbf u$, después el coeficiente. Escribe `mi_proyeccion = (a, b, c)`; puedes usar fracciones como texto (`"1/3"`) o decimales.

In [ ]:
mi_proyeccion = None     # tres números, por ejemplo (1, -2, 0.5) o ("1/3", "2/3", 1)

if mi_proyeccion is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mia = vector(mi_proyeccion, "tu proyección")
    except ValueError as err:
        print(f"Revisa tu proyección: {str(err).rstrip('.')}.")
    else:
        x_c, u_c = vector("0, 0, 6"), vector("1, 2, 2")          # los datos del enunciado
        mf, xf, uf = (_flotante(v).ravel() for v in (mia, x_c, u_c)) if mia.rows == 3 else (None, None, None)
        if mia.rows != 3:          # respuestas imposibles: la referencia todavía no se muestra
            print(f"x y u son de R³, así que la proyección tiene tres componentes (escribiste {mia.rows}): revisa tu "
                  "respuesta y vuelve a ejecutar la celda.")
        elif np.linalg.norm(mf) > np.linalg.norm(xf) * (1 + 1e-9):
            print(f"La proyección no puede medir más que x (‖x‖ = {cifras(np.linalg.norm(xf), 4)} y tu vector mide "
                  f"{cifras(np.linalg.norm(mf), 4)}). Revisa el coeficiente: se divide entre u·u = ‖u‖², no entre ‖u‖. "
                  "Vuelve a ejecutar la celda.")
        elif np.linalg.norm(mf) < 0.01 * np.linalg.norm(xf):
            print("Tu vector es casi el cero. La proyección sobre la recta es cero solo si x es perpendicular a u: "
                  "calcula x·u, revisa y vuelve a ejecutar la celda.")
        else:
            ref = proyeccion_recta(x_c, u_c)
            pf = _flotante(ref["p"]).ravel()
            print(f"La calculadora da: p = {_tupla(ref['p'], 4)}.")
            paralelo = np.linalg.norm(mf - (mf @ uf) / (uf @ uf) * uf) <= 0.01 * np.linalg.norm(mf)   # relativa
            factor = (mf @ pf) / (pf @ pf)
            if all(abs(float(a - b)) <= 0.01 for a, b in zip(mia, ref["p"])):
                print("Coinciden (con tolerancia de 0.01 por componente).")
            elif paralelo and abs(factor - 1) <= 0.02:
                print("Coinciden salvo redondeo (tu vector difiere de p en menos de 2 %; escribe las fracciones exactas "
                      "si puedes).")
            elif paralelo:
                print(f"NO coinciden: tu vector sí está sobre la recta de u, pero es {cifras(factor, 4)} veces p: "
                      "el coeficiente debe ser (x·u)/(u·u).")
            else:
                print("NO coinciden: la proyección sobre una recta es un múltiplo de u; la tuya no lo es.")
            print("\nEl procedimiento completo:")
            _informe_recta(ref, 4)

## 3. Proceso de Gram-Schmidt

**Idea.** A partir de vectores $\mathbf v_1,\dots,\mathbf v_k$ se construyen vectores **ortogonales** $\mathbf u_1,\dots,\mathbf u_k$ que generan lo mismo: a cada $\mathbf v_j$ se le quita su proyección sobre los $\mathbf u$ anteriores (que ya son ortogonales, así que basta sumar proyecciones, sección 2) y queda solo su parte perpendicular:

$$\mathbf u_1=\mathbf v_1,\qquad \mathbf u_j=\mathbf v_j-\sum_{i<j}\frac{\mathbf v_j\cdot\mathbf u_i}{\mathbf u_i\cdot\mathbf u_i}\,\mathbf u_i,\qquad \mathbf q_j=\frac{\mathbf u_j}{\|\mathbf u_j\|}.$$

- En cada paso, $\operatorname{gen}\{\mathbf u_1,\dots,\mathbf u_j\}=\operatorname{gen}\{\mathbf v_1,\dots,\mathbf v_j\}$, porque $\mathbf u_j$ es $\mathbf v_j$ menos una combinación de los anteriores.
- Si $\mathbf v_j$ es combinación de los anteriores, su parte perpendicular es cero: $\mathbf u_j=\mathbf 0$, se descarta y se sigue con el siguiente. Así, el proceso también detecta dependencia.
- El resultado depende del **orden** de los vectores (el primero se queda como está), pero el subespacio que generan no.
- Los $\mathbf u_j$ son ortogonales; los $\mathbf q_j$ son, además, de longitud 1 (ortonormales). Para no arrastrar raíces en las cuentas, conviene normalizar al final.

**Factorización QR.** Si las columnas de $A=[\,\mathbf v_1\ \cdots\ \mathbf v_k\,]$ son independientes, $A=QR$ con

$$Q=[\,\mathbf q_1\ \cdots\ \mathbf q_k\,]\quad(Q^TQ=I),\qquad R=Q^TA,\quad r_{ij}=\mathbf q_i\cdot\mathbf v_j .$$

$R$ es triangular superior: $r_{ij}=0$ si $i>j$, porque $\mathbf v_j$ está en $\operatorname{gen}\{\mathbf q_1,\dots,\mathbf q_j\}$, que es perpendicular a $\mathbf q_i$. Su diagonal es positiva, $r_{jj}=\|\mathbf u_j\|$, así que $R$ es invertible. Si las columnas son dependientes, algún $\mathbf u_j$ es cero y no hay QR con $R$ invertible.

**Nota numérica.** En decimales, Gram-Schmidt clásico pierde ortogonalidad con vectores casi dependientes: al restar dos vectores casi iguales, el error de redondeo queda del tamaño del resultado. Por eso numpy no lo usa: `np.linalg.qr` trabaja con reflexiones de Householder, que conservan la ortogonalidad, y puede devolver columnas de $Q$ y renglones de $R$ con el signo cambiado (la diagonal de su $R$ puede ser negativa). Cambiar el signo de una columna de $Q$ y del renglón correspondiente de $R$ no cambia el producto $QR$.

In [ ]:
def _resta(c, nombre):
    """El término que se resta, con el signo que toca: c = 1/2 -> '- (1/2)·u1'; c = -2 -> '+ 2·u1'; c = 0 -> '- 0·u1'."""
    return f"- 0·{nombre}" if c == 0 else _termino(-c, nombre, False)


def gram_schmidt(vectores, normalizar=False, n_cifras=4):
    """Gram-Schmidt exacto. Diccionario con u (los u_j distintos de cero, exactos), indices (el j de cada uno: u_j
    viene de v_j), q (los u_j normalizados, exactos, con radicales), normas (‖u_j‖), pasos (textos legibles:
    'u2 = v2 - (1/2)·u1 = (1/2, -1/2, 1)'; con normalizar=True, también 'q1 = u1/‖u1‖ = ...'), dependientes (los j
    de los v_j que dieron u_j = 0, desde 1), coeficientes ({j: [(i, (v_j·u_i)/(u_i·u_i)), ...]}), gram (matriz de
    Gram de los u, diagonal) y A."""
    A = lista_vectores(vectores)
    n, k = A.shape
    us, indices, pasos, dependientes, coefs = [], [], [], [], {}
    for j in range(1, k + 1):
        v = A[:, j - 1]
        cs = [_punto(v, u) / _punto(u, u) for u in us]
        w = v - sum((c * u for c, u in zip(cs, us)), sp.zeros(n, 1))
        coefs[j] = list(zip(indices, cs))
        texto = f"u{j} = v{j}" + "".join(" " + _resta(c, f"u{i}") for i, c in zip(indices, cs)) + \
                f" = {_tupla(w, None if _exacta_legible(w) else n_cifras)}"
        if _es_nulo(w):
            dependientes.append(j)
            texto += (f": v{j} es el vector cero y se descarta" if _es_nulo(v) else
                      f": v{j} es combinación de los anteriores y se descarta")
        else:
            us.append(w)
            indices.append(j)
        pasos.append(texto)
    normas = [sp.sqrt(_punto(u, u)) for u in us]
    q = [u / nu for u, nu in zip(us, normas)]
    if normalizar:
        for i, u, nu, qq in zip(indices, us, normas, q):
            pasos.append(f"q{i} = u{i}/‖u{i}‖, con ‖u{i}‖ = {_valor(nu, n_cifras)}:  q{i} = {_tupla(qq, n_cifras)}")
    U = sp.Matrix.hstack(*us) if us else sp.zeros(n, 0)
    return {"u": us, "indices": indices, "q": q, "normas": normas, "pasos": pasos, "dependientes": dependientes,
            "coeficientes": coefs, "gram": U.T * U, "A": A}


def columnas_ortonormales(Q):
    """True si las columnas de Q son ortonormales, es decir, si QᵀQ = I (comprobado en forma exacta)."""
    Q = sp.Matrix(Q)
    return all(_es_cero(e) for e in Q.T * Q - sp.eye(Q.cols))


def factorizacion_qr(vectores):
    """A = QR exacta, con los vectores como columnas de A: Q = [q1 ... qk] (columnas ortonormales) y R = QᵀA
    (triangular superior, diagonal positiva). Diccionario con Q, R, A, QtQ_es_I y QR_es_A. Si las columnas son
    dependientes, ValueError."""
    gs = gram_schmidt(vectores)
    if gs["dependientes"]:
        j = gs["dependientes"][0]
        raise ValueError(f"las columnas son dependientes; QR con R invertible no existe (v{j} da u{j} = 0)")
    A = gs["A"]
    Q = sp.Matrix.hstack(*gs["q"])
    R = Q.T * A
    for i in range(R.rows):
        for j in range(i):
            if _es_cero(R[i, j]):
                R[i, j] = 0                      # abajo de la diagonal: cero exacto
    return {"Q": Q, "R": R, "A": A, "QtQ_es_I": columnas_ortonormales(Q),
            "QR_es_A": all(_es_cero(e) for e in Q * R - A)}


def compara_numpy_qr(vectores):
    """Compara la QR exacta con la de numpy (np.linalg.qr, reflexiones de Householder). Diccionario con Q_numpy,
    R_numpy, Q_exacta y R_exacta (en punto flotante), signos (el signo de cada entrada de la diagonal de R_numpy),
    coinciden (True si, al cambiar esos signos, numpy da la misma Q y la misma R, con tolerancia 1e-12 relativa)
    y texto (la explicación)."""
    ex = factorizacion_qr(vectores)
    Af = _flotante(ex["A"])
    Qn, Rn = np.linalg.qr(Af)
    Qe, Re = _flotante(ex["Q"]), _flotante(ex["R"])
    signos = np.where(np.diag(Rn) < 0, -1.0, 1.0)
    escala = max(1.0, float(np.abs(Af).max()))
    coinciden = bool(np.allclose(Qn * signos, Qe, rtol=0, atol=1e-12)
                     and np.allclose(signos[:, None] * Rn, Re, rtol=0, atol=1e-12 * escala))
    cambiadas = [j + 1 for j in range(len(signos)) if signos[j] < 0]
    if not cambiadas:
        texto = "numpy dio la misma factorización, con la diagonal de R positiva."
    else:
        cols = _lista(cambiadas)
        texto = (f"numpy usa reflexiones de Householder y aquí dio {'la columna' if len(cambiadas) == 1 else 'las columnas'} "
                 f"{cols} de Q con el signo cambiado, y {'el renglón' if len(cambiadas) == 1 else 'los renglones'} {cols} "
                 "de R también (por eso su diagonal tiene entradas negativas). Al cambiar esos signos, el producto QR no "
                 "cambia y queda la factorización exacta, con la diagonal de R positiva.")
    if not coinciden:
        texto += (" Aun con los signos ajustados quedan diferencias mayores que 1e-12: los vectores están cerca de ser "
                  "dependientes y el redondeo pesa.")
    return {"Q_numpy": Qn, "R_numpy": Rn, "Q_exacta": Qe, "R_exacta": Re, "signos": signos, "coinciden": coinciden,
            "texto": texto}


def _informe_gs(gs, normalizar, n_cifras):
    A = gs["A"]
    _nombra_vectores(A)
    pasos = gs["pasos"]
    k = A.cols
    for j in range(1, k + 1):
        cs = gs["coeficientes"][j]
        if cs:
            print("  " + ",  ".join(f"(v{j}·u{i})/(u{i}·u{i}) = {_valor(c, n_cifras)}" for i, c in cs))
        print(pasos[j - 1])
    if not gs["u"]:
        print("Todos los vectores son el cero: no queda ningún u." if k > 1 else "v1 es el vector cero: no queda ningún u.")
        return
    quedan = len(gs["u"])
    if gs["dependientes"]:
        dep = gs["dependientes"]
        print(f"Se {'descartó' if len(dep) == 1 else 'descartaron'} {_lista([f'v{j}' for j in dep])}: " +
              ("queda 1 vector, que genera" if quedan == 1 else f"quedan {quedan} vectores ortogonales, que generan") +
              f" lo mismo que los {k} dados.")
    nombres = [f"u{i}" for i in gs["indices"]]
    muestra(f"matriz de Gram de {_coma(nombres)}", gs["gram"], n_cifras)
    diagonal = all(gs["gram"][i, j] == 0 for i in range(len(nombres)) for j in range(len(nombres)) if i != j)
    if quedan == 1:
        print(f"Queda un solo vector, {nombres[0]}: no hay pares que revisar.")
    else:
        print(f"La matriz de Gram es diagonal: {_productos_nulos(nombres)}." if diagonal else
              "¡La matriz de Gram NO es diagonal!")
    if normalizar:
        for linea in pasos[k:]:
            print(linea)
        mide = all(_es_cero(_punto(q, q) - 1) for q in gs["q"])
        print((f"q{gs['indices'][0]} mide 1: " if quedan == 1 else "Cada q mide 1: ") + ("sí." if mide else "¡no!"))


def _informe_qr(gs, n_cifras):
    if gs["dependientes"]:
        j = gs["dependientes"][0]
        print(f"QR: las columnas de A son dependientes (v{j} dio u{j} = 0), así que no hay QR con R invertible.")
        return
    qr = factorizacion_qr(gs["A"])
    k = gs["A"].cols
    muestra("Q", qr["Q"], n_cifras)
    muestra("R = QᵀA", qr["R"], n_cifras, tex=r"R=Q^TA")
    diag = ", ".join(f"r{j}{j} = ‖u{j}‖" for j in range(1, k + 1)) if k < 10 else "cada r de la diagonal es la norma de su u"
    print(f"QᵀQ = I: {'sí' if qr['QtQ_es_I'] else 'no'};  QR = A: {'sí' if qr['QR_es_A'] else 'no'};  "
          + ("R es triangular superior con la diagonal positiva" if k > 1 else "R tiene una sola entrada, positiva")
          + f" ({diag}).")
    cmp = compara_numpy_qr(gs["A"])
    print("Comparación con np.linalg.qr (decimales): " + cmp["texto"])
    muestra_decimales("R de numpy", cmp["R_numpy"], n_cifras, tex=r"R_{\text{numpy}}")


def _gs_clasico_flotante(V):
    """Gram-Schmidt clásico en punto flotante (solo para la nota numérica): devuelve Q con columnas 'ortonormales'."""
    Q = []
    for v in V.T:
        w = v - sum((q @ v) * q for q in Q)
        Q.append(w / np.linalg.norm(w))
    return np.column_stack(Q)

**Recorrido paso a paso.** Con $\mathbf v_1=(1,1,0)$, $\mathbf v_2=(1,0,1)$ y $\mathbf v_3=(0,1,1)$: $\mathbf u_1=(1,1,0)$; $\mathbf u_2=\mathbf v_2-\tfrac12\mathbf u_1=(\tfrac12,-\tfrac12,1)$; $\mathbf u_3=\mathbf v_3-\tfrac12\mathbf u_1-\tfrac13\mathbf u_2=(-\tfrac23,\tfrac23,\tfrac23)$, y sus normalizados $\mathbf q_j$. Después, la QR de las dos primeras columnas, con $R=\begin{bmatrix}\sqrt2&\sqrt2/2\\0&\sqrt6/2\end{bmatrix}$, y la comparación con numpy. Al final, con $(2,1,1)$ en lugar de $(0,1,1)$, el tercer vector es $\mathbf v_1+\mathbf v_2$ y da $\mathbf u_3=\mathbf 0$.

In [ ]:
_informe_gs(gram_schmidt("1, 1, 0; 1, 0, 1; 0, 1, 1", normalizar=True), True, 4)
print("\n" + "=" * 90 + "\n\nQR de las dos primeras columnas, A = [v1 v2]:")
_informe_qr(gram_schmidt("1, 1, 0; 1, 0, 1"), 4)
print("\n" + "=" * 90 + "\n\nCon (2, 1, 1) en lugar de (0, 1, 1):")
_informe_gs(gram_schmidt("1, 1, 0; 1, 0, 1; 2, 1, 1"), False, 4)

**La nota numérica, en números.** Los vectores $(1,\varepsilon,0,0)$, $(1,0,\varepsilon,0)$ y $(1,0,0,\varepsilon)$ con $\varepsilon=10^{-8}$ son independientes pero casi iguales. En punto flotante, $1+\varepsilon^2$ se redondea a $1$, y Gram-Schmidt clásico entrega $\mathbf q_2$ y $\mathbf q_3$ que forman un ángulo de unos $60^\circ$ en lugar de $90^\circ$. El cálculo exacto y `np.linalg.qr` no tienen ese problema.

In [ ]:
_eps = sp.Rational(1, 10 ** 8)
_V = lista_vectores([(1, _eps, 0, 0), (1, 0, _eps, 0), (1, 0, 0, _eps)])
_Qc = _gs_clasico_flotante(_flotante(_V))
_Qn = np.linalg.qr(_flotante(_V))[0]
_qe = gram_schmidt(_V)["q"]
print(f"Gram-Schmidt clásico en decimales: q2·q3 ≈ {cifras(_Qc[:, 1] @ _Qc[:, 2], 4)}  (debería ser 0)")
print(f"np.linalg.qr (Householder):        |q2·q3| ≈ {cifras(abs(_Qn[:, 1] @ _Qn[:, 2]), 2)}")
print(f"Gram-Schmidt exacto (fracciones):  q2·q3 = {sp.simplify(_punto(_qe[1], _qe[2]))}")

**Calculadora.** Escribe los vectores **uno por renglón**, en el orden en que quieres ortogonalizarlos. La calculadora muestra cada paso, los $\mathbf u_j$, la matriz de Gram y, si lo pides, los $\mathbf q_j$ y la factorización $A=QR$ comparada con la de numpy. Prueba cambiar el orden de los vectores, agregar uno dependiente y usar vectores de $\mathbb R^4$.

In [ ]:
def calculadora_gram_schmidt(vectores_txt, normalizar, mostrar_qr, n_cifras):
    with bloque():                 # el resultado aparece completo, de una sola vez
        try:
            gs = gram_schmidt(vectores_txt, normalizar, n_cifras)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _aviso_coma(vectores_txt)
        _informe_gs(gs, normalizar, n_cifras)
        if mostrar_qr:
            print()
            _informe_qr(gs, n_cifras)


_CONTROLES_3 = dict(
    vectores_txt=widgets.Textarea(value="1, 1, 0\n1, 0, 1\n0, 1, 1", description="vectores:", continuous_update=False,
                                  layout=widgets.Layout(width="340px", height="130px")),
    normalizar=widgets.Checkbox(value=True, description="normalizar (q)"),
    mostrar_qr=widgets.Checkbox(value=False, description="mostrar QR"),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_gram_schmidt, **_CONTROLES_3);

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
_GS3 = "1, 1, 0; 1, 0, 1; 0, 1, 1"             # los tres vectores del recorrido
PRUEBAS_3 = [
    ("gram_schmidt((1, 1, 0), (1, 0, 1)) → u1 = (1, 1, 0), u2 = (1/2, -1/2, 1)",
     lambda: (lambda g: g["u"][0] == vector("1,1,0") and g["u"][1] == vector("1/2, -1/2, 1"))(
         gram_schmidt("1,1,0; 1,0,1"))),
    ("con (0, 1, 1) como tercer vector: u3 = (-2/3, 2/3, 2/3) y la matriz de Gram es diag(2, 3/2, 4/3)",
     lambda: (lambda g: g["u"][2] == vector("-2/3, 2/3, 2/3")
                        and g["gram"] == sp.diag(2, sp.Rational(3, 2), sp.Rational(4, 3)))(gram_schmidt(_GS3))),
    ("con (2, 1, 1) como tercer vector: dependientes = [3] y quedan dos u",
     lambda: (lambda g: g["dependientes"] == [3] and len(g["u"]) == 2 and g["indices"] == [1, 2])(
         gram_schmidt("1,1,0; 1,0,1; 2,1,1"))),
    ("normalizar: q1 = (1/√2, 1/√2, 0) y cada q mide 1",
     lambda: (lambda g: g["q"][0] == sp.Matrix([1 / sp.sqrt(2), 1 / sp.sqrt(2), 0])
                        and all(sp.simplify(_punto(q, q)) == 1 for q in g["q"]))(gram_schmidt(_GS3, normalizar=True))),
    ("QR de [(1, 1, 0) (1, 0, 1)]: R = [[√2, √2/2], [0, √6/2]], QᵀQ = I y QR = A",
     lambda: (lambda f: sp.simplify(f["R"] - sp.Matrix([[sp.sqrt(2), sp.sqrt(2) / 2], [0, sp.sqrt(6) / 2]]))
                        == sp.zeros(2, 2) and f["QtQ_es_I"] and f["QR_es_A"]
                        and sp.simplify(f["Q"].T * f["Q"]) == sp.eye(2))(factorizacion_qr("1,1,0; 1,0,1"))),
    ("el orden importa: con (1, 0, 1), (1, 1, 0) sale u1 = (1, 0, 1) y otro u2, pero el mismo plano (rango 2)",
     lambda: (lambda g, h: g["u"][0] == vector("1,0,1") and g["u"][1] != h["u"][1]
                           and sp.Matrix.hstack(*g["u"], *h["u"]).rank() == 2)(
         gram_schmidt("1,0,1; 1,1,0"), gram_schmidt("1,1,0; 1,0,1"))),
    ("una base ya ortogonal, (1, 1, 1), (1, -1, 0), (1, 1, -2), no cambia",
     lambda: gram_schmidt("1,1,1; 1,-1,0; 1,1,-2")["u"] == [vector("1,1,1"), vector("1,-1,0"), vector("1,1,-2")]),
    ("en R⁴: gram_schmidt((1, 1, 0, 0), (1, 0, 1, 1)) → u2 = (1/2, -1/2, 1, 1)",
     lambda: gram_schmidt("1,1,0,0; 1,0,1,1")["u"][1] == vector("1/2, -1/2, 1, 1")),
    ("np.linalg.qr del recorrido: |Q de numpy| = |Q exacta| columna por columna (tolerancia 1e-12) y coinciden "
     "salvo signos",
     lambda: (lambda c: np.allclose(np.abs(c["Q_numpy"]), np.abs(c["Q_exacta"]), rtol=0, atol=1e-12)
                        and c["coinciden"])(compara_numpy_qr(_GS3))),
    ("gram_schmidt((0, 0, 0), (1, 0, 0)) → dependientes = [1] y u = [(1, 0, 0)]",
     lambda: (lambda g: g["dependientes"] == [1] and g["u"] == [vector("1,0,0")])(gram_schmidt("0,0,0; 1,0,0"))),
    ("factorizacion_qr((1, 2), (2, 4)) → ValueError «las columnas son dependientes»",
     lambda: _mensaje(lambda: factorizacion_qr("1,2; 2,4")).startswith("las columnas son dependientes")),
    ("prueba de humo: la calculadora corre con sus valores iniciales, también con QR y con un vector dependiente",
     lambda: _humo(calculadora_gram_schmidt, _CONTROLES_3)
             and _humo(calculadora_gram_schmidt, _CONTROLES_3, mostrar_qr=True)
             and _humo(calculadora_gram_schmidt, _CONTROLES_3, vectores_txt="1, 1, 0\n1, 0, 1\n2, 1, 1", mostrar_qr=True)),
    ("los textos de los pasos llevan el signo de la resta: «u2 = v2 - (1/2)·u1 = (1/2, -1/2, 1)»; con coeficiente "
     "negativo, (1, 0), (-1, 1) → «u2 = v2 + 1·u1 = (0, 1)»",
     lambda: gram_schmidt("1,1,0; 1,0,1")["pasos"][1] == "u2 = v2 - (1/2)·u1 = (1/2, -1/2, 1)"
             and gram_schmidt("1,0; -1,1")["pasos"][1] == "u2 = v2 + 1·u1 = (0, 1)"),
    ("compara_numpy_qr con columnas casi dependientes, (1, 1, 1), (1, 1, 1.00000001): coinciden = False",
     lambda: compara_numpy_qr("1,1,1; 1,1,1.00000001")["coinciden"] is False
             and compara_numpy_qr(_GS3)["coinciden"] is True),
    ("columnas_ortonormales: la Q del recorrido sí; [[1, 1], [0, 1], [0, 0]] y [[1, 0], [0, 2], [0, 0]] no",
     lambda: columnas_ortonormales(factorizacion_qr(_GS3)["Q"])
             and not columnas_ortonormales(sp.Matrix([[1, 1], [0, 1], [0, 0]]))
             and not columnas_ortonormales(sp.Matrix([[1, 0], [0, 2], [0, 0]]))),
]
corre_pruebas(PRUEBAS_3, 3)

**Contrasta (sección 3).** Aplica a mano Gram-Schmidt a $\mathbf v_1=(1,0,1)$ y $\mathbf v_2=(0,1,1)$, sin normalizar: $\mathbf u_1=\mathbf v_1$ y $\mathbf u_2=\mathbf v_2-\dfrac{\mathbf v_2\cdot\mathbf u_1}{\mathbf u_1\cdot\mathbf u_1}\,\mathbf u_1$. Escribe `mi_u2 = (a, b, c)`; puedes usar fracciones como texto (`"1/3"`) o decimales.

In [ ]:
mi_u2 = None     # tres números, por ejemplo (1, -2, 0.5) o ("1/3", "2/3", 1)

if mi_u2 is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mio = vector(mi_u2, "tu u2")
    except ValueError as err:
        print(f"Revisa tu u2: {str(err).rstrip('.')}.")
    else:
        if mio.rows != 3:          # respuesta imposible: la referencia todavía no se muestra
            print(f"v1 y v2 son de R³, así que u2 tiene tres componentes (escribiste {mio.rows}): revisa tu respuesta "
                  "y vuelve a ejecutar la celda.")
        elif np.linalg.norm(_flotante(mio).ravel()) < 0.01 * np.linalg.norm(_flotante(vector("0, 1, 1")).ravel()):
            # imposible: v1 y v2 no son paralelos, así que u2 no es (casi) cero
            print("Tu u2 es (casi) el vector cero, y u2 = 0 solo pasa si v2 es múltiplo de v1, que aquí no lo es: "
                  "revisa tu cálculo y vuelve a ejecutar la celda.")
        else:
            ref_gs = gram_schmidt("1, 0, 1; 0, 1, 1")
            ref, v1 = ref_gs["u"][1], ref_gs["A"][:, 0]
            print(f"La calculadora da: u2 = {_tupla(ref, 4)}.")
            mf, rf, v1f = _flotante(mio).ravel(), _flotante(ref).ravel(), _flotante(v1).ravel()
            producto = float(mf @ v1f)
            factor = (mf @ rf) / (rf @ rf)
            multiplo = np.linalg.norm(mf - factor * rf) <= 0.01 * np.linalg.norm(mf)          # tolerancia relativa
            if np.all(np.abs(mf - rf) <= 0.01):
                print("Coinciden (con tolerancia de 0.01 por componente).")
            elif multiplo and abs(factor - 1) <= 0.02:
                print("Coinciden salvo redondeo (tu vector difiere de u2 en menos de 2 %; escribe las fracciones exactas "
                      "si puedes).")
            elif multiplo:
                print(f"Tu vector es un múltiplo de la respuesta del método ({cifras(factor, 4)} veces u2): también "
                      "sirve, porque es perpendicular a v1 y genera, con v1, el mismo plano. El método, sin escalar, "
                      "da el u2 de arriba.")
            elif abs(producto) > 0.01 * np.linalg.norm(mf) * np.linalg.norm(v1f):
                print(f"NO coinciden: tu u2 no es perpendicular a v1, porque u2·v1 ≈ {cifras(producto, 3)} (debería ser 0). "
                      "Revisa el coeficiente (v2·u1)/(u1·u1) y la resta.")
            else:
                print("NO coinciden: tu vector es perpendicular a v1, pero no está en el plano de v1 y v2. u2 debe ser "
                      "v2 menos un múltiplo de u1.")
            print("\nEl procedimiento completo:")
            _informe_gs(ref_gs, False, 4)

## 4. Mínimos cuadrados y ajuste de modelos lineales

**El problema.** Con más ecuaciones que incógnitas, un sistema $A\mathbf x=\mathbf b$ casi nunca tiene solución: $\mathbf b$ no está en $\operatorname{Col}(A)$. *Mínimos cuadrados* busca el $\hat{\mathbf x}$ que hace más pequeño el error $\|\mathbf b-A\mathbf x\|$. Por la sección 2, el punto de $\operatorname{Col}(A)$ más cercano a $\mathbf b$ es su proyección: $A\hat{\mathbf x}=\operatorname{proy}_{\operatorname{Col}A}\mathbf b$, y el residuo $\mathbf r=\mathbf b-A\hat{\mathbf x}$ es perpendicular a cada columna de $A$, es decir, $A^T(\mathbf b-A\hat{\mathbf x})=\mathbf 0$. Esas son las **ecuaciones normales**

$$A^TA\,\hat{\mathbf x}=A^T\mathbf b ,$$

con solución única si las columnas de $A$ son independientes.

**Ajuste de una recta.** Con datos $(x_i,y_i)$, $i=1,\dots,n$, el modelo $y=\beta_0+\beta_1x$ da un renglón $\beta_0+\beta_1x_i=y_i$ por dato: $A$ tiene una columna de unos y la columna de los $x_i$, y las ecuaciones normales son $A^TA\,\boldsymbol\beta=A^T\mathbf y$. Se resuelven en forma cerrada:

$$\beta_1=\frac{S_{xy}}{S_{xx}},\qquad \beta_0=\bar y-\beta_1\bar x,\qquad\text{con}\quad S_{xy}=\sum_i(x_i-\bar x)(y_i-\bar y),\quad S_{xx}=\sum_i(x_i-\bar x)^2 .$$

La recta pasa por $(\bar x,\bar y)$: el renglón de la columna de unos en $A^T\mathbf r=\mathbf 0$ dice que **los residuos suman cero**.

**Modelos lineales en los parámetros.** Lo que debe ser lineal son los coeficientes, no la gráfica: un polinomio $y=\beta_0+\beta_1x+\dots+\beta_gx^g$ (columnas $1,x,\dots,x^g$), una señal $y=\beta_0+\beta_1\operatorname{sen}(\omega t)+\beta_2\cos(\omega t)$ con $\omega$ conocida (columnas $1$, $\operatorname{sen}\omega t$, $\cos\omega t$) o un plano $z=\beta_0+\beta_1x+\beta_2y$ (columnas $1,x,y$).

**Linealización.** $y=a\,e^{kx}$ se vuelve $\ln y=\ln a+kx$, una recta en $\ln y$ (necesita $y>0$); $y=a\,x^b$ se vuelve $\ln y=\ln a+b\ln x$ (necesita $x>0$ y $y>0$). Así se minimiza el error en $\ln y$, **no** en $y$: es otro problema de mínimos cuadrados, y los datos con $y$ pequeño pesan más.

**Medir el ajuste.** Con el residuo $\mathbf r=\mathbf y-\hat{\mathbf y}$ (dato menos predicho):

$$\mathrm{SSE}=\|\mathbf r\|^2,\qquad \mathrm{RMSE}=\sqrt{\mathrm{SSE}/n},\qquad R^2=1-\frac{\mathrm{SSE}}{\mathrm{SST}},\quad \mathrm{SST}=\sum_i(y_i-\bar y)^2 .$$

RMSE está en las unidades de $y$. $R^2$ solo tiene sentido si el modelo tiene ordenada al origen (columna de unos); sin ella puede salir negativo, y la calculadora no lo da.

**Precauciones.**

- Un polinomio de grado $n-1$ pasa por los $n$ datos (residuo cero) y no por eso predice mejor: entre los datos y fuera de ellos puede oscilar mucho.
- Extrapolar (predecir fuera del intervalo de los datos) es riesgoso: nada garantiza que el modelo siga valiendo ahí.
- Un dato atípico pesa mucho, porque su residuo se eleva al cuadrado.
- Si los $x$ están lejos del origen (años como 2003, por ejemplo), las columnas $1,x,x^2,\dots$ son casi paralelas: $A$ está *mal condicionada* y los coeficientes en potencias de $x$ son enormes y casi se cancelan. Conviene usar $t=x-c$ con $c$ un valor central; la calculadora lo avisa y da también esa forma.
- Las bibliotecas no forman $A^TA$, porque su número de condición es el cuadrado del de $A$ y eso empeora el redondeo: `np.linalg.lstsq` resuelve con la descomposición en valores singulares y otras usan QR. `np.polyfit` devuelve los coeficientes **de la potencia más alta a la más baja**, al revés que $\beta_0,\beta_1,\dots$

In [ ]:
_MODELOS = ["recta", "recta por el origen", "polinomio", "exponencial", "potencia"]
_MAX_EXACTO = 12                 # con hasta 12 datos (y grado hasta 3) el cálculo es exacto
_COND_ALTA = 1e8                 # número de condición de A a partir del cual se avisa


class _Desborde(ValueError):
    """La predicción no cabe en los números decimales."""


def _modelo(modelo):
    """El nombre del modelo (acepta mayúsculas y acentos): uno de _MODELOS."""
    t = normaliza_texto(modelo) if isinstance(modelo, str) else ""
    for m in _MODELOS:
        if t == normaliza_texto(m):
            return m
    raise ValueError(f"el modelo debe ser uno de: {', '.join(_MODELOS)} (escribiste «{modelo}»)")


def _grado(grado):
    """El grado del polinomio: un entero de 1 a 6."""
    if grado is None:
        raise ValueError("indica el grado del polinomio, un entero de 1 a 6")
    g = numero(grado, "el grado")
    if not g.is_integer or not 1 <= g <= 6:
        raise ValueError(f"el grado del polinomio debe ser un entero de 1 a 6 (escribiste {_txt(g)})")
    return int(g)


def _potencia(letra, p):
    """'' (p = 0), 'x', 'x²', 'x³', ...: la misma escritura en la forma del modelo y en la ecuación."""
    return "" if p == 0 else (letra if p == 1 else f"{letra}{str(p).translate(_SUP)}")


def _forma(m, grado=1):
    """La forma del modelo con sus coeficientes: 'y = β0 + β1 x + β2 x²', 'z = β0 + β1 x + β2 y', ..."""
    if m == "recta":
        return "y = β0 + β1 x"
    if m == "recta por el origen":
        return "y = β1 x"
    if m == "polinomio":
        return "y = " + " + ".join(f"β{p}" + (f" {_potencia('x', p)}" if p else "") for p in range(grado + 1))
    if m == "exponencial":
        return "y = a·e^(k x), linealizado como ln y = ln a + k x"
    if m == "potencia":
        return "y = a·x^b, linealizado como ln y = ln a + b ln x"
    return "z = β0 + β1 x + β2 y"


def _exacto_en(c, texto):
    """True si el texto (un decimal) es exactamente el racional c, es decir, si se muestra sin redondear."""
    try:
        return isinstance(c, sp.Rational) and Fraction(texto) == Fraction(int(c.p), int(c.q))
    except (ValueError, ZeroDivisionError):
        return False


def _coef_txt(c, n):
    """Un coeficiente: '10.02' (decimal exacto), '31/14 ≈ 2.214' (fracción corta) o '≈ 9.941' (redondeado)."""
    if isinstance(c, sp.Rational):
        t = _num(c, n)
        if _exacto_en(c, t):
            return t
        if len(_txt(c)) <= 20:
            return f"{_txt(c)} ≈ {cifras(c, n)}"
    return f"≈ {cifras(c, n)}"


def _medida_txt(c, n):
    """SSE, RMSE o R²: el decimal exacto si es corto ('0.099'); si no, redondeado ('≈ 0.1407')."""
    t = _num(c, n) if isinstance(c, sp.Rational) else None
    return t if t is not None and _exacto_en(c, t) else f"≈ {cifras(c, n)}"


def _decimales_escritos(textos):
    """El mayor número de decimales con que se escribieron los datos ('10.0' -> 1, '3' -> 0), o None si alguno no
    es un decimal simple (fracción o notación científica)."""
    d = 0
    for t in textos:
        m = re.fullmatch(r"[+-]?\d*(?:\.(\d*))?", str(t).strip())
        if not m:
            return None
        d = max(d, len(m.group(1) or ""))
    return d


def _fijo(v, d):
    """v (racional exacto o decimal) redondeado (5 sube) a d decimales fijos: (14, 1) -> '14.0'; sin '-0.00'."""
    with localcontext() as ctx:
        ctx.prec = 60
        x = (Decimal(int(v.p)) / Decimal(int(v.q))) if isinstance(v, sp.Rational) else Decimal(repr(float(v)))
        t = format(x.quantize(Decimal(1).scaleb(-d), rounding=ROUND_HALF_UP), "f")
    return t[1:] if t.startswith("-") and float(t) == 0 else t


def _decimales_exactos(v):
    """Decimales de un racional con decimal finito (3/4 -> 2), o None si no es finito (1/3) o no es racional."""
    if not isinstance(v, sp.Rational):
        return None
    q, a, b = int(v.q), 0, 0
    while q % 2 == 0:
        q, a = q // 2, a + 1
    while q % 5 == 0:
        q, b = q // 5, b + 1
    return max(a, b) if q == 1 else None


def _columna(valores, n):
    """Textos de una columna calculada (predichos o residuos), todos con los mismos decimales: si todos son decimales
    exactos que caben, tal cual (10.02, 14.00, -0.20); si no, redondeados a n cifras significativas del mayor. Con
    valores muy grandes o muy chicos, cada uno con n cifras y notación ×10^."""
    grande = max(abs(float(v)) for v in valores)
    if grande == 0:
        return ["0"] * len(valores)
    d_sig = n - 1 - math.floor(math.log10(grande))
    exactos = [_decimales_exactos(v) for v in valores]
    if all(e is not None for e in exactos) and max(exactos) <= max(d_sig, 0) and grande < 1e15:
        return [_fijo(v, max(exactos)) for v in valores]
    if grande >= 10 ** n or grande < 1e-4:
        return [cifras(v, n) for v in valores]
    return [_fijo(v, d_sig) for v in valores]


def _como_datos(v, textos, n):
    """Un valor calculado con los decimales de los datos escritos (ȳ = 14 con datos '10.0' -> '14.0'), si cabe
    exacto; si no, redondeado a n cifras. Devuelve (texto, exacto)."""
    d = _decimales_escritos(textos)
    e = _decimales_exactos(v)
    if d is not None and e is not None and e <= d:
        return _fijo(v, d), True
    t = _num(v, n)
    return t, _exacto_en(v, t)


def _redondea(c, m):
    """c redondeado (5 sube) a m cifras significativas, como número decimal."""
    with localcontext() as ctx:
        ctx.prec = 60
        if isinstance(c, sp.Rational):
            if c == 0:
                return 0.0
            d = Decimal(int(c.p)) / Decimal(int(c.q))
        else:
            v = float(c)
            if v == 0 or not math.isfinite(v):
                return v
            d = Decimal(repr(v))
        return float(d.quantize(Decimal(1).scaleb(d.adjusted() - m + 1), rounding=ROUND_HALF_UP))


def _busca_cifras(n, evalua, coefs, pred):
    """Las cifras (de n a 15) con las que una ecuación reproduce el modelo: con sus coeficientes redondeados a esas
    cifras, los predichos cambian menos que 10^-n veces el mayor de ellos. Devuelve (cifras, reproduce)."""
    pred = np.asarray(pred, dtype=float)
    escala = max(float(np.max(np.abs(pred))), 1e-300)
    for m in range(n, 16):
        with np.errstate(all="ignore"):
            v = np.asarray(evalua([_redondea(c, m) for c in coefs]), dtype=float)
        if np.all(np.isfinite(v)) and float(np.max(np.abs(v - pred))) <= 10.0 ** (-n) * escala:
            return m, True
    return n, False


def _ecuacion_lineal(izq, coef, monomios, m):
    """'y = 10.02 + 0.0199 x', 'y = 1 + x²', 'z ≈ 1.02 + 0.55 x + 1.95 y': coeficientes con m cifras; '≈' si alguno
    se redondeó. Un cero exacto no se escribe."""
    exactos, partes = True, []
    for c, mono in zip(coef, monomios):
        if isinstance(c, sp.Rational) and c == 0:
            continue
        negativo = float(c) < 0
        mag = -c if negativo else c
        t = _num(mag, m)
        exactos &= _exacto_en(mag, t)
        cuerpo = mono if (mono and t == "1") else (f"{t} {mono}" if mono else t)
        partes.append((("- " if partes else "-") if negativo else ("+ " if partes else "")) + cuerpo)
    return f"{izq} {'=' if exactos else '≈'} {' '.join(partes) if partes else '0'}"


def _ecuacion_log(m_modelo, a, k, L, m, letra="x"):
    """Exponencial y ≈ a·e^(k x) o potencia y ≈ a·x^b; si |ln a| > 700 (a no cabe en los decimales), en la forma
    y ≈ e^(ln a + k x) o y ≈ e^(ln a + b ln x)."""
    if abs(L) > 700:
        resto = f"{_num(abs(k), m)} {letra}" if m_modelo == "exponencial" else f"{_num(abs(k), m)} ln {letra}"
        return f"y ≈ e^({_num(L, m)} {'-' if k < 0 else '+'} {resto})"
    if m_modelo == "exponencial":
        return f"y ≈ {_num(a, m)}·e^({_num(k, m)} {letra})"
    return f"y ≈ {_num(a, m)}·{letra}^{_num(k, m)}"


def _a_potencias(d, c, h):
    """Coeficientes en potencias de x del polinomio d0 + d1·s + ... + dg·s^g con s = (x - c)/h. La conversión se
    hace con fracciones exactas a partir de los decimales de d, c y h; solo sirve para MOSTRAR los coeficientes:
    para evaluar el modelo se usa la forma en s, que no pierde cifras."""
    C, H = Fraction(c), Fraction(h)
    b = [Fraction(0)] * len(d)
    for p, dp in enumerate(d):
        D = Fraction(float(dp)) / H ** p
        for k in range(p + 1):
            b[k] += D * math.comb(p, k) * (-C) ** (p - k)
    return np.array([float(v) for v in b])


def _fraccion(c):
    return Fraction(int(c.p), int(c.q)) if isinstance(c, sp.Rational) else Fraction(c)


def _centro(valores, textos):
    """Un valor central 'redondo' para t = x - c: el promedio, redondeado a los decimales de los datos escritos."""
    media = sum(valores) / len(valores)
    d = _decimales_escritos(textos)
    with localcontext() as ctx:
        ctx.prec = 60
        x = Decimal(int(media.p)) / Decimal(int(media.q))
        q = x.quantize(Decimal(1).scaleb(-(d if d is not None else 4)), rounding=ROUND_HALF_UP)
    return sp.Rational(str(q))


def _medidas(y, pred, n, con_ordenada, exacto):
    """SSE, RMSE y R² (None sin ordenada al origen o si todos los y son iguales), exactos o en decimales."""
    if exacto:
        r = [a - b for a, b in zip(y, pred)]
        sse = sum(e ** 2 for e in r)
        media = sum(y) / n
        sst = sum((v - media) ** 2 for v in y)
        return sse, sp.sqrt(sse / n), (1 - sse / sst if con_ordenada and sst != 0 else None), sst
    r = np.asarray(y, float) - np.asarray(pred, float)
    sse = float(r @ r)
    yy = np.asarray(y, float)
    sst = float(((yy - yy.mean()) ** 2).sum())
    return sse, math.sqrt(sse / n), (1 - sse / sst if con_ordenada and sst > 0 else None), sst


def _revisa_finito(*arreglos):
    for a in arreglos:
        if not np.all(np.isfinite(np.asarray(a, dtype=float))):
            raise ValueError("los datos dan valores fuera del intervalo que maneja esta calculadora (desbordamiento): "
                             "usa números más pequeños o cambia de modelo")


def _aviso_cond(m, cond, n_cifras, centrada):
    """El aviso de A mal condicionada, con el consejo que corresponde al modelo; centrada dice si abajo va la forma
    centrada."""
    inicio = f"la matriz A está mal condicionada (número de condición ≈ {cifras(cond, n_cifras)}): "
    if m == "potencia":
        return (inicio + "los valores de ln x están muy juntos; cambia la unidad o la escala de x (por ejemplo, x/1000) "
                "o usa otro modelo")
    texto = inicio + ("los coeficientes son muy sensibles al redondeo" if m == "plano" else
                      "los coeficientes en potencias de x son muy sensibles al redondeo y casi se cancelan")
    if centrada:
        return texto + ("; abajo va también la forma centrada, que es la que conviene usar" if m == "plano" else
                        "; abajo va también la forma con t = x - c, con c un valor central, que es la que conviene usar")
    return texto + ("; conviene restar a x y a y valores centrales" if m == "plano" else
                    "; conviene restar a x un valor central (t = x - c)")


def ajuste(x, y, modelo, grado=None, n_cifras=4):
    """Ajuste por mínimos cuadrados de y contra x. modelo: 'recta', 'recta por el origen', 'polinomio' (con grado de
    1 a 6), 'exponencial' (y = a·e^(k x), exige y > 0) o 'potencia' (y = a·x^b, exige x > 0 y y > 0); el grado solo se
    usa con 'polinomio'. Con datos racionales, hasta 12 datos y grado hasta 3 (recta, recta por el origen o polinomio)
    el cálculo es EXACTO con las ecuaciones normales AᵀA β = Aᵀy; si no, en decimales con np.linalg.lstsq (con x
    centrado y escalado). Diccionario con A, ATA, ATy, coeficientes (β0, β1, ... o a y k / a y b), nombres, ecuacion
    (con las cifras necesarias para que reproduzca el modelo, al menos n_cifras), ecuacion_centrada (None, o la forma
    con t = x - c si A está mal condicionada), cond (número de condición de A), predichos, residuos, SSE, RMSE, R2
    (None sin ordenada al origen), avisos, exacto, modelo, x, y (y sus textos x_txt, y_txt), n y potencias. Para
    'exponencial' y 'potencia', A, ATA y ATy son los del problema en ln y, y R2 se calcula para ln y. Si las columnas
    de A son dependientes (todos los x iguales, o grado >= número de x distintos), ValueError «las columnas de A son
    dependientes: ...»."""
    m = _modelo(modelo)
    xs, x_txt = datos(x, "los datos de x", con_texto=True)
    ys, y_txt = datos(y, "los datos de y", con_texto=True)
    if len(xs) != len(ys):
        raise ValueError(f"hay {_cuenta(len(xs), 'dato', 'datos')} de x y {len(ys)} de y: debe haber un y por cada x"
                         f"{pista_coma(x) or pista_coma(y)}")
    n = len(xs)
    potencias = list(range(_grado(grado) + 1)) if m == "polinomio" else ([1] if m == "recta por el origen" else [0, 1])
    if m in ("exponencial", "potencia"):
        for lista, letra in ((ys, "y"), (xs, "x")) if m == "potencia" else ((ys, "y"),):
            malos = [j for j, v in enumerate(lista, 1) if v <= 0]
            if malos:
                forma = "y = a·e^(k x)" if m == "exponencial" else "y = a·x^b"
                raise ValueError(f"el modelo {m} {forma} necesita todos los {letra} positivos, porque se toma ln {letra}: "
                                 f"el dato {malos[0]} tiene {letra} = {(x_txt if letra == 'x' else y_txt)[malos[0] - 1]}")
    distintos = len(set(xs))
    if m == "recta por el origen":
        if all(v == 0 for v in xs):
            raise ValueError("las columnas de A son dependientes: todos los x son 0, así que la única columna de A es cero")
    elif distintos < len(potencias):
        detalle = ("todos los x son iguales, así que la columna de x es un múltiplo de la columna de unos" if distintos == 1
                   else f"con {distintos} valores distintos de x, el grado puede ser a lo más {distintos - 1}")
        raise ValueError("las columnas de A son dependientes: " + detalle)
    avisos = []
    if pista_coma(x) or pista_coma(y):
        avisos.append("hay comas entre dígitos (1,5) y números separados solo por espacios; si escribiste decimales con "
                      "coma, usa punto decimal (1.5, no 1,5): la calculadora leyó cada coma como separador")
    if len(potencias) == n:
        avisos.append({"polinomio": "el polinomio", "recta": "la recta"}.get(m, "el modelo") +
                      " pasa por todos los datos; residuo cero no significa mejor modelo")
    con_ordenada = 0 in potencias
    exacto = m in ("recta", "recta por el origen", "polinomio") and n <= _MAX_EXACTO and max(potencias) <= 3
    coef_ln, escala = None, None
    X = np.array([float(v) for v in xs])
    if exacto:
        A = sp.Matrix([[xi ** p for p in potencias] for xi in xs])
        Y = sp.Matrix(ys)
        ATA, ATy = A.T * A, A.T * Y
        coef = list(ATA.LUsolve(ATy))
        predichos = list(A * sp.Matrix(coef))
        residuos = [a - b for a, b in zip(ys, predichos)]
        sse, rmse, r2, sst = _medidas(ys, predichos, n, con_ordenada, True)
        cond = float(np.linalg.cond(_flotante(A)))
    else:
        Yv = np.array([float(v) for v in ys])
        Xl = np.log(X) if m == "potencia" else X
        Yl = np.log(Yv) if m in ("exponencial", "potencia") else Yv
        with np.errstate(all="ignore"):
            A = np.column_stack([Xl ** p for p in potencias])
            _revisa_finito(A)
            ATA, ATy = A.T @ A, A.T @ Yl
            cond = float(np.linalg.cond(A))
            if con_ordenada:                      # x centrado y escalado: mismas columnas generadas, bien condicionadas
                c0 = float(np.mean(Xl))
                h = float(np.max(np.abs(Xl - c0))) or 1.0
                S = np.column_stack([((Xl - c0) / h) ** p for p in potencias])
                d = np.linalg.lstsq(S, Yl, rcond=None)[0]
                escala = (c0, h, [float(v) for v in d])
                cl = _a_potencias(d, c0, h)
                ajuste_l = S @ d
            else:
                cl = np.linalg.lstsq(A, Yl, rcond=None)[0]
                ajuste_l = A @ cl
            if m in ("exponencial", "potencia"):
                coef_ln = [float(v) for v in cl]
                coef = [float(np.exp(cl[0])), float(cl[1])]
                predichos = np.exp(ajuste_l)
                avisos.append("el ajuste se hace con ln y (una recta en ln y): minimiza el error en ln y, no en y; "
                              "R² se calcula para ln y")
                if abs(coef_ln[0]) > 700:
                    consejo = ("mide x desde otro origen (abajo va la forma con t = x - c)" if m == "exponencial" else
                               "cambia la unidad o la escala de x (por ejemplo, x/1000) o usa otro modelo")
                    avisos.append(f"a = e^(ln a), con ln a ≈ {cifras(coef_ln[0], n_cifras)}, no cabe en los números "
                                  f"decimales (o casi): {consejo}")
                _revisa_finito(coef[1:], predichos, ATA, ATy)
            else:
                coef = [float(v) for v in cl]
                predichos = ajuste_l
                _revisa_finito(coef, predichos, ATA, ATy)
        predichos = [float(v) for v in predichos]
        residuos = [float(a) - b for a, b in zip(ys, predichos)]
        sse, rmse, r2, sst = _medidas(ys, predichos, n, con_ordenada, False)
        if m in ("exponencial", "potencia"):
            _, _, r2, _ = _medidas(list(Yl), list(ajuste_l), n, True, False)
    if con_ordenada and (sst == 0) and m not in ("exponencial", "potencia"):
        avisos.append("todos los y son iguales: R² no está definido")
    # ecuación en x, con las cifras que hagan falta para reproducir el modelo
    pf = [float(v) for v in predichos]
    if m in ("exponencial", "potencia"):
        lx = np.log(X) if m == "potencia" else X
        if abs(coef_ln[0]) > 700:
            cifras_ec, reproduce = _busca_cifras(n_cifras, lambda c: np.exp(c[0] + c[1] * lx), coef_ln, pf)
        else:
            cifras_ec, reproduce = _busca_cifras(n_cifras, lambda c: c[0] * np.exp(c[1] * lx), coef, pf)
        ecuacion = _ecuacion_log(m, coef[0], coef[1], coef_ln[0], cifras_ec)
    else:
        cifras_ec, reproduce = _busca_cifras(n_cifras, lambda c: sum(ci * X ** p for ci, p in zip(c, potencias)), coef, pf)
        ecuacion = _ecuacion_lineal("y", coef, [_potencia("x", p) for p in potencias], cifras_ec)
    # forma centrada t = x - c, si se lee mejor que la de x
    ecuacion_centrada = None
    if m != "potencia" and con_ordenada and (cond > _COND_ALTA or cifras_ec > n_cifras or not reproduce
                                            or (coef_ln is not None and abs(coef_ln[0]) > 700)):
        c = _centro(xs, x_txt)
        T = X - float(c)
        if m == "exponencial":
            c0, h, d = escala
            Lc, kc = d[0] + d[1] * (float(c) - c0) / h, d[1] / h
            cc, rep = _busca_cifras(n_cifras, lambda q: q[0] * np.exp(q[1] * T), [math.exp(Lc), kc], pf)
            if rep and (cc < cifras_ec or not reproduce or abs(coef_ln[0]) > 700):
                ecuacion_centrada = _ecuacion_log(m, math.exp(Lc), kc, Lc, cc, "t") + f", con t = x - {_num(c, 15)}"
        else:
            if exacto:
                gam = [sum(coef[k] * sp.binomial(k, j) * c ** (k - j) for k in range(j, len(potencias)))
                       for j in range(len(potencias))]
            else:
                c0, h, d = escala
                gam = list(_a_potencias(d, Fraction(c0) - _fraccion(c), h))
            cc, rep = _busca_cifras(n_cifras, lambda q: sum(qi * T ** p for p, qi in enumerate(q)), gam, pf)
            if rep and (cc < cifras_ec or not reproduce):
                ecuacion_centrada = (_ecuacion_lineal("y", gam, [_potencia("t", p) for p in potencias], cc) +
                                     f", con t = x - {_num(c, 15)}")
    if cond > _COND_ALTA:
        avisos.append(_aviso_cond(m, cond, n_cifras, ecuacion_centrada is not None))
    if cifras_ec > n_cifras:
        avisos.append(f"la ecuación se escribe con {cifras_ec} cifras, y no con {n_cifras}, para que reproduzca el "
                      "modelo: sus términos casi se cancelan")
    if not reproduce:
        avisos.append("con los coeficientes redondeados, la ecuación en x no reproduce el modelo (sus términos casi se "
                      "cancelan)" + (": usa la forma centrada" if ecuacion_centrada else ""))
    nombres = {"exponencial": ["a", "k"], "potencia": ["a", "b"]}.get(m, [f"β{p}" for p in potencias])
    return {"modelo": m, "grado": max(potencias), "x": xs, "y": ys, "x_txt": x_txt, "y_txt": y_txt, "n": n, "A": A,
            "ATA": ATA, "ATy": ATy, "coeficientes": coef, "coeficientes_ln": coef_ln, "nombres": nombres,
            "ecuacion": ecuacion, "cifras_ecuacion": cifras_ec, "ecuacion_reproduce": reproduce,
            "ecuacion_centrada": ecuacion_centrada, "cond": cond, "predichos": predichos, "residuos": residuos,
            "SSE": sse, "RMSE": rmse, "R2": r2, "avisos": avisos, "exacto": exacto, "potencias": potencias,
            "escala": escala}


def ajuste_plano(x, y, z, n_cifras=4):
    """Ajuste del plano z = β0 + β1 x + β2 y por mínimos cuadrados (exacto con hasta 12 datos; con más, en decimales
    con x y y centrados y escalados). El mismo diccionario que ajuste(), con modelo 'plano', y_datos (los y) y z.
    Si los puntos (x, y) están sobre una recta, ValueError."""
    xs, x_txt = datos(x, "los datos de x", con_texto=True)
    ys, y_txt = datos(y, "los datos de y", con_texto=True)
    zs, z_txt = datos(z, "los datos de z", con_texto=True)
    if not len(xs) == len(ys) == len(zs):
        raise ValueError(f"hay {_cuenta(len(xs), 'dato', 'datos')} de x, {len(ys)} de y y {len(zs)} de z: deben ser "
                         "los mismos")
    n = len(xs)
    G = sp.Matrix([[n, sum(xs), sum(ys)], [sum(xs), sum(a * a for a in xs), sum(a * b for a, b in zip(xs, ys))],
                   [sum(ys), sum(a * b for a, b in zip(xs, ys)), sum(b * b for b in ys)]])
    if G.det() == 0:
        raise ValueError("las columnas de A son dependientes: los puntos (x, y) están todos sobre una misma recta (o son "
                         "el mismo punto), así que el plano no queda determinado")
    avisos = ["el plano pasa por los tres datos; residuo cero no significa mejor modelo"] if n == 3 else []
    exacto = n <= _MAX_EXACTO
    X, Yd = np.array([float(v) for v in xs]), np.array([float(v) for v in ys])
    escala = None
    if exacto:
        A = sp.Matrix([[1, a, b] for a, b in zip(xs, ys)])
        Z = sp.Matrix(zs)
        ATA, ATy = A.T * A, A.T * Z
        coef = list(ATA.LUsolve(ATy))
        predichos = list(A * sp.Matrix(coef))
        residuos = [a - b for a, b in zip(zs, predichos)]
        cond = float(np.linalg.cond(_flotante(A)))
    else:
        A = np.column_stack([np.ones(n), X, Yd])
        Zf = np.array([float(v) for v in zs])
        ATA, ATy = A.T @ A, A.T @ Zf
        cond = float(np.linalg.cond(A))
        cx, cy = float(X.mean()), float(Yd.mean())
        hx, hy = float(np.max(np.abs(X - cx))) or 1.0, float(np.max(np.abs(Yd - cy))) or 1.0
        S = np.column_stack([np.ones(n), (X - cx) / hx, (Yd - cy) / hy])
        d = np.linalg.lstsq(S, Zf, rcond=None)[0]
        b1, b2 = Fraction(float(d[1])) / Fraction(hx), Fraction(float(d[2])) / Fraction(hy)
        coef = [float(Fraction(float(d[0])) - b1 * Fraction(cx) - b2 * Fraction(cy)), float(b1), float(b2)]
        predichos = [float(v) for v in S @ d]
        escala = (cx, hx, cy, hy, [float(v) for v in d])
        _revisa_finito(coef, predichos)
        residuos = [float(a) - b for a, b in zip(zs, predichos)]
    sse, rmse, r2, sst = _medidas(zs, predichos, n, True, exacto)
    if sst == 0:
        avisos.append("todos los z son iguales: R² no está definido")
    Xc0, Yc0 = X - X.mean(), Yd - Yd.mean()
    colineal = float(np.linalg.cond(np.column_stack([Xc0 / np.linalg.norm(Xc0), Yc0 / np.linalg.norm(Yc0)])))
    if colineal > 1e3:
        avisos.append("los puntos (x, y) están casi sobre una recta: el plano queda mal determinado y sus coeficientes "
                      "son muy sensibles a los datos (un cambio pequeño en un dato los cambia mucho)")
    pf = [float(v) for v in predichos]
    cifras_ec, reproduce = _busca_cifras(n_cifras, lambda c: c[0] + c[1] * X + c[2] * Yd, coef, pf)
    ecuacion_centrada = None
    if cond > _COND_ALTA or cifras_ec > n_cifras or not reproduce:
        cx_, cy_ = _centro(xs, x_txt), _centro(ys, y_txt)
        g0 = (coef[0] + coef[1] * cx_ + coef[2] * cy_) if exacto else float(
            Fraction(coef[0]) + Fraction(coef[1]) * _fraccion(cx_) + Fraction(coef[2]) * _fraccion(cy_))
        gam = [g0, coef[1], coef[2]]
        Xc, Yc = X - float(cx_), Yd - float(cy_)
        cc, rep = _busca_cifras(n_cifras, lambda q: q[0] + q[1] * Xc + q[2] * Yc, gam, pf)
        if rep and (cc < cifras_ec or not reproduce):
            ecuacion_centrada = _ecuacion_lineal("z", gam, ["", f"(x - {_num(cx_, 15)})", f"(y - {_num(cy_, 15)})"], cc)
    if cond > _COND_ALTA:
        avisos.append(_aviso_cond("plano", cond, n_cifras, ecuacion_centrada is not None))
    if cifras_ec > n_cifras:
        avisos.append(f"la ecuación se escribe con {cifras_ec} cifras, y no con {n_cifras}, para que reproduzca el "
                      "modelo: sus términos casi se cancelan")
    if not reproduce:
        avisos.append("con los coeficientes redondeados, la ecuación no reproduce el modelo"
                      + (": usa la forma centrada" if ecuacion_centrada else ""))
    return {"modelo": "plano", "grado": 1, "x": xs, "y_datos": ys, "y": zs, "z": zs, "x_txt": x_txt, "yd_txt": y_txt,
            "y_txt": z_txt, "n": n, "A": A, "ATA": ATA, "ATy": ATy, "coeficientes": coef, "coeficientes_ln": None,
            "nombres": ["β0", "β1", "β2"], "ecuacion": _ecuacion_lineal("z", coef, ["", "x", "y"], cifras_ec),
            "cifras_ecuacion": cifras_ec, "ecuacion_reproduce": reproduce, "ecuacion_centrada": ecuacion_centrada,
            "cond": cond, "predichos": predichos, "residuos": residuos, "SSE": sse, "RMSE": rmse, "R2": r2,
            "avisos": avisos, "exacto": exacto, "potencias": None, "escala": escala}


def _evalua_flotante(res, t):
    """El modelo evaluado en un arreglo t (para dibujar y predecir en decimales); lo que se desborda queda como nan."""
    m, c = res["modelo"], [float(v) for v in res["coeficientes"]]
    t = np.asarray(t, dtype=float)
    with np.errstate(all="ignore"):
        tl = np.log(t) if m == "potencia" else t
        if res.get("escala"):                       # la forma en s = (x - c)/h, sin pérdida de cifras
            c0, h, d = res["escala"]
            s = (tl - c0) / h
            v = sum(dp * s ** p for p, dp in enumerate(d))
        elif m in ("exponencial", "potencia"):
            v = res["coeficientes_ln"][0] + c[1] * tl
        else:
            v = sum(ci * tl ** p for ci, p in zip(c, res["potencias"]))
        if m in ("exponencial", "potencia"):
            v = np.exp(v)
    v = np.array(v, dtype=float)
    v[~np.isfinite(v)] = np.nan
    return v


def _evalua(res, x0):
    """El modelo evaluado en x0: exacto si el ajuste fue exacto; si no, en decimales (_Desborde si no cabe)."""
    if res["exacto"]:
        return sum(ci * x0 ** p for ci, p in zip(res["coeficientes"], res["potencias"]))
    v = float(_evalua_flotante(res, np.array([float(x0)]))[0])
    if not math.isfinite(v):
        raise _Desborde(f"la predicción en x0 = {_num(x0, 15)} se desborda: el valor no cabe en los números decimales")
    return v


def predice(resultado, x0):
    """Predicción del modelo en x0: diccionario con x0, y (exacto si el ajuste fue exacto) y aviso (None, o
    'extrapolación: x0 = ... está fuera del intervalo de los datos [min, max]...')."""
    if resultado["modelo"] == "plano":
        raise ValueError("predice es para los modelos de una variable; para el plano evalúa z = β0 + β1 x + β2 y")
    x0 = numero(x0, "x0")
    if resultado["modelo"] == "potencia" and x0 <= 0:
        raise ValueError("el modelo potencia y = a·x^b solo se evalúa en x positivos")
    lo, hi = min(resultado["x"]), max(resultado["x"])
    aviso = None
    if not lo <= x0 <= hi:
        aviso = (f"extrapolación: x0 = {_num(x0, 15)} está fuera del intervalo de los datos [{_num(lo, 15)}, "
                 f"{_num(hi, 15)}]; nada garantiza que el modelo valga ahí")
    return {"x0": x0, "y": _evalua(resultado, x0), "aviso": aviso}


def compara_bibliotecas(x, y, grado):
    """Coeficientes de un polinomio de grado dado por tres caminos: ecuaciones normales (np.linalg.solve con AᵀA),
    np.linalg.lstsq (descomposición en valores singulares, sin AᵀA) y np.polyfit (que los da AL REVÉS, de la
    potencia más alta a la más baja). Diccionario con propios (los de ajuste()), normales, lstsq, polyfit,
    cond_A y cond_ATA (los números de condición de A y de AᵀA: el segundo es el cuadrado del primero), coinciden y
    texto."""
    res = ajuste(x, y, "polinomio", grado)
    g = res["grado"]
    X, Y = np.array([float(v) for v in res["x"]]), np.array([float(v) for v in res["y"]])
    A = np.column_stack([X ** p for p in range(g + 1)])
    cond_A, cond_ATA = float(np.linalg.cond(A)), float(np.linalg.cond(A.T @ A))
    try:
        normales = np.linalg.solve(A.T @ A, A.T @ Y)
    except np.linalg.LinAlgError:
        raise ValueError("AᵀA resultó singular en decimales: A está demasiado mal condicionada para las ecuaciones "
                         "normales") from None
    lstsq = np.linalg.lstsq(A, Y, rcond=None)[0]
    polyfit = np.polyfit(X, Y, g)
    propios = np.array([float(c) for c in res["coeficientes"]])
    escala = max(1.0, float(np.abs(propios).max()))
    coinciden = all(np.allclose(v, propios, rtol=0, atol=1e-9 * escala) for v in (normales, lstsq, polyfit[::-1]))
    orden = ", ".join(f"β{p}" for p in range(g, -1, -1))
    al_reves = ", ".join(f"β{p}" for p in range(g + 1))
    texto = (f"np.polyfit devuelve ({orden}): de la potencia más alta a la más baja. Leídos al revés, "
             f"({al_reves}), son los mismos que dan las ecuaciones normales y np.linalg.lstsq."
             + ("" if coinciden else " (Aquí difieren más de lo esperado: A está mal condicionada.)")
             + f" Números de condición: cond(A) ≈ {cifras(cond_A, 4)} y cond(AᵀA) ≈ {cifras(cond_ATA, 4)}, que es "
               "cond(A)²: por eso las bibliotecas no forman AᵀA.")
    return {"propios": propios, "normales": normales, "lstsq": lstsq, "polyfit": polyfit, "cond_A": cond_A,
            "cond_ATA": cond_ATA, "coinciden": coinciden, "texto": texto}


# --- informes ---------------------------------------------------------------------------------------
def _num_exacto(e, n):
    """Una entrada exacta de AᵀA o de Aᵀy: tal cual si su decimal es finito y corto (12015, 159.9); si no, con n
    cifras y notación ×10^ (nunca un entero redondeado que parezca exacto)."""
    d = _decimales_exactos(e)
    if d is not None and len(_fijo(e, d).replace("-", "").replace(".", "").lstrip("0")) <= 20:
        return _fijo(e, d)
    return cifras(e, n)


def _muestra_num(nombre, M, n, tex, referencia=False):
    """Muestra AᵀA o Aᵀy: exactas (= si ninguna entrada se redondeó, ≈ si alguna sí) o, si son de numpy,
    redondeadas a n cifras."""
    if isinstance(M, np.ndarray):
        muestra_decimales(nombre, M, n, tex)
        return
    textos = [[_num_exacto(M[i, j], n) for j in range(M.cols)] for i in range(M.rows)]
    igual = all(_exacto_en(M[i, j], textos[i][j]) for i in range(M.rows) for j in range(M.cols))
    signo, signo_tex = ("=", "=") if igual else ("≈", r"\approx")
    filas_tex = r" \\ ".join(" & ".join(_tex_cifra(t) for t in fila) for fila in textos)
    anchos = [max(len(textos[i][j]) for i in range(M.rows)) for j in range(M.cols)]
    plano = "\n".join("[ " + "  ".join(t.rjust(a) for t, a in zip(fila, anchos)) + " ]" for fila in textos)
    _emite(_Formula(f"{tex} {signo_tex} " + r"\left[\begin{matrix}" + filas_tex + r"\end{matrix}\right]" +
                    f"\\qquad ({M.rows}\\times {M.cols})", f"{nombre}  ({dims(M)}) {signo}\n{plano}"))


def _tabla(res, n_cifras, max_filas=20):
    """Tabla de datos (como se escribieron), predichos y residuos (con los mismos decimales en cada columna)."""
    plano = res["modelo"] == "plano"
    letra, sombrero = ("z", "ẑ") if plano else ("y", "ŷ")
    k = min(res["n"], max_filas)
    columnas = [[str(i + 1) for i in range(k)], res["x_txt"][:k]] + ([res["yd_txt"][:k]] if plano else [])
    columnas += [res["y_txt"][:k], _columna(res["predichos"][:k], n_cifras), _columna(res["residuos"][:k], n_cifras)]
    enc = ["dato", "x"] + (["y"] if plano else []) + [letra, f"{sombrero} (predicho)", f"r = {letra} - {sombrero}"]
    anchos = [max(len(t) for t in col + [e]) for col, e in zip(columnas, enc)]
    print("  ".join(e.rjust(a) for e, a in zip(enc, anchos)))
    for i in range(k):
        print("  ".join(col[i].rjust(a) for col, a in zip(columnas, anchos)))
    if res["n"] > max_filas:
        print(f"  ... ({res['n']} datos; la tabla muestra los primeros {max_filas})")


def _informe_ajuste(res, n_cifras, prediccion=None):
    m, n = res["modelo"], res["n"]
    letra = "z" if m == "plano" else "y"
    nombre = f"polinomio de grado {res['grado']}" if m == "polinomio" else m
    print(f"Modelo: {nombre}, {_forma(m, res['grado'])}; {_cuenta(n, 'dato', 'datos')}.")
    if res["exacto"]:
        print(f"Cálculo exacto con fracciones: ecuaciones normales AᵀA β = Aᵀ{letra}.")
    else:
        razon = ("el modelo usa logaritmos" if m in ("exponencial", "potencia") else
                 f"hay más de {_MAX_EXACTO} datos" if n > _MAX_EXACTO else "el grado es mayor que 3")
        print(f"Cálculo en decimales con np.linalg.lstsq (sin formar AᵀA), porque {razon}.")
    cols = {"exponencial": ["1", "x"], "potencia": ["1", "ln x"], "plano": ["1", "x", "y"]}.get(
        m, [_potencia("x", p) or "1" for p in (res["potencias"] or [])])
    print(f"A tiene {_cuenta(n, 'renglón', 'renglones')} (uno por dato) y {_cuenta(len(cols), 'columna', 'columnas')}: "
          f"{_lista(cols)}" + ("; el lado derecho es ln y." if m in ("exponencial", "potencia") else "."))
    ATA = res["ATA"]
    k = ATA.rows if isinstance(ATA, sp.MatrixBase) else ATA.shape[0]
    derecha, tex_der = (("Aᵀ(ln y)", r"A^T(\ln\mathbf y)") if m in ("exponencial", "potencia") else
                        (f"Aᵀ{letra}", rf"A^T\mathbf {letra}"))
    if k <= 4:
        if not res["exacto"]:
            print("AᵀA y " + derecha + ", solo como referencia (lstsq no los usa):")
        _muestra_num("AᵀA", ATA, n_cifras, r"A^TA")
        _muestra_num(derecha, res["ATy"], n_cifras, tex_der)
    else:
        print(f"AᵀA es de {k}×{k}: no se muestra aquí (está en el resultado de ajuste, clave 'ATA').")
    if res["coeficientes_ln"] is not None:
        b = "k" if m == "exponencial" else "b"
        print(f"Recta en ln y: ln a ≈ {cifras(res['coeficientes_ln'][0], n_cifras)}, {b} ≈ "
              f"{cifras(res['coeficientes_ln'][1], n_cifras)};  a = e^(ln a)")
    partes = []
    for nm, c in zip(res["nombres"], res["coeficientes"]):
        if nm == "a" and abs(res["coeficientes_ln"][0]) > 700:
            partes.append(f"a = e^(ln a), con ln a ≈ {cifras(res['coeficientes_ln'][0], n_cifras)} (no cabe en los "
                          "números decimales)")
        else:
            partes.append(_eq(nm, _coef_txt(c, n_cifras)))
    print("Coeficientes: " + ",  ".join(partes))
    print("Ecuación: " + res["ecuacion"])
    if res["ecuacion_centrada"]:
        print("Forma centrada (se lee mejor): " + res["ecuacion_centrada"])
    _tabla(res, n_cifras)
    if m in ("recta", "polinomio", "plano"):
        suma = sum(res["residuos"])
        cero = suma == 0 if res["exacto"] else abs(suma) <= 1e-9 * max(1.0, max(abs(float(v)) for v in res["y"]))
        print(_eq("Suma de residuos", _num(suma, n_cifras) if res["exacto"] else f"≈ {cifras(suma, n_cifras)}") +
              ((": es cero por la columna de unos." if res["exacto"] else
                ": es cero salvo el redondeo, por la columna de unos.") if cero else ""))
        if m == "recta":
            tx, ex = _como_datos(sum(res["x"]) / n, res["x_txt"], n_cifras)
            ty, ey = _como_datos(sum(res["y"]) / n, res["y_txt"], n_cifras)
            print(f"La recta pasa por (x̄, ȳ) {'=' if ex and ey else '≈'} ({tx}, {ty}).")
    print(f"{_eq('SSE', _medida_txt(res['SSE'], n_cifras))},  {_eq('RMSE', _medida_txt(res['RMSE'], n_cifras))}" +
          (f",  {_eq('R²', _medida_txt(res['R2'], n_cifras))}" + (" (de ln y)" if m in ("exponencial", "potencia") else "")
           if res["R2"] is not None else ",  R²: no se da (el modelo no tiene ordenada al origen o todos los "
                                         f"{letra} son iguales)"))
    if res["exacto"]:
        print("SSE = 0: los datos están exactamente sobre el modelo (el sistema sí tenía solución)." if res["SSE"] == 0
              else "SSE > 0: ningún modelo de esta forma pasa por todos los datos; el ajuste es la solución de "
                   "mínimos cuadrados.")
    for aviso in res["avisos"]:
        print("Aviso:", aviso + ".")
    if prediccion is not None:
        print(f"Predicción en x = {_num(prediccion['x0'], 15)}: {_eq('ŷ', _coef_txt(prediccion['y'], n_cifras))}")
        if prediccion["aviso"]:
            print("Aviso:", prediccion["aviso"] + ".")


# --- dibujos ----------------------------------------------------------------------------------------
def _residuos_dibujo(res, n_cifras=4):
    """Los residuos para dibujar y el título del panel: si el modelo pasa por todos los datos en decimales, los
    residuos son solo redondeo y se dibujan como 0."""
    R = np.array([float(v) for v in res["residuos"]])
    s = _medida_txt(res["SSE"], n_cifras)
    titulo = f"Residuos: SSE {s}" if s.startswith("≈") else f"Residuos: SSE = {s}"
    escala = max(1.0, max(abs(float(v)) for v in res["y"]))
    if not res["exacto"] and np.all(np.abs(R) <= 1e-9 * escala):
        return np.zeros_like(R), "Residuos: 0 (el modelo pasa por todos los datos; lo demás es redondeo)"
    return R, titulo


def dibuja_ajuste(resultado, etiqueta_x="x", etiqueta_y="y", unidad_y="", prediccion=None, n_cifras=4):
    """Dos paneles: los datos con el modelo (curva suave, 400 puntos) y los residuos (tallos con la línea en cero).
    etiqueta_x y etiqueta_y rotulan los ejes (magnitud y unidad); unidad_y es la unidad de los residuos."""
    if resultado["modelo"] == "plano":
        return _dibuja_plano(resultado, n_cifras)
    X = np.array([float(v) for v in resultado["x"]])
    Y = np.array([float(v) for v in resultado["y"]])
    R, titulo_r = _residuos_dibujo(resultado, n_cifras)
    lo, hi = float(X.min()), float(X.max())
    if prediccion is not None:
        lo, hi = min(lo, float(prediccion["x0"])), max(hi, float(prediccion["x0"]))
    if resultado["modelo"] == "recta por el origen":
        lo, hi = min(lo, 0.0), max(hi, 0.0)
    ancho = (hi - lo) or max(1.0, abs(hi))
    a, b = lo - 0.05 * ancho, hi + 0.05 * ancho
    if resultado["modelo"] == "potencia" and a <= 0:
        a = 0.5 * lo if lo > 0 else float(X.min()) * 0.5
    t = np.linspace(a, b, 400)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.8))
    ax1.plot(t, _evalua_flotante(resultado, t), "-", color=_COLORES[0], lw=2.3,
             label=textwrap.fill(f"modelo: {resultado['ecuacion']}", 48))
    ax1.plot(X, Y, "o", color="black", ms=6, zorder=3, label="datos")
    if prediccion is not None:
        ax1.plot(float(prediccion["x0"]), float(prediccion["y"]), "X", color=_COLORES[1], ms=11, zorder=4,
                 label=f"predicción en x = {_num(prediccion['x0'], 15)}" + (" (extrapolación)" if prediccion["aviso"] else ""))
    ax1.set_xlabel(etiqueta_x); ax1.set_ylabel(etiqueta_y); ax1.grid(alpha=0.4)
    ax1.set_title(f"Datos y modelo ajustado ({_cuenta(resultado['n'], 'dato', 'datos')})", fontsize=10)
    ax1.legend(fontsize=8, loc="best")
    ax2.vlines(X, 0, R, colors="0.35", linestyles="-", lw=1.6)
    ax2.plot(X, R, "s", color=_COLORES[2], ms=6, zorder=3, label="residuo r = y - ŷ (tallo y cuadro)")
    ax2.axhline(0, color="black", lw=1.0, ls="--", label="r = 0")
    ax2.set_xlabel(etiqueta_x)
    ax2.set_ylabel("residuo y - ŷ" + (f" ({unidad_y})" if unidad_y else ""))
    ax2.grid(alpha=0.4)
    ax2.set_title(textwrap.fill(titulo_r, 55), fontsize=10)
    ax2.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()


def _dibuja_plano(res, n_cifras):
    """Plano ajustado con los datos y sus residuos (segmentos verticales), y los residuos por número de dato."""
    X = np.array([float(v) for v in res["x"]]); Y = np.array([float(v) for v in res["y_datos"]])
    Z = np.array([float(v) for v in res["z"]]); Zp = np.array([float(v) for v in res["predichos"]])
    fig = plt.figure(figsize=(11.5, 5.0))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    gx = np.linspace(X.min(), X.max(), 12) if X.max() > X.min() else np.array([X.min() - 1, X.min() + 1])
    gy = np.linspace(Y.min(), Y.max(), 12) if Y.max() > Y.min() else np.array([Y.min() - 1, Y.min() + 1])
    GX, GY = np.meshgrid(gx, gy)
    b0, b1, b2 = (float(v) for v in res["coeficientes"])
    with np.errstate(all="ignore"):
        GZ = b0 + b1 * GX + b2 * GY
    ax.plot_surface(GX, GY, GZ, color="0.7", alpha=0.35, edgecolor="0.45", linewidth=0.4)
    ax.scatter(X, Y, Z, color="black", s=30, depthshade=False, label="datos")
    for xi, yi, zi, zp in zip(X, Y, Z, Zp):
        ax.plot([xi, xi], [yi, yi], [zi, zp], ":", color=_COLORES[1], lw=1.8)
    ax.plot([], [], ":", color=_COLORES[1], lw=1.8, label="residuo (vertical, punteado)")
    ax.plot([], [], "s", color="0.7", label=textwrap.fill(f"plano: {res['ecuacion']}", 48))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_title("Datos y plano ajustado", fontsize=10)
    ax.legend(fontsize=8, loc="upper left")
    ax2 = fig.add_subplot(122)
    k = np.arange(1, res["n"] + 1)
    R, titulo_r = _residuos_dibujo(res, n_cifras)
    ax2.vlines(k, 0, R, colors="0.35", lw=1.6)
    ax2.plot(k, R, "s", color=_COLORES[2], ms=6, label="residuo r = z - ẑ")
    ax2.axhline(0, color="black", lw=1.0, ls="--", label="r = 0")
    ax2.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))     # números de dato enteros
    ax2.set_xlabel("número de dato"); ax2.set_ylabel("residuo z - ẑ"); ax2.grid(alpha=0.4)
    ax2.set_title(textwrap.fill(titulo_r, 55), fontsize=10)
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

**Recorrido paso a paso.** La semilla del libro: un resorte cuelga con masas de 0, 100, 200, 300 y 400 g y mide 10.0, 12.1, 13.8, 16.2 y 17.9 cm. Los cinco puntos no están sobre una recta, así que el sistema $A\boldsymbol\beta=\mathbf y$ no tiene solución. Mínimos cuadrados da $y=10.02+0.0199\,x$ (con $x$ en gramos y $y$ en centímetros), con residuos $(-0.02,\,0.09,\,-0.20,\,0.21,\,-0.08)$ y $\mathrm{SSE}=0.099$. La recta pasa por $(\bar x,\bar y)=(200,\,14.0)$ y por ninguno de los datos; la recta que une el primer y el último dato tiene $\mathrm{SSE}=0.11375$, más grande. La pendiente da la constante del resorte: 100 g (peso $0.1\cdot 9.81=0.981$ N) alargan $1.99$ cm $=0.0199$ m, así que $k=\dfrac{0.981}{0.0199}\approx 49.3$ N/m.

In [ ]:
_xs, _ys = "0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9"
_semilla = ajuste(_xs, _ys, "recta")
_informe_ajuste(_semilla, 4)
_X, _Y = datos(_xs), datos(_ys)
_xb, _yb = sum(_X) / 5, sum(_Y) / 5
_Sxy = sum((a - _xb) * (b - _yb) for a, b in zip(_X, _Y))
_Sxx = sum((a - _xb) ** 2 for a in _X)
print(f"\nEn forma cerrada: x̄ = {_num(_xb)}, ȳ = {_fijo(_yb, 1)}, Sxy = {_num(_Sxy)}, Sxx = {_num(_Sxx)};  "
      f"β1 = Sxy/Sxx = {_num(_Sxy / _Sxx)},  β0 = ȳ - β1·x̄ = {_num(_yb - _Sxy / _Sxx * _xb)}")
print("¿La recta pasa por algún dato?", "sí" if any(r == 0 for r in _semilla["residuos"]) else "no, por ninguno")
_m = (_Y[-1] - _Y[0]) / (_X[-1] - _X[0])
_sse_extremos = sum((b - (_Y[0] + _m * (a - _X[0]))) ** 2 for a, b in zip(_X, _Y))
print(f"Recta que une el primer y el último dato: y = {_fijo(_Y[0], 1)} + {_num(_m)} x, con SSE = {_num(_sse_extremos, 6)} "
      f"> {_num(_semilla['SSE'])}")
_b1 = _semilla["coeficientes"][1]
_k = sp.Rational(981, 1000) / _b1          # 100 g alargan 100·β1 cm = β1 m (numéricamente)
print(f"Constante del resorte: 100 g (0.1 kg, peso 0.1·9.81 = 0.981 N) alargan 100·{_num(_b1)} = {_num(100 * _b1)} cm "
      f"= {_num(_b1)} m, así que k = 0.981/{_num(_b1)} ≈ {cifras(_k, 3)} N/m")
dibuja_ajuste(_semilla, "masa (g)", "longitud del resorte (cm)", "cm")

**Las ecuaciones normales a la vista.** Con los datos $(0,1)$, $(1,3)$, $(2,4)$ y $(3,4)$ y el modelo $y=\beta_0+\beta_1x$:

$$A=\begin{bmatrix}1&0\\1&1\\1&2\\1&3\end{bmatrix},\qquad A^TA=\begin{bmatrix}4&6\\6&14\end{bmatrix},\qquad A^T\mathbf y=\begin{bmatrix}12\\23\end{bmatrix},$$

y resolver $A^TA\,\boldsymbol\beta=A^T\mathbf y$ da $(\beta_0,\beta_1)=(3/2,\,1)$: $y=\tfrac32+x$. Al final, la descarga de un capacitor: $t=0,1,2,3$ s y $v=10.0,\ 6.1,\ 3.6,\ 2.3$ V. El modelo $v=V_0\,e^{kt}$ se linealiza como $\ln v=\ln V_0+kt$; sale $V_0\approx 9.94$ V y la constante de tiempo $\tau=-1/k\approx 2.03$ s. También se comparan las bibliotecas con la semilla, con los números de condición de $A$ y de $A^TA$.

In [ ]:
_r2 = ajuste("0 1 2 3", "1 3 4 4", "recta")
_informe_ajuste(_r2, 4)
print("\n" + "=" * 90 + "\n\nDescarga de un capacitor:")
_desc = ajuste("0 1 2 3", "10.0 6.1 3.6 2.3", "exponencial")
_informe_ajuste(_desc, 4)
_V0, _kd = _desc["coeficientes"]
print(f"V0 = a ≈ {cifras(_V0, 3)} V,  k ≈ {cifras(_kd, 3)} 1/s,  τ = -1/k ≈ {cifras(-1 / _kd, 3)} s")
dibuja_ajuste(_desc, "tiempo t (s)", "voltaje v (V)", "V")
print("\n" + "=" * 90 + "\n\nLa semilla con tres caminos de numpy:")
_cmp = compara_bibliotecas(_xs, _ys, 1)
muestra_decimales("ecuaciones normales (β0, β1)", _cmp["normales"], 6, tex=r"\text{ecuaciones normales }(\beta_0,\ \beta_1)")
muestra_decimales("np.linalg.lstsq (β0, β1)", _cmp["lstsq"], 6, tex=r"\text{np.linalg.lstsq }(\beta_0,\ \beta_1)")
muestra_decimales("np.polyfit (β1, β0)", _cmp["polyfit"], 6, tex=r"\text{np.polyfit }(\beta_1,\ \beta_0)")
print(_cmp["texto"])

**Calculadora de ajuste.** Escribe los $x$ y los $y$ como listas de números separados por espacios o comas (el mismo número de datos), elige el modelo y, para el polinomio, el grado (el control de grado solo se usa con «polinomio»). Si escribes un valor en «predecir en x =», la calculadora lo evalúa y avisa si es una extrapolación. Con hasta 12 datos y grado hasta 3 (recta, recta por el origen o polinomio) el cálculo es exacto; si no, se hace en decimales con `np.linalg.lstsq`. Prueba subir el grado hasta que el polinomio pase por todos los datos, predecir en $x=1000$, cambiar un dato por uno atípico y usar años como $x$ (2000, 2001, ...) para ver el aviso de mal condicionamiento.

In [ ]:
def calculadora_ajuste(x_txt, y_txt, modelo, grado, x0_txt, n_cifras):
    with bloque():                 # el texto aparece completo, de una sola vez; las gráficas van después
        try:
            res = ajuste(x_txt, y_txt, modelo, grado if modelo == "polinomio" else None, n_cifras=n_cifras)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        pred, problema = None, None
        if str(x0_txt).strip():          # un problema con x0 no borra el ajuste: se dice al final
            try:
                pred = predice(res, x0_txt)
            except _Desborde as err:
                problema = f"Aviso: {err}."
            except ValueError as err:
                problema = f"Revisa «predecir en x =»: {err}"
        _informe_ajuste(res, n_cifras, pred)
        if problema:
            print(problema)
    dibuja_ajuste(res, "x", "y", "", pred, n_cifras)


_CONTROLES_4A = dict(
    x_txt=widgets.Textarea(value="0 100 200 300 400", description="x:", continuous_update=False,
                           layout=widgets.Layout(width="420px", height="60px")),
    y_txt=widgets.Textarea(value="10.0 12.1 13.8 16.2 17.9", description="y:", continuous_update=False,
                           layout=widgets.Layout(width="420px", height="60px")),
    modelo=widgets.Dropdown(options=_MODELOS, value="recta", description="modelo:"),
    grado=widgets.IntSlider(value=2, min=1, max=6, description="grado", continuous_update=False),
    x0_txt=widgets.Text(value="", description="predecir en x =", continuous_update=False,
                        style={"description_width": "initial"}),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_ajuste, **_CONTROLES_4A);

**Calculadora del plano.** Escribe un dato por renglón como `x, y, z`. La calculadora ajusta $z=\beta_0+\beta_1x+\beta_2y$ con las columnas $1$, $x$ y $y$ (exacto con hasta 12 datos) y dibuja el plano con los residuos.

In [ ]:
def _tabla_plano(tabla_txt):
    """Las tres listas x, y, z (como se escribieron) de una tabla con un dato por renglón."""
    if not str(tabla_txt).strip():
        raise ValueError("escribe un dato por renglón como x, y, z, por ejemplo: 0, 0, 1.0")
    try:
        T = matriz(tabla_txt, "la tabla de datos")
    except ValueError as err:
        if str(err).startswith("escribe "):
            raise ValueError("escribe un dato por renglón como x, y, z, por ejemplo: 0, 0, 1.0") from None
        raise
    if T.cols != 3:
        raise ValueError(f"cada renglón es un dato con tres números, x, y, z (escribiste "
                         f"{_cuenta(T.cols, 'número', 'números')} por renglón), por ejemplo: 0, 0, 1.0")
    if T.rows > _MAX_DATOS:
        raise ValueError(f"la tabla tiene {T.rows} datos; esta calculadora acepta hasta {_MAX_DATOS}")
    filas, _ = _trozos(tabla_txt)
    return [f[0] for f in filas], [f[1] for f in filas], [f[2] for f in filas]


def calculadora_plano(tabla_txt, n_cifras):
    with bloque():
        try:
            res = ajuste_plano(*_tabla_plano(tabla_txt), n_cifras=n_cifras)
        except ValueError as err:
            print("Revisa la entrada:", err); return
        _aviso_coma(tabla_txt)
        _informe_ajuste(res, n_cifras)
    dibuja_ajuste(res, n_cifras=n_cifras)


_CONTROLES_4B = dict(
    tabla_txt=widgets.Textarea(value="0, 0, 1.0\n2, 0, 2.1\n0, 2, 4.9\n2, 2, 6.0\n1, 1, 3.6", description="x, y, z:",
                               continuous_update=False, layout=widgets.Layout(width="300px", height="150px")),
    n_cifras=widgets.IntSlider(value=4, min=2, max=8, description="cifras", continuous_update=False))
widgets.interact(calculadora_plano, **_CONTROLES_4B);

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
_semilla_t = ajuste("0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9", "recta")
_cerca = lambda a, b, tol=1e-9: abs(float(a) - float(b)) <= tol
_x14, _y14 = list(range(14)), [1, 3, 2, 4, 5, 5, 6, 8, 8, 9, 11, 12, 12, 14]       # rama decimal: más de 12 datos
_x13 = list(range(13))
_y13 = [(3 * i) % 5 for i in _x13]                                                  # puntos (x, y) no colineales
_z13 = [1 + 2 * a - b for a, b in zip(_x13, _y13)]                                  # z = 1 + 2x - y exacto
_anios, _crece = "2000 2001 2002 2003 2004 2005", "3 4 6 9 13 18"
PRUEBAS_4 = [
    ("recta por (0, 1), (1, 3), (2, 4), (3, 4) → (β0, β1) = (3/2, 1) exactos; residuos (-1/2, 1/2, 1/2, -1/2)",
     lambda: (lambda r: r["coeficientes"] == [sp.Rational(3, 2), 1] and r["exacto"]
                        and r["residuos"] == [sp.Rational(-1, 2), sp.Rational(1, 2), sp.Rational(1, 2), sp.Rational(-1, 2)])(
         ajuste("0 1 2 3", "1 3 4 4", "recta"))),
    ("mismos datos: Aᵀr = 0, AᵀA = [[4, 6], [6, 14]] y Aᵀy = (12, 23)",
     lambda: (lambda r: _es_nulo(r["A"].T * sp.Matrix(r["residuos"])) and r["ATA"] == sp.Matrix([[4, 6], [6, 14]])
                        and r["ATy"] == vector("12, 23"))(ajuste("0 1 2 3", "1 3 4 4", "recta"))),
    ("semilla del resorte (masa en gramos) → (β0, β1) = (10.02, 0.0199), SSE = 0.099 y «y = 10.02 + 0.0199 x»",
     lambda: _cerca(_semilla_t["coeficientes"][0], 10.02) and _cerca(_semilla_t["coeficientes"][1], 0.0199)
             and _semilla_t["SSE"] == sp.Rational(99, 1000) and _semilla_t["ecuacion"] == "y = 10.02 + 0.0199 x"),
    ("la recta de la semilla pasa por (x̄, ȳ) = (200, 14)",
     lambda: _semilla_t["coeficientes"][0] + 200 * _semilla_t["coeficientes"][1] == 14
             and sum(_semilla_t["x"]) / 5 == 200 and sum(_semilla_t["y"]) / 5 == 14),
    ("recta por el origen con (1, 2), (2, 4), (3, 7) → β1 = 31/14 y R2 = None",
     lambda: (lambda r: r["coeficientes"] == [sp.Rational(31, 14)] and r["R2"] is None)(
         ajuste("1 2 3", "2 4 7", "recta por el origen"))),
    ("polinomio de grado 2 con (0, 1), (1, 2), (2, 5), (3, 10) → (1, 0, 1) y SSE = 0",
     lambda: (lambda r: r["coeficientes"] == [1, 0, 1] and r["SSE"] == 0)(ajuste("0 1 2 3", "1 2 5 10", "polinomio", 2))),
    ("polinomio de grado 3 con 4 datos → SSE = 0 y el aviso de interpolación",
     lambda: (lambda r: r["SSE"] == 0 and "el polinomio pasa por todos los datos; residuo cero no significa mejor modelo"
                        in r["avisos"])(ajuste("0 1 2 3", "1 3 2 5", "polinomio", 3))),
    ("exponencial con y = 2·e^(0.5 x) exacto, x = 0, ..., 4 → a ≈ 2 y k ≈ 0.5 (tolerancia 1e-9)",
     lambda: (lambda r: _cerca(r["coeficientes"][0], 2) and _cerca(r["coeficientes"][1], 0.5))(
         ajuste([0, 1, 2, 3, 4], [2 * math.exp(0.5 * t) for t in range(5)], "exponencial"))),
    ("exponencial con un y = 0 → ValueError (se necesita y > 0)",
     lambda: "positivos" in _mensaje(lambda: ajuste("0 1 2", "1 0 2", "exponencial"))),
    ("recta con todos los x iguales → ValueError «las columnas de A son dependientes»",
     lambda: _mensaje(lambda: ajuste("2 2 2", "1 2 3", "recta")).startswith("las columnas de A son dependientes")),
    ("compara_bibliotecas con la semilla, grado 1: np.polyfit al revés = coeficientes propios (tolerancia 1e-9)",
     lambda: (lambda c: np.allclose(c["polyfit"][::-1], c["propios"], rtol=0, atol=1e-9) and c["coinciden"]
                        and not np.allclose(c["polyfit"], c["propios"], rtol=0, atol=1e-3))(
         compara_bibliotecas("0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9", 1))),
    ("predice con la semilla: en x = 1000 avisa extrapolación; en x = 250 no avisa y da 14.995",
     lambda: "extrapolación" in (predice(_semilla_t, 1000)["aviso"] or "")
             and predice(_semilla_t, 250)["aviso"] is None and predice(_semilla_t, 250)["y"] == sp.Rational(14995, 1000)),
    ("datos exactamente en y = 3 - 2x → R2 = 1",
     lambda: ajuste("0 1 2 5", "3 1 -1 -7", "recta")["R2"] == 1),
    ("ajuste_plano con z = 1 + 2x - y en 4 puntos no colineales → (1, 2, -1)",
     lambda: ajuste_plano("0 1 0 1", "0 0 1 1", "1 3 0 2")["coeficientes"] == [1, 2, -1]),
    ("potencia con y = 3·x² → a ≈ 3 y b ≈ 2",
     lambda: (lambda r: _cerca(r["coeficientes"][0], 3) and _cerca(r["coeficientes"][1], 2))(
         ajuste("1 2 3 4 5", "3 12 27 48 75", "potencia"))),
    ("prueba de humo: las dos calculadoras corren con sus valores iniciales; la de ajuste, con cada modelo, con una "
     "predicción y con el deslizador en grado 4 (rama decimal, que sí usa el grado)",
     lambda: _humo(calculadora_ajuste, _CONTROLES_4A) and _humo(calculadora_plano, _CONTROLES_4B)
             and all(_humo(calculadora_ajuste, _CONTROLES_4A, modelo=m, x0_txt="5") for m in _MODELOS if m != "potencia")
             and _humo(calculadora_ajuste, _CONTROLES_4A, modelo="potencia", x_txt="1 2 3 4 5", x0_txt="5")
             and (lambda t, e: e is None and "polinomio de grado 4" in t and "x⁴" in t and "decimales" in t)(
                 *_captura(calculadora_ajuste, "0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9", "polinomio", 4, "", 4))),
    ("rama decimal: polinomio de grado 4 con 6 datos y de grado 3 con 14 datos → los coeficientes de np.polyfit, al revés",
     lambda: all((lambda r, g, x, y: not r["exacto"] and np.allclose(
         [float(c) for c in r["coeficientes"]], np.polyfit(x, y, g)[::-1], rtol=1e-8, atol=1e-10))(
             ajuste(x, y, "polinomio", g), g, x, y)
         for g, x, y in ((4, [0, 1, 2, 3, 4, 5], [1, 3, 2, 5, 4, 6]), (3, _x14, _y14)))),
    ("plano en decimales (13 datos sobre z = 1 + 2x - y) → (β0, β1, β2) = (1, 2, -1) en ese orden",
     lambda: (lambda r: not r["exacto"] and np.allclose([float(c) for c in r["coeficientes"]], [1, 2, -1],
                                                       rtol=0, atol=1e-9))(ajuste_plano(_x13, _y13, _z13))),
    ("AᵀA y Aᵀy de la rama decimal (recta con 13 datos) → [[13, 78], [78, 650]] y (Σy, Σxy)",
     lambda: (lambda r: not r["exacto"] and np.allclose(r["ATA"], [[13, 78], [78, 650]])
                        and np.allclose(r["ATy"], [sum(_y13), sum(a * b for a, b in zip(_x13, _y13))]))(
         ajuste(_x13, _y13, "recta"))),
    ("RMSE y R²: en la semilla, RMSE = √(0.099/5) y R² = 1 - 0.099/39.7 exactos; en la rama decimal (14 datos), "
     "R² = 1 - SSE/SST calculado aparte",
     lambda: _semilla_t["RMSE"] == sp.sqrt(sp.Rational(99, 1000) / 5)
             and _semilla_t["R2"] == 1 - sp.Rational(99, 1000) / sp.Rational(397, 10)
             and (lambda r: _cerca(r["R2"], 1 - r["SSE"] / float(((np.array(_y14) - np.mean(_y14)) ** 2).sum()), 1e-12)
                            and _cerca(r["RMSE"], math.sqrt(r["SSE"] / 14), 1e-12))(ajuste(_x14, _y14, "polinomio", 3))),
    ("extrapolación hacia abajo: predice con la semilla en x = -100 avisa",
     lambda: "extrapolación" in (predice(_semilla_t, -100)["aviso"] or "")),
    ("años 2000 a 2005 con grado 2 (rama exacta): aviso de A mal condicionada, forma centrada en t = x - 2003, "
     "ecuación en x que reproduce el modelo (con más cifras) y AᵀA con 12015 exacto (no 12020)",
     lambda: (lambda r, t: any("mal condicionada" in a for a in r["avisos"]) and r["exacto"]
                           and "t = x - 2003" in (r["ecuacion_centrada"] or "") and r["ecuacion_reproduce"]
                           and r["cifras_ecuacion"] > 4 and "12015" in t and "12020" not in t)(
         ajuste(_anios, _crece, "polinomio", 2),
         _captura(calculadora_ajuste, _anios, _crece, "polinomio", 2, "", 4)[0])),
    ("con x en años: la exponencial avisa que ln a no cabe (sin error) y da la forma con t; la potencia aconseja cambiar "
     "la unidad de x, no restar el promedio",
     lambda: (lambda e, p: any("ln a" in a and "otro origen" in a for a in e["avisos"]) and e["ecuacion_centrada"]
                           and any("unidad" in a for a in p["avisos"]) and not any("promedio" in a for a in p["avisos"]))(
         ajuste(_anios, _crece, "exponencial"), ajuste(_anios, _crece, "potencia"))),
    ("un problema con «predecir en x =» no borra el ajuste: con «abc» y con un desborde (exponencial, x0 = 1e15)",
     lambda: (lambda a, b: "Ecuación:" in a and "Revisa «predecir en x =»" in a and "Revisa la entrada" not in a
                           and "Ecuación:" in b and "se desborda" in b)(
         _captura(calculadora_ajuste, "0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9", "recta", 2, "abc", 4)[0],
         _captura(calculadora_ajuste, "0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9", "exponencial", 2, "1e15", 4)[0])),
    ("compara_bibliotecas da cond(A) y cond(AᵀA), y con la semilla cond(AᵀA) ≈ cond(A)²",
     lambda: (lambda c: abs(c["cond_ATA"] / c["cond_A"] ** 2 - 1) < 1e-3 and "cond(AᵀA)" in c["texto"])(
         compara_bibliotecas("0 100 200 300 400", "10.0 12.1 13.8 16.2 17.9", 1))),
    ("plano con los puntos (x, y) casi sobre una recta: aviso, y la ecuación con las cifras que la hacen reproducir "
     "el modelo",
     lambda: (lambda r: any("casi sobre una recta" in a for a in r["avisos"]) and r["ecuacion_reproduce"]
                        and r["cifras_ecuacion"] > 4)(ajuste_plano("0 1 2 3", "0 1 2.000001 3", "1 2 3 4.1"))),
]
corre_pruebas(PRUEBAS_4, 4)

**Contrasta (sección 4).** Calcula a mano la recta de mínimos cuadrados $y=\beta_0+\beta_1x$ para los puntos $(0,0)$, $(1,1)$ y $(2,1)$: arma $A^TA$ y $A^T\mathbf y$ y resuelve, o usa $\beta_1=S_{xy}/S_{xx}$ y $\beta_0=\bar y-\beta_1\bar x$. Escribe tus dos coeficientes en el orden $(\beta_0,\beta_1)$, por ejemplo `mi_recta = (2, -0.5)`; puedes usar fracciones como texto (`"1/3"`) o decimales.

In [ ]:
mi_recta = None     # dos números (β0, β1), por ejemplo (2, -0.5) o ("1/3", 2)

if mi_recta is None:     # la referencia se muestra después de tu cálculo
    print("Falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda.")
else:
    try:
        mia = vector(mi_recta, "tu recta")
    except ValueError as err:
        print(f"Revisa tu recta: {str(err).rstrip('.')}.")
    else:
        if mia.rows != 2:          # respuesta imposible: la referencia todavía no se muestra
            print(f"La recta tiene dos coeficientes, β0 y β1 (escribiste {mia.rows}): revisa tu respuesta y vuelve a "
                  "ejecutar la celda.")
        else:
            ref = ajuste("0 1 2", "0 1 1", "recta")
            b = ref["coeficientes"]
            print(f"La calculadora da: {_eq('β0', _coef_txt(b[0], 4))}, {_eq('β1', _coef_txt(b[1], 4))}  "
                  f"({ref['ecuacion']}).")
            if all(abs(float(a - c)) <= 0.01 for a, c in zip(mia, b)):
                print("Coinciden (con tolerancia de 0.01).")
            elif all(abs(float(a - c)) <= 0.01 for a, c in zip(mia, reversed(b))):
                print("NO coinciden: probablemente invertiste el orden. β0 es la ordenada al origen (el valor de y en "
                      "x = 0) y β1 es la pendiente; escribe (β0, β1).")
            else:
                print("NO coinciden: revisa AᵀA (n, Σx; Σx, Σx²) y Aᵀy (Σy, Σxy), o calcula x̄, ȳ, Sxy y Sxx.")
            print("\nEl procedimiento completo:")
            _informe_ajuste(ref, 4)

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
_ESPERADAS = {1: 19, 2: 17, 3: 15, 4: 26}          # pruebas por sección (el libro cita estos números)
fallas, total = [], 0
for k, pruebas in enumerate([PRUEBAS_1, PRUEBAS_2, PRUEBAS_3, PRUEBAS_4], 1):
    assert len(pruebas) == _ESPERADAS[k], (f"la sección {k} tiene {len(pruebas)} pruebas y deben ser "
                                           f"{_ESPERADAS[k]}: falta o sobra alguna")
    resultados = evalua(pruebas)                      # cada caso se ejecuta una sola vez
    print(f"Sección {k}: {sum(ok for _, ok in resultados)}/{len(pruebas)}")
    fallas += [f"sección {k}: {nombre}" for nombre, ok in resultados if not ok]
    total += len(pruebas)
for falla in fallas:
    print("FALLA", falla)
assert not fallas, f"Fallan {len(fallas)} de {total} pruebas (están arriba)"
print(f"Todas las verificaciones pasaron: {total}/{total}")